# WeatherNext 3 cyclone guidance
Source: https://deepmind.google.com/science/weatherlab

Edit the **SETTINGS** block in the first cell, then *Run All*. The first cell finds the newest WNV3 run, downloads it to the disposable folder and prints the potential storms (use those IDs for the optional storm-specific sections).

In [ ]:
%matplotlib inline
# ============================================================================
# SETTINGS  --  edit this block, then Run All
# ============================================================================

# ---- Run selection ---------------------------------------------------------
# None = automatically find the newest WeatherNext 3 (WNV3) run on Weather Lab.
# Or force a run, e.g. "2026_10_04T18_00".
RUN_OVERRIDE = None
# RUN_OVERRIDE = "2026_10_05T00_00"
MAX_LOOKBACK_CYCLES = 12          # how many 6-hour cycles to search backward

# ---- Output (disposable: wiped and recreated each run) ----------------------
# Layout: data/ (downloads), global_tracks/, probability_maps/, storm_tracks/,
# intensity_charts/, impact_cards/ (each with dark/ and light/ image folders),
# plus datawrapper/ and landfall_odds/ (GeoJSON and CSVs).
SAVE_DIR = "/tmp/weathernext3"
DELETE_EXISTING_FOLDER = True

# ---- Always-on products ----------------------------------------------------
# 1) Global WNV3 member tracks map
# 2) Cumulative probability maps (gridded Weather Lab product)

# Cumulative-probability maps to draw (gridded fields in the .nc file).
CUMPROB_SHOW_TROPICAL_STORM = True    # 34_knot_strike_probability
CUMPROB_SHOW_HURRICANE = True         # 64_knot_strike_probability
CUMPROB_SHOW_50KT = False             # 50_knot_strike_probability
CUMPROB_SHOW_TRACK = False            # track_probability
# None = full forecast window (360 h). Or e.g. 120 for a five-day map.
CUMPROB_MAX_LEAD_HOURS = None
# Weather Lab currently publishes cumulative fields for FNV3_LARGE_ENSEMBLE
# only; WNV3 is tried first in case it is added later.
CUMPROB_MODEL_PREFERENCE = ["WNV3", "FNV3_LARGE_ENSEMBLE"]
KEEP_RAW_CUMPROB_FILE = False         # raw file is ~2 GB uncompressed
# Datawrapper GeoJSON of the tropical-storm- and hurricane-force wind chances
# (locator map -> area markers -> import GeoJSON). Same categories/colors as maps.
EXPORT_DATAWRAPPER_GEOJSON = True
DATAWRAPPER_BANDS = "exclusive"       # "exclusive" (5-30, 30-60, 60+; no overlap)
                                      # or "cumulative" (5+, 30+, 60+; stacked)
DATAWRAPPER_DECIMALS = 3              # coordinate precision (3 = ~100 m)
DATAWRAPPER_MIN_AREA_DEG2 = 0.01      # drop specks smaller than this

# ---- Optional products (off for now) -----------------------------------------
GENERATE_INDIVIDUAL_STORM_TRACKS = False
GENERATE_INTENSITY_CHART = False

# Storm(s) used by the individual-storm sections. Pick IDs from the list
# printed by this cell: named IDs ("EP152026", also matches "EP152026(2)"),
# new-genesis id numbers (2 or "2"), or a mix. Several genesis ids near the
# same spot are often one system split across members, e.g. [0, 1, 2, 3, 4].
STORM_TRACK_IDS = ["AL092026"]
STORM_DISPLAY_NAME = "Hurricane\nIsaias"    # map label ("\n" = line break)
STORM_CHART_NAME = "Isaias"                 # intensity-chart title
# Named storms drop "Potential" from the track-map key title. None = auto
# (NHC active storms, else STORM_DISPLAY_NAME); or force True/False.
STORM_IS_NAMED = None

# ---- Storm impact cards (social, 4K) ---------------------------------------
# One card per storm: chance, timing and strength of winds by country, from
# each WNV3 member's wind radii. Saved in every layout listed.
GENERATE_STORM_IMPACT_CARDS = True
IMPACT_STORM_IDS = None              # None = every storm/invest in >= half the members
# Unnamed systems (new-genesis tracks) are grouped across members by where and
# when they form; those forming in >= IMPACT_MIN_GENESIS_SHARE get cards too.
IMPACT_INCLUDE_GENESIS = True
IMPACT_MIN_GENESIS_SHARE = 0.3       # e.g. 0.3 = forms in at least 30% of members
GENESIS_RADIUS_KM = 600              # same system if first fixes are this close...
GENESIS_WINDOW_HOURS = 72            # ...and this close in time
IMPACT_STORM_NAMES = {}              # overrides, e.g. {"WP272026": "Typhoon Halong"} (NHC/JTWC auto)
IMPACT_MIN_MEMBER_SHARE = 0.5        # auto mode: storm must appear in this share of members
IMPACT_MIN_CHANCE = 5                # % chance of TS-force winds for a country to be listed
IMPACT_PANEL_ROWS = {"3x2": 6, "1x1": 4}  # storm rows ("Storm forms", "Becomes a ...") + countries
IMPACT_LAYOUTS = ["3x2", "1x1"]
IMPACT_TITLE = "{storm} potential tracks and threat assessment"
IMPACT_TITLE_GENESIS = "Where a storm could form in the {basin}"
IMPACT_LABEL_COUNTRIES = True        # name the listed countries on the map
# Peak-wind block on the map: floor / most likely / ceiling of members' peak winds.
IMPACT_SHOW_PEAK_WINDS = True
IMPACT_WIND_PERCENTILES = (10, 90)   # floor and ceiling percentiles
# Gold glow on the average track where the storm is at or near peak strength.
IMPACT_HIGHLIGHT_PEAK = True
IMPACT_PEAK_FRACTION = 0.9           # median wind within 90% of its forecast max
# Compare each storm's forecast peak with the basin's strongest storms this
# calendar year (IBTrACS); shown in the peak-wind block when notable.
IMPACT_SEASON_CONTEXT = True
IMPACT_RECORD_MIN_CHANCE = 10        # % of members topping the year's strongest
# Left off the cards (still in the CSVs).
IMPACT_HIDE_COUNTRIES = ["United States Minor Outlying Islands"]

# ---- Other analyses ---------------------------------------------------------
# U.S. landfall odds (CONUS + each state) for WeatherNext 3 and the
# 1,000-member ensemble, printed separately and side by side.
RUN_LANDFALL_ODDS = True
LANDFALL_MODELS = ["WNV3", "FNV3_LARGE_ENSEMBLE"]
LANDFALL_TRACK_IDS = None            # None = all systems; or e.g. ["AL142026"]
LANDFALL_TOP_STATES = 10             # how many states to list in the printout

# ---- Shared style ----------------------------------------------------------
# Every map, chart and card is saved in each mode (dark/ and light/
# subfolders, "_dark"/"_light" in the filename); only DISPLAY_MODES are shown
# inline here, to keep the notebook light.
COLOR_MODES = ["dark", "light"]
DISPLAY_MODES = ["dark"]
LAYOUT = "3x2"                       # "3x2" or "1x1"
# Map domains for the track + probability maps; one image per domain.
# Options: "global", "Gulf of Mexico", "Gulf of Mexico + Caribbean",
#          "Western Gulf", "Eastern Gulf"   (edit/add in the style cell)
MAP_DOMAINS = ["global", "Gulf of Mexico"]
OUTPUT_WIDTH_PX = 3840               # global maps saved at 4K width
# Image format for every saved map/chart. "jpg" is far smaller than "png".
OUTPUT_FORMAT = "jpg"                # "jpg" or "png"
JPEG_QUALITY = 90                    # 85-95; lower = smaller files
JPEG_SUBSAMPLING = 2                 # 2 = smallest files; 0 = crisper color edges
                                     # (3x2 -> 3840x2560, 1x1 -> 3840x3840)
SSL_VERIFY = False
DOWNLOAD_TIMEOUT = 180
FONT_DIR = "assets/fonts"
SHOW_CREDITS = True                  # data/byline credits on every map and chart
DATA_CREDIT_TEXT = "Data: Google DeepMind"
SHOW_INIT_IN_CREDIT = True          # append model init time, e.g. ", initialized 00 UTC Oct. 7"
BYLINE_CREDIT_TEXT = "BEN NOLL/THE WASHINGTON POST"
# ============================================================================


import os
import re
import shutil
from contextlib import contextmanager
from datetime import datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import urllib3
import matplotlib.pyplot as plt
from matplotlib import font_manager

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://deepmind.google.com/science/weatherlab/download/cyclones"
TRACK_MODEL = "WNV3"
SESSION = requests.Session()
SESSION.verify = SSL_VERIFY

MONTH_ABBR = {
    "Jan": "Jan.", "Feb": "Feb.", "Mar": "March", "Apr": "April",
    "May": "May", "Jun": "June", "Jul": "July", "Aug": "Aug.",
    "Sep": "Sept.", "Oct": "Oct.", "Nov": "Nov.", "Dec": "Dec.",
}


# ---------------------------------------------------------------- fonts
def load_font(filename, size=None):
    path = os.path.join(FONT_DIR, filename)
    if os.path.exists(path):
        return font_manager.FontProperties(fname=path, size=size)
    print(f"Font not found, using default: {path}")
    return font_manager.FontProperties(size=size)


font_properties = load_font("FranklinITCStd-Black.otf")
font_properties2 = load_font("FranklinITCStd-Bold.otf")
font_properties3 = load_font("FranklinITCStd-Light.otf", size=24)
font_properties4 = load_font("FranklinITCStd-Light.otf", size=16)


# ---------------------------------------------------------------- helpers
def ap_date(ts):
    """AP-style short date, e.g. 'Oct. 4'."""
    return f"{MONTH_ABBR[ts.strftime('%b')]} {ts.day}"


DATA_CREDIT_BASE = DATA_CREDIT_TEXT


MODEL_DISPLAY_NAMES = {"WNV3": "WeatherNext 3"}


def data_credit_text(run, model=None):
    """Data credit (optionally naming the model) plus the run's init time,
    e.g. 'Data: Google DeepMind WeatherNext 3, initialized 12 UTC Oct. 13'."""
    base = DATA_CREDIT_BASE
    if model:
        base = f"{base} {MODEL_DISPLAY_NAMES.get(model, model)}"
    if not SHOW_INIT_IN_CREDIT or not run:
        return base
    init = datetime.strptime(run, "%Y_%m_%dT%H_%M")
    return f"{base}, initialized {init:%H} UTC {ap_date(init)}"


def canonical_id(value):
    """'EP152026(2)' -> 'EP152026'."""
    return re.sub(r"\(\d+\)$", "", str(value).strip())


def candidate_runs(lookback=MAX_LOOKBACK_CYCLES):
    now = datetime.now(timezone.utc)
    newest = now.replace(hour=now.hour - now.hour % 6, minute=0,
                         second=0, microsecond=0)
    return [(newest - timedelta(hours=6 * i)).strftime("%Y_%m_%dT%H_%M")
            for i in range(lookback + 1)]


def url_exists(url):
    try:
        r = SESSION.head(url, timeout=20, allow_redirects=True)
        if r.status_code == 405:          # HEAD not allowed -> light GET
            r = SESSION.get(url, timeout=20, stream=True)
            r.close()
        return r.status_code == 200
    except requests.RequestException:
        return False


def track_urls(model, run):
    return {
        "members": f"{BASE_URL}/{model}/ensemble/cyclogenesis/csv/"
                   f"{model}_{run}_cyclogenesis.csv",
        "ensemble_mean": f"{BASE_URL}/{model}/ensemble_mean/paired/csv/"
                         f"{model}_{run}_paired.csv",
    }


def cumprob_url(model, run):
    return (f"{BASE_URL}/{model}/ensemble/cyclogenesis/netcdf/"
            f"cumulative_probability_fields/"
            f"{model}_{run}_cumulative_probability_fields.nc.gz.base64")


def find_latest_run(url_builder, runs):
    for run in runs:
        if url_exists(url_builder(run)):
            return run
    return None


def download(url, folder):
    path = os.path.join(folder, url.rsplit("/", 1)[-1])
    with SESSION.get(url, stream=True, timeout=DOWNLOAD_TIMEOUT) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1 << 20):
                f.write(chunk)
    print(f"Saved {os.path.getsize(path) / 1e6:6.2f} MB  {path}")
    return path


# Current color mode; color_mode() switches it for each render pass.
COLOR_MODE = COLOR_MODES[0]
DARK_MODE = COLOR_MODE == "dark"


@contextmanager
def color_mode(mode):
    """Render in `mode` ("dark"/"light"); hide inline output unless displayed."""
    global COLOR_MODE, DARK_MODE
    previous, show = (COLOR_MODE, DARK_MODE), plt.show
    COLOR_MODE, DARK_MODE = mode, mode == "dark"
    if mode not in DISPLAY_MODES:
        plt.show = lambda *args, **kwargs: plt.close("all")
    try:
        yield
    finally:
        plt.show = show
        COLOR_MODE, DARK_MODE = previous


def for_each_color_mode(func, *args, **kwargs):
    for mode in COLOR_MODES:
        print(f"\n--- {mode} mode ---")
        with color_mode(mode):
            func(*args, **kwargs)


def output_dir(*parts):
    """SAVE_DIR/<parts...>, created on demand."""
    path = os.path.join(SAVE_DIR, *parts)
    os.makedirs(path, exist_ok=True)
    return path


def mode_dir(product):
    """Image folder for the current color mode, e.g. impact_cards/dark."""
    return output_dir(product, COLOR_MODE)


def save_figure(fig, path, **savefig_kwargs):
    """Save in OUTPUT_FORMAT (extension fixed automatically); print file size.

    The current color mode is appended to the filename ("..._dark.jpg").
    """
    root = os.path.splitext(os.fspath(path))[0]
    if not root.endswith(f"_{COLOR_MODE}"):
        root += f"_{COLOR_MODE}"
    if OUTPUT_FORMAT.lower() in ("jpg", "jpeg"):
        path = root + ".jpg"
        savefig_kwargs.update(format="jpeg", pil_kwargs={
            "quality": JPEG_QUALITY, "optimize": True, "progressive": True,
            "subsampling": JPEG_SUBSAMPLING})
    else:
        path = root + ".png"
        savefig_kwargs.update(format="png")
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    fig.savefig(path, **savefig_kwargs)
    print(f"Saved {os.path.getsize(path) / 1e6:.2f} MB: {path}")
    return path


def read_track_csv(path):
    """Weather Lab CSVs start with '#' comment lines; skip them."""
    if path is None:
        return None
    frame = pd.read_csv(path, comment="#", low_memory=False)
    frame.columns = frame.columns.str.strip()
    for col in ("init_time", "valid_time"):
        if col in frame:
            frame[col] = pd.to_datetime(frame[col], errors="coerce")
    return frame


_ONES = ["ONE", "TWO", "THREE", "FOUR", "FIVE", "SIX", "SEVEN", "EIGHT", "NINE"]
NUMBER_WORDS = set(_ONES + [
    "TEN", "ELEVEN", "TWELVE", "THIRTEEN", "FOURTEEN", "FIFTEEN", "SIXTEEN",
    "SEVENTEEN", "EIGHTEEN", "NINETEEN"] + [
    t + o for t in ("TWENTY", "THIRTY", "FORTY") for o in [""] + _ONES])


def is_number_name(name):
    """'Nine', 'Twenty-One' (NHC) or 'TWENTYEIGH' (JTWC, truncated) -> True."""
    key = re.sub(r"[^A-Z]", "", str(name).upper())
    return key in NUMBER_WORDS or (len(key) >= 9 and any(
        w.startswith(key) for w in NUMBER_WORDS))


NHC_CLASS = {"TD": "Tropical Depression", "STD": "Subtropical Depression",
             "TS": "Tropical Storm", "STS": "Subtropical Storm",
             "HU": "Hurricane", "PTC": "Post-Tropical Cyclone",
             "PC": "Potential Tropical Cyclone"}
JTWC_BASE = "https://www.metoc.navy.mil/jtwc"


# ---- Regional intensity conventions ------------------------------------------
# Storms are classified by the official agency (RSMC) for where they are.
# Model winds are 1-minute averages; for agencies using longer averages the
# model wind is multiplied by WIND_AVERAGING_FACTOR first (WMO at-sea
# conversion; the 3-minute value is approximate).
WIND_AVERAGING_FACTOR = {1: 1.0, 3: 0.95, 10: 0.93}
_AUS_TRACKS = [(34, "Category 1-2"), (64, "Category 3"), (86, "Category 4-5")]
REGIONAL_SCALES = {
    # classes: (agency kt, name before a storm's name); tracks: the three
    # track colors on the cards (agency kt, key label); winds: wording for
    # 34-kt and 64-kt winds; milestone: (name, agency kt, averaging minutes)
    # for the "Becomes a ..." chance on the cards.
    "NHC": {"scale": "Saffir-Simpson scale", "averaging": 1,
            "show_scale": False,       # familiar to U.S. readers; not in the key
            "classes": [(0, "Tropical Depression"), (34, "Tropical Storm"),
                        (64, "Hurricane")],
            "tracks": [(34, "Tropical storm"), (64, "Hurricane"),
                       (96, "Major hurricane")],
            "winds": ("tropical-storm-force", "hurricane-force"),
            "milestone": ("major hurricane", 96, 1)},
    "JMA": {"scale": "Japan Meteorological Agency scale", "averaging": 10,
            "classes": [(0, "Tropical Depression"), (34, "Tropical Storm"),
                        (48, "Severe Tropical Storm"), (64, "Typhoon")],
            "tracks": [(34, "Tropical storm"), (64, "Typhoon"),
                       (85, "Very strong typhoon or stronger")],
            "winds": ("tropical-storm-force", "typhoon-force"),
            # JTWC/Hong Kong usage (JMA has no "super typhoon" class).
            "milestone": ("super typhoon", 130, 1)},
    "IMD": {"scale": "India Meteorological Department scale", "averaging": 3,
            "classes": [(0, "Depression"), (28, "Deep Depression"),
                        (34, "Cyclonic Storm"), (48, "Severe Cyclonic Storm"),
                        (64, "Very Severe Cyclonic Storm"),
                        (90, "Extremely Severe Cyclonic Storm"),
                        (120, "Super Cyclonic Storm")],
            "tracks": [(34, "Cyclonic storm"), (64, "Very severe cyclonic storm"),
                       (90, "Extremely severe or stronger")],
            "winds": ("gale-force", "hurricane-force"),
            "milestone": ("extremely severe cyclonic storm", 90, 3)},
    "MFR": {"scale": "Météo-France La Réunion scale", "averaging": 10,
            "classes": [(0, "Tropical Depression"), (34, "Moderate Tropical Storm"),
                        (48, "Severe Tropical Storm"), (64, "Tropical Cyclone"),
                        (90, "Intense Tropical Cyclone"),
                        (115, "Very Intense Tropical Cyclone")],
            "tracks": [(34, "Tropical storm"), (64, "Tropical cyclone"),
                       (90, "Intense tropical cyclone")],
            "winds": ("gale-force", "hurricane-force"),
            "milestone": ("intense tropical cyclone", 90, 10)},
    "BOM": {"scale": "Australian cyclone scale", "averaging": 10,
            "classes": [(0, "Tropical Low"), (34, "Tropical Cyclone"),
                        (64, "Severe Tropical Cyclone")],
            "tracks": _AUS_TRACKS,
            "winds": ("gale-force", "very destructive"),
            "milestone": ("severe tropical cyclone", 64, 10)},
    "FMS": {"scale": "Australian cyclone scale (Fiji RSMC)", "averaging": 10,
            "classes": [(0, "Tropical Depression"), (34, "Tropical Cyclone"),
                        (64, "Severe Tropical Cyclone")],
            "tracks": _AUS_TRACKS,
            "winds": ("gale-force", "very destructive"),
            "milestone": ("severe tropical cyclone", 64, 10)},
}


def intensity_region(lat, lon, basin=None):
    """Agency key for a storm ID prefix (if known) or position."""
    if basin in ("AL", "EP", "CP"):
        return "NHC"
    if basin == "WP":
        return "JMA"
    if basin == "IO":
        return "IMD"
    lon = (float(lon) + 180) % 360 - 180
    if lat < 0:
        if 30 <= lon < 90:
            return "MFR"                      # southwest Indian Ocean
        if 90 <= lon < 160:
            return "BOM"                      # Australian region
        if lon >= 160 or lon < -60:
            return "FMS"                      # South Pacific
        return "NHC"
    if 40 <= lon < 100:
        return "IMD"
    if lon >= 100:
        return "JMA"
    return "NHC"


def agency_wind(kt, region):
    """Model 1-minute wind -> the agency's averaging period."""
    return kt * WIND_AVERAGING_FACTOR[REGIONAL_SCALES[region]["averaging"]]


def regional_class(kt, region):
    """Classification name for a 1-minute wind, e.g. 'Severe Tropical Cyclone'."""
    w = agency_wind(kt or 0, region)
    return [name for t, name in REGIONAL_SCALES[region]["classes"] if w >= t][-1]


def track_thresholds(region):
    """The three track-color thresholds as 1-minute model winds."""
    f = WIND_AVERAGING_FACTOR[REGIONAL_SCALES[region]["averaging"]]
    return [t / f for t, _ in REGIONAL_SCALES[region]["tracks"]]


def strength_label(kt, region):
    """Short category for a 1-minute wind, e.g. 'Cat. 3 hurricane',
    'Cat. 2 cyclone' (Australian scale), 'Super typhoon', 'Typhoon'."""
    if region == "NHC":
        if kt < 34:
            return "Tropical depression"
        if kt < 64:
            return "Tropical storm"
        cat = sum(kt >= t for t in (64, 83, 96, 113, 137))
        return f"Cat. {cat} hurricane"
    if region in ("BOM", "FMS"):
        w = agency_wind(kt, region)
        if w < 34:
            return "Tropical low" if region == "BOM" else "Tropical depression"
        return f"Cat. {sum(w >= t for t in (34, 48, 64, 86, 108))} cyclone"
    if region == "JMA" and kt >= 130:
        return "Super typhoon"
    name = regional_class(kt, region)
    return name[0] + name[1:].lower()


def basin_letter(basin, lon):
    lon = (float(lon) + 180) % 360 - 180
    if basin == "IO":
        return "A" if lon < 77 else "B"
    if basin == "SH":
        return "S" if 0 <= lon < 135 else "P"
    return {"AL": "L", "EP": "E", "CP": "C", "WP": "W"}.get(basin, "")


def jtwc_label(basin, num, name, kt, lat, lon):
    """'Typhoon Koguma', 'Severe Tropical Cyclone Alfred', 'Tropical Depression 28W'."""
    region = intensity_region(lat, lon, basin)
    word = regional_class(kt, region) if kt is not None else "Tropical Cyclone"
    return f"{word} {name}" if name else f"{word} {num:02d}{basin_letter(basin, lon)}"


def nhc_advisory_name(storm):
    """Storm name exactly as NHC heads its latest public advisory, e.g.
    "Post-Tropical Cyclone Josephine" or "Remnants of Josephine"."""
    url = (storm.get("publicAdvisory") or {}).get("url")
    if not url:
        return None
    try:
        text = re.sub(r"<[^>]+>", "", SESSION.get(url, timeout=10).text)
    except requests.RequestException:
        return None
    head = re.search(r"^\s*([A-Z][A-Za-z-]*(?: [A-Za-z-]+)*?)\s+"
                     r"(?:Intermediate |Special )?Advisory Number", text, re.M)
    return head.group(1).strip() if head else None


def official_storms(_cache={}):
    """Active storms from NHC and JTWC (best effort, fetched once per run).

    {'WP272026': {'label': 'Typhoon Koguma', 'named': True, 'source': 'JTWC'}}
    NHC wins for its basins (AL/EP/CP); JTWC covers everything else.
    """
    if "storms" in _cache:
        return _cache["storms"]
    storms = {}
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        rss = SESSION.get(f"{JTWC_BASE}/rss/jtwc.rss", timeout=15,
                          headers=headers).text
        for basin, num, yy in sorted(set(re.findall(
                r"products/([a-z]{2})(\d{2})(\d{2})\.tcw", rss))):
            if int(num) >= 90:                      # invests: no warning file
                continue
            try:
                lines = SESSION.get(f"{JTWC_BASE}/products/{basin}{num}{yy}.tcw",
                                    timeout=15, headers=headers).text.splitlines()
            except requests.RequestException:
                continue
            head = re.match(r"^\d{10}\s+\d{2}[A-Z]\s+(\S+)",
                            lines[2] if len(lines) > 2 else "")
            fix = re.match(r"^T000\s+(\d+)([NS])\s+(\d+)([EW])\s+(\d+)",
                           lines[3] if len(lines) > 3 else "")
            name = head.group(1) if head else None
            if name and is_number_name(name):
                name = None
            name = name.title() if name else None
            lat = lon = 0.0
            if fix:
                lat = int(fix.group(1)) / 10 * (1 if fix.group(2) == "N" else -1)
                lon = int(fix.group(3)) / 10 * (1 if fix.group(4) == "E" else -1)
            storms[f"{basin.upper()}{num}20{yy}"] = {
                "label": jtwc_label(basin.upper(), int(num), name,
                                    int(fix.group(5)) if fix else None, lat, lon),
                "named": name is not None, "source": "JTWC"}
    except Exception as exc:
        print(f"JTWC names unavailable: {exc}")
    try:
        data = SESSION.get("https://www.nhc.noaa.gov/CurrentStorms.json",
                           timeout=10).json()
        for s in data.get("activeStorms", []):
            name = str(s.get("name", "")).strip()
            cls = str(s.get("classification", "")).strip()
            storms[s["id"].upper()] = {
                "label": (nhc_advisory_name(s)
                          or f"{NHC_CLASS.get(cls, cls)} {name.title()}".strip()),
                "named": bool(name) and not is_number_name(name)
                         and not name.lower().startswith("invest"),
                "source": "NHC"}
    except Exception as exc:
        print(f"NHC names unavailable: {exc}")
    _cache["storms"] = storms
    return storms


def storm_is_named(track_ids=None, display_name=None):
    """True if any track ID is a named storm (NHC/JTWC), else judge the display name."""
    if STORM_IS_NAMED is not None:
        return bool(STORM_IS_NAMED)
    storms = official_storms()
    known = [storms[i] for i in (canonical_id(t) for t in (track_ids or []))
             if i in storms]
    if known:
        return any(s["named"] for s in known)
    return bool(re.search(r"\b(storm|hurricane|typhoon|cyclone)\b",
                          str(display_name or ""), re.I))


def print_storm_list(members_df, mean_df):
    total = members_df["sample"].nunique()
    storms = official_storms()
    wind = "maximum_sustained_wind_speed_knots"
    g = members_df.assign(base_id=members_df["track_id"].map(canonical_id))
    named = g[g["base_id"].str.match(r"^[A-Z]{2}\d{6}$")]
    genesis = g[~g.index.isin(named.index)]

    rows = []
    for base_id, grp in named.groupby("base_id"):
        first = grp.sort_values("valid_time").iloc[0]
        rows.append({
            "track_id": base_id,
            "name (NHC/JTWC)": storms.get(base_id, {}).get("label", ""),
            "raw ids": ", ".join(sorted(grp["track_id"].astype(str).unique())),
            "members": f'{grp["sample"].nunique()}/{total}',
            "peak kt (median / max)":
                f'{grp.groupby("sample")[wind].max().median():.0f} / '
                f'{grp[wind].max():.0f}' if wind in grp else "",
            "start": f'{first["lat"]:.1f}, {first["lon"]:.1f}',
        })
    print(f"\nPotential storms in {TRACK_MODEL} {RUN} ({total} members)")
    if rows:
        print(pd.DataFrame(rows).to_string(index=False))
    else:
        print("  (no existing/invest systems)")

    if not genesis.empty:
        support = (genesis.groupby("track_id")["sample"].nunique()
                   .sort_values(ascending=False))
        strong = support[support >= max(2, int(0.1 * total))]
        print(f"\nNew-genesis candidates: {len(support)} IDs; "
              f"{len(strong)} with >=10% member support")
        for tid, n in strong.items():
            grp = genesis[genesis["track_id"] == tid].sort_values("valid_time")
            print(f"  id {tid:>3}: {n}/{total} members, first seen "
                  f'{grp["lat"].iloc[0]:.1f}, {grp["lon"].iloc[0]:.1f}')
    if mean_df is not None:
        print("\nOfficial ensemble-mean IDs:",
              ", ".join(mean_df["track_id"].astype(str).unique()))


# ---------------------------------------------------------------- run
if DELETE_EXISTING_FOLDER and os.path.isdir(SAVE_DIR):
    shutil.rmtree(SAVE_DIR)
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Output folder: {SAVE_DIR}")

RUNS_TO_TRY = [RUN_OVERRIDE] if RUN_OVERRIDE else candidate_runs()
RUN = find_latest_run(lambda r: track_urls(TRACK_MODEL, r)["members"],
                      RUNS_TO_TRY)
if RUN is None:
    raise RuntimeError(f"No {TRACK_MODEL} run found in: {RUNS_TO_TRY}")
DATE_STR, TIME_STR = RUN.split("T")
print(f"Latest {TRACK_MODEL} run: {RUN}")
DATA_CREDIT_TEXT = data_credit_text(RUN)   # every map/chart credit shows the init time

urls = track_urls(TRACK_MODEL, RUN)
df = read_track_csv(download(urls["members"], output_dir("data")))
try:
    ensemble_mean_df = read_track_csv(download(urls["ensemble_mean"],
                                                output_dir("data")))
except requests.RequestException as err:
    print(f"Ensemble mean unavailable: {err}")
    ensemble_mean_df = None

# Forecast window labels (Eastern time) shared by later sections.
_valid_et = df["valid_time"].dt.tz_localize("UTC").dt.tz_convert("America/New_York")
formatted_earliest = ap_date(_valid_et.loc[df["lead_time_hours"].idxmin()])
formatted_latest = ap_date(_valid_et.loc[df["lead_time_hours"].idxmax()])

print_storm_list(df, ensemble_mean_df)


### Shared global-map style
One place for the look of both global maps (track + probability).

In [ ]:
"""Shared global-map style used by BOTH the track map and probability maps.

Change a value here and both products follow (layout, title position,
legend row, credits, colors, borders, 4K output size).
"""

import re

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.feature import ShapelyFeature
import geopandas as gpd
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.offsetbox import AnchoredOffsetbox, HPacker, TextArea, DrawingArea

GLOBAL_MAP = {
    "figsizes": {"3x2": (15, 10), "1x1": (10, 10)},
    "map_pos": {"3x2": [0.015, 0.065, 0.970, 0.825],
                "1x1": [0.015, 0.175, 0.970, 0.650]},
    "title_y": {"3x2": 0.977, "1x1": 0.887},
    "title_size": {"3x2": 31, "1x1": 28},
    "legend_y": {"3x2": 0.913, "1x1": 0.830},
    "legend_size": {"3x2": 25, "1x1": 22},
    "credit_y": {"3x2": 0.024, "1x1": 0.028},
    "credit_size": {"3x2": 12, "1x1": 9.5},
    "central_longitude": 180,
    "show_coastlines": True,             # track map
    "show_coastlines_probability": True, # probability maps (land edge)
    # Coastline underglow: (color, alpha, linewidth) layers drawn beneath the
    # coastline, widest first. Set to [] to turn off.
    "coast_glow": {
        True: [("#8fc3ff", 0.10, 4.2), ("#8fc3ff", 0.22, 2.2)],
        False: [("#5f86b3", 0.08, 3.6), ("#5f86b3", 0.16, 1.9)],
    },
    "legend_map_gap": 0.012,   # min gap (fig fraction) between key row and map
    "show_country_borders": True,
    "show_state_borders": True,
    "show_great_lakes": True,
    "great_lakes_shapefile":
        "assets/shapefiles/GL250515_lam/GL250515_lam.shp",
    "themes": {
        True: {"fig": "black", "ocean": "#0b141c", "land": "#666666",
               "coast": "#aeb5bb", "border": "#e9e9e9", "text": "#f4f4f4"},
        False: {"fig": "white", "ocean": "#ffffff", "land": "#d5d5d5",
                "coast": "#333333", "border": "#555555", "text": "black"},
    },
}

# Regional domains: [lon_min, lon_max, lat_min, lat_max]. The extent is
# widened as needed to fill the frame without distortion (Mercator).
DOMAINS = {
    "global": None,
    "Gulf of Mexico": [-104.0, -75.0, 16.0, 36.5],   # Gulf + much of the South/Southeast U.S.
    "Gulf of Mexico + Caribbean": [-98.5, -59.0, 8.5, 32.0],
    "Western Gulf": [-99.5, -87.0, 18.0, 31.0],
    "Eastern Gulf": [-92.0, -79.0, 21.0, 31.5],
}
REGIONAL_FEATURE_SCALE = "10m"   # sharper coastlines/borders when zoomed in


def is_global(domain):
    return DOMAINS[domain] is None


def domain_slug(domain):
    return re.sub(r"[^a-z0-9]+", "_", domain.lower()).strip("_")


def domain_central_longitude(domain):
    if is_global(domain):
        return GLOBAL_MAP["central_longitude"]
    lon0, lon1 = DOMAINS[domain][:2]
    return 0.5 * (lon0 + lon1)


def domain_projection(domain):
    if is_global(domain):
        return ccrs.Robinson(central_longitude=GLOBAL_MAP["central_longitude"])
    return ccrs.Mercator(central_longitude=domain_central_longitude(domain))


def domain_feature_scale(domain):
    return "50m" if is_global(domain) else REGIONAL_FEATURE_SCALE


def apply_domain_extent(ax, domain, layout=LAYOUT, pos=None):
    """Global -> set_global; regional -> extent expanded to the frame aspect."""
    if is_global(domain):
        ax.set_global()
        return
    lon0, lon1, lat0, lat1 = DOMAINS[domain]
    pts = ax.projection.transform_points(
        ccrs.PlateCarree(), np.array([lon0, lon1, lon0, lon1]),
        np.array([lat0, lat0, lat1, lat1]))
    x0, x1 = pts[:, 0].min(), pts[:, 0].max()
    y0, y1 = pts[:, 1].min(), pts[:, 1].max()
    fw, fh = GLOBAL_MAP["figsizes"][layout]
    pos = pos or GLOBAL_MAP["map_pos"][layout]
    target = (pos[2] * fw) / (pos[3] * fh)
    xc, yc, w, h = (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0
    if w / h < target:
        w = h * target
    else:
        h = w / target
    ax.set_xlim(xc - w / 2, xc + w / 2)
    ax.set_ylim(yc - h / 2, yc + h / 2)


def domain_lonlat_bounds(domain, margin=0.0):
    """Lon/lat box of the (aspect-expanded) regional frame, for filtering."""
    if is_global(domain):
        return None
    lon0, lon1, lat0, lat1 = DOMAINS[domain]
    # generous: account for aspect expansion
    dlon, dlat = lon1 - lon0, lat1 - lat0
    pad_lon, pad_lat = max(dlon, 2 * dlat) * 0.5, max(dlat, dlon / 2) * 0.5
    return (lon0 - pad_lon - margin, lon1 + pad_lon + margin,
            max(-80, lat0 - pad_lat - margin), min(80, lat1 + pad_lat + margin))


font_map_bold = load_font("FranklinITCStd-Bold.otf")
font_map_light = load_font("FranklinITCStd-Light.otf")


def global_map_dpi(layout):
    return OUTPUT_WIDTH_PX / GLOBAL_MAP["figsizes"][layout][0]


def new_global_map(layout=LAYOUT, dark_mode=None, domain="global"):
    """Figure + map axes (Robinson global or Mercator regional), ocean + land."""
    dark_mode = DARK_MODE if dark_mode is None else dark_mode
    theme = GLOBAL_MAP["themes"][dark_mode]
    fig = plt.figure(figsize=GLOBAL_MAP["figsizes"][layout], facecolor=theme["fig"])
    ax = fig.add_axes(GLOBAL_MAP["map_pos"][layout],
                      projection=domain_projection(domain))
    apply_domain_extent(ax, domain, layout)
    ax.set_facecolor(theme["ocean"])
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.add_feature(cfeature.NaturalEarthFeature(
        "physical", "land", domain_feature_scale(domain),
        facecolor=theme["land"], edgecolor="none"), zorder=1)
    return fig, ax, theme


def add_coastline_with_glow(ax, theme, dark_mode=None, scale="50m",
                            linewidth=None, zorder=100):
    """Coastline with a soft underglow (GLOBAL_MAP["coast_glow"])."""
    dark_mode = DARK_MODE if dark_mode is None else dark_mode
    for i, (color, alpha, lw) in enumerate(GLOBAL_MAP["coast_glow"][dark_mode]):
        ax.add_feature(cfeature.NaturalEarthFeature(
            "physical", "coastline", scale, facecolor="none", edgecolor=color),
            linewidth=lw, alpha=alpha, zorder=zorder - 2 + 0.1 * i)
    ax.add_feature(cfeature.NaturalEarthFeature(
        "physical", "coastline", scale, facecolor="none",
        edgecolor=theme["coast"]),
        linewidth=linewidth or (0.50 if dark_mode else 0.65), zorder=zorder)


def fit_map_below_legend(fig, ax, legend, domain="global", layout=LAYOUT):
    """Lower the map's top edge so it never touches the key row/title."""
    if legend is None:
        return
    renderer = fig.canvas.get_renderer()
    bottom = legend.get_window_extent(renderer).transformed(
        fig.transFigure.inverted()).y0
    x0, y0, w, h = GLOBAL_MAP["map_pos"][layout]
    top = min(y0 + h, bottom - GLOBAL_MAP["legend_map_gap"])
    if top >= y0 + h:
        return
    new_pos = [x0, y0, w, top - y0]
    ax.set_position(new_pos)
    apply_domain_extent(ax, domain, layout, pos=new_pos)


def add_global_overlays(ax, theme, dark_mode=None, show_coastlines=None,
                        scale="50m"):
    """Coastlines/borders/Great Lakes drawn on top of the data layer."""
    dark_mode = DARK_MODE if dark_mode is None else dark_mode
    if show_coastlines is None:
        show_coastlines = GLOBAL_MAP["show_coastlines"]
    if show_coastlines:
        add_coastline_with_glow(ax, theme, dark_mode, scale)
    if GLOBAL_MAP["show_country_borders"]:
        ax.add_feature(cfeature.NaturalEarthFeature(
            "cultural", "admin_0_boundary_lines_land", scale,
            facecolor="none", edgecolor=theme["border"]),
            linewidth=0.45, alpha=0.55 if dark_mode else 0.72, zorder=101)
    if GLOBAL_MAP["show_state_borders"]:
        ax.add_feature(cfeature.NaturalEarthFeature(
            "cultural", "admin_1_states_provinces_lines", scale,
            facecolor="none", edgecolor=theme["border"]),
            linewidth=0.35, alpha=0.40 if dark_mode else 0.62, zorder=102)
    if GLOBAL_MAP["show_great_lakes"]:
        try:
            gl = gpd.read_file(GLOBAL_MAP["great_lakes_shapefile"])
            if gl.crs is not None and gl.crs.to_epsg() != 4326:
                gl = gl.to_crs("EPSG:4326")
            show_c = show_coastlines
            ax.add_feature(ShapelyFeature(
                gl.geometry, ccrs.PlateCarree(),
                edgecolor=theme["coast"] if show_c else "none",
                facecolor=theme["ocean"], linewidth=0.25 if show_c else 0,
                zorder=103))
        except Exception as exc:
            print(f"Great Lakes outline unavailable: {exc}")


def _fits(fig, artist, max_frac=0.96):
    renderer = fig.canvas.get_renderer()
    return artist.get_window_extent(renderer).width <= max_frac * fig.bbox.width


def add_global_title(fig, text, theme, layout=LAYOUT):
    """Same position/size as the track-map title; shrinks only if too wide."""
    size = GLOBAL_MAP["title_size"][layout]
    artist = fig.text(0.5, GLOBAL_MAP["title_y"][layout], text, ha="center",
                      va="top", color=theme["text"],
                      fontproperties=font_map_bold, fontsize=size)
    while not _fits(fig, artist) and size > 12:
        size -= 1
        artist.set_fontsize(size)
    return artist


def add_global_legend_row(fig, entries, theme, layout=LAYOUT, kind="line"):
    """Centered key row beneath the title. entries = [(color, label), ...]."""
    if not entries:
        return None
    size = GLOBAL_MAP["legend_size"][layout]
    while True:
        items = []
        for color, label in entries:
            if kind == "line":
                swatch = DrawingArea(19, 12, 0, 0)
                swatch.add_artist(Line2D([0, 19], [6, 6], color=color,
                                         linewidth=3.5, solid_capstyle="round"))
            else:
                swatch = DrawingArea(24, 14, 0, 0)
                swatch.add_artist(Rectangle((0, 0), 24, 14, facecolor=color,
                                            edgecolor="none"))
            text = TextArea(label, textprops=dict(
                color=theme["text"], fontproperties=font_map_light, fontsize=size))
            items.append(HPacker(children=[swatch, text], align="center",
                                 pad=0, sep=6))
        box = AnchoredOffsetbox(
            loc="upper center",
            child=HPacker(children=items, align="center", pad=0, sep=18),
            pad=0, borderpad=0, frameon=False,
            bbox_to_anchor=(0.5, GLOBAL_MAP["legend_y"][layout]),
            bbox_transform=fig.transFigure)
        fig.add_artist(box)
        if _fits(fig, box) or size <= 10:
            return box
        box.remove()
        size -= 1


def add_global_credits(fig, theme, layout=LAYOUT, data_text=None):
    if not SHOW_CREDITS:
        return
    y, size = GLOBAL_MAP["credit_y"][layout], GLOBAL_MAP["credit_size"][layout]
    fig.text(0.018, y, data_text or DATA_CREDIT_TEXT, ha="left", va="bottom",
             color=theme["text"], fontproperties=font_map_light, fontsize=size)
    fig.text(0.982, y, BYLINE_CREDIT_TEXT, ha="right", va="bottom",
             color=theme["text"], fontproperties=font_map_light, fontsize=size)


def save_global_map(fig, path, layout=LAYOUT):
    w, h = GLOBAL_MAP["figsizes"][layout]
    print(f"Image size: {int(w * global_map_dpi(layout))} x "
          f"{int(round(h * global_map_dpi(layout)))} px")
    return save_figure(fig, path, dpi=global_map_dpi(layout),
                       facecolor=fig.get_facecolor(), edgecolor="none",
                       bbox_inches=None, pad_inches=0)


### 1. Global member tracks (always on)

In [ ]:
def run_global_tracks():
    """Global member tracks (isolated scope)."""
    """Plot all Weather Lab ensemble members and official ensemble-mean tracks.

    Run the companion downloader first so that ``df`` and ``ensemble_mean_df``
    exist, or import this file and call ``plot_all_member_tracks`` directly.
    """

    import os
    import re

    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from matplotlib import font_manager
    from matplotlib.collections import LineCollection
    from matplotlib.lines import Line2D
    from matplotlib.offsetbox import AnchoredOffsetbox, HPacker, TextArea, DrawingArea

    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    from cartopy.feature import ShapelyFeature
    import geopandas as gpd


    # ============================================================
    # User switches
    # ============================================================
    DARK_MODE = globals()["DARK_MODE"]

    LAYOUT = globals()["LAYOUT"]

    # Pacific-centered global Robinson map, matching the supplied reference.
    CENTRAL_LONGITUDE = GLOBAL_MAP["central_longitude"]

    # Map frame/style values come from GLOBAL_MAP (shared-style cell).
    SHOW_COASTLINES = GLOBAL_MAP["show_coastlines"]
    SHOW_COUNTRY_BORDERS = GLOBAL_MAP["show_country_borders"]
    SHOW_STATE_BORDERS = GLOBAL_MAP["show_state_borders"]
    SHOW_GREAT_LAKES = GLOBAL_MAP["show_great_lakes"]
    SHOW_CREDITS = globals()["SHOW_CREDITS"]

    GREAT_LAKES_SHAPEFILE = GLOBAL_MAP["great_lakes_shapefile"]

    SHOW_TITLE = True
    TITLE_TEXT = "Potential tropical cyclone tracks"
    SHOW_SUBTITLE = True
    SUBTITLE_TIMEZONE = "America/New_York"

    # Requires scipy (pip install scipy). Original forecast fixes are retained.
    SMOOTH_TRACKS = True
    SMOOTH_SUBDIVISIONS = 6  # Drawing intervals per original forecast interval.
    PLOT_DOTS = False
    PLOT_MEAN_TRACKS = True

    # Intensity is read from sustained wind speed, never inferred from track ID.
    # Set these explicitly if your downloader renames the wind column.
    WIND_COLUMN = None  # Auto-detect column names that include units.
    WIND_UNITS = "knots"  # Used only when WIND_COLUMN is explicitly set.
    SHOW_LEGEND = True
    TROPICAL_STORM_THRESHOLD_KT = 34.0
    HURRICANE_THRESHOLD_KT = 64.0
    MAJOR_HURRICANE_THRESHOLD_KT = 96.0
    HURRICANE_COLOR_LIGHT = "#dc303b"
    HURRICANE_COLOR_DARK = "#ff555f"
    MAJOR_HURRICANE_COLOR_LIGHT = "#7a1fa2"
    MAJOR_HURRICANE_COLOR_DARK = "#d77cff"
    TITLE_Y = GLOBAL_MAP["title_y"]
    TITLE_SIZES = GLOBAL_MAP["title_size"]
    SUBTITLE_SIZES = GLOBAL_MAP["legend_size"]
    LEGEND_SIZES = SUBTITLE_SIZES.copy()

    # Tropical-storm-strength colors.
    MEMBER_TRACK_COLOR_LIGHT = "#03afca"
    MEMBER_TRACK_COLOR_DARK = "#63c5ff"

    # Flat segment caps prevent rounded, overlapping ends from resembling dots.
    MEMBER_LINEWIDTH = 1.30
    MEMBER_ALPHA = 0.70
    DOT_SIZE = 8
    DOT_ALPHA = 0.70

    # Subtle yellow highlight beneath the official ensemble-mean cores.
    MEAN_GLOW_LINEWIDTH = 7.0
    MEAN_GLOW_ALPHA = 0.23
    MEAN_GLOW_COLOR = "#ffe36e"
    MEAN_CORE_LINEWIDTH = 2.3
    MEAN_CORE_ALPHA = 0.95

    DATA_CREDIT_TEXT = globals()["DATA_CREDIT_TEXT"]
    BYLINE_CREDIT_TEXT = globals()["BYLINE_CREDIT_TEXT"]

    SAVE_FIG = True
    SAVE_DIR = globals()["SAVE_DIR"]

    # Set to a full path to override the automatic output filename.
    OUTPUT_PATH = None

    # 4K output: dpi = OUTPUT_WIDTH_PX / figure width (shared-style cell).
    FIGSIZES = GLOBAL_MAP["figsizes"]
    MAP_POSITIONS = GLOBAL_MAP["map_pos"]
    CREDIT_SIZES = GLOBAL_MAP["credit_size"]


    # ============================================================
    # Franklin fonts
    # ============================================================
    FONT_BLACK = FONT_DIR + "/FranklinITCStd-Black.otf"
    FONT_BOLD = FONT_DIR + "/FranklinITCStd-Bold.otf"
    FONT_LIGHT = FONT_DIR + "/FranklinITCStd-Light.otf"


    def load_font(path, size=None):
        if os.path.exists(path):
            return font_manager.FontProperties(fname=path, size=size)
        return font_manager.FontProperties(size=size)


    font_black = load_font(FONT_BLACK)
    font_bold = load_font(FONT_BOLD)
    font_light = load_font(FONT_LIGHT)
    font_light.set_weight(300)


    # ============================================================
    # Styling and data helpers
    # ============================================================
    def get_theme(dark_mode=True):
        """Return the same palette used by the regional track plotter."""
        if dark_mode:
            return {
                **GLOBAL_MAP["themes"][True],
                "member": MEMBER_TRACK_COLOR_DARK,
                "hurricane": HURRICANE_COLOR_DARK,
                "major_hurricane": MAJOR_HURRICANE_COLOR_DARK,
                "mean": "#ffffff",
            }

        return {
            **GLOBAL_MAP["themes"][False],
            "member": MEMBER_TRACK_COLOR_LIGHT,
            "hurricane": HURRICANE_COLOR_LIGHT,
            "major_hurricane": MAJOR_HURRICANE_COLOR_LIGHT,
            "mean": "black",
        }


    def validate_track_dataframe(input_df):
        required_columns = {"track_id", "sample", "valid_time", "lat", "lon"}
        missing = sorted(required_columns - set(input_df.columns))

        if missing:
            raise ValueError(
                "The ensemble-member dataframe is missing required columns: "
                + ", ".join(missing)
            )


    def prepare_track_dataframe(input_df):
        """Clean and sort the complete ensemble-member dataframe."""
        validate_track_dataframe(input_df)

        result = input_df.copy()
        result["valid_time"] = pd.to_datetime(
            result["valid_time"],
            errors="coerce",
        )
        result["lat"] = pd.to_numeric(result["lat"], errors="coerce")
        result["lon"] = pd.to_numeric(result["lon"], errors="coerce")

        result = result.dropna(
            subset=["track_id", "sample", "valid_time", "lat", "lon"]
        )
        result = result.loc[result["lat"].between(-90, 90)].copy()
        result = result.sort_values(
            ["track_id", "sample", "valid_time"]
        )

        if result.empty:
            raise ValueError("No valid ensemble-member track points remain.")

        return result


    def prepare_mean_track_dataframe(input_df):
        """Clean official means; each unique track_id is one mean track."""
        if input_df is None:
            raise ValueError(
                "PLOT_MEAN_TRACKS is True, but ensemble_mean_df was not provided."
            )

        required_columns = {"track_id", "valid_time", "lat", "lon"}
        missing = sorted(required_columns - set(input_df.columns))
        if missing:
            raise ValueError(
                "The ensemble-mean dataframe is missing required columns: "
                + ", ".join(missing)
            )

        result = input_df.copy()
        result["valid_time"] = pd.to_datetime(
            result["valid_time"],
            errors="coerce",
        )
        result["lat"] = pd.to_numeric(result["lat"], errors="coerce")
        result["lon"] = pd.to_numeric(result["lon"], errors="coerce")
        result = result.dropna(subset=["track_id", "valid_time", "lat", "lon"])
        result = result.loc[result["lat"].between(-90, 90)].copy()
        result = result.sort_values(["track_id", "valid_time"])

        if result.empty:
            raise ValueError("No valid official ensemble-mean points remain.")

        return result


    def longitude_on_projection_branch(longitudes, central_longitude):
        """Place longitude values on the branch centered on the map."""
        values = np.asarray(longitudes, dtype=float)
        return (
            (values - central_longitude + 180.0) % 360.0
            - 180.0
            + central_longitude
        )


    def smooth_track_dataframe(frame, group_columns):
        """Interpolate drawing geometry only; never alter the input forecast table.

        PCHIP preserves the coordinate shape without cubic-spline overshoot.
        Longitude is unwrapped before interpolation for dateline crossings.
        Winds stay piecewise linear, retaining the original intensity timing.
        """
        if not SMOOTH_TRACKS:
            return frame
        from scipy.interpolate import PchipInterpolator

        count = int(SMOOTH_SUBDIVISIONS)
        if count < 1:
            raise ValueError("SMOOTH_SUBDIVISIONS must be at least 1.")
        key = group_columns[0] if len(group_columns) == 1 else group_columns
        output = []
        for _, group in frame.groupby(key, sort=False):
            group = group.sort_values("valid_time").copy()
            if len(group) < 3:
                output.append(group)
                continue
            times = pd.to_datetime(group["valid_time"], utc=True)
            t = (times - times.iloc[0]).dt.total_seconds().to_numpy()
            if np.any(np.diff(t) <= 0):
                # Ambiguous duplicate fixes: retain the supplied geometry.
                output.append(group)
                continue
            dense = np.concatenate([
                (t[:-1, None] + np.diff(t)[:, None]
                 * np.arange(count)[None, :] / count).ravel(), t[-1:]])
            owner = np.minimum(np.searchsorted(t, dense, side="right") - 1, len(t)-1)
            drawn = group.iloc[owner].copy().reset_index(drop=True)
            drawn["valid_time"] = times.iloc[0] + pd.to_timedelta(dense, unit="s")
            longitude = np.rad2deg(np.unwrap(np.deg2rad(group["lon"].to_numpy())))
            drawn["lon"] = (PchipInterpolator(t, longitude)(dense) + 180) % 360 - 180
            drawn["lat"] = PchipInterpolator(t, group["lat"].to_numpy())(dense)
            if "_wind_kt" in group:
                drawn["_wind_kt"] = np.interp(dense, t, group["_wind_kt"].to_numpy())
            output.append(drawn)
        return pd.concat(output, ignore_index=True) if output else frame.copy()


    def build_track_segments(plot_df, central_longitude, group_columns):
        """Create fast line segments, excluding jumps across the map seam."""
        segment_groups = []
        skipped_seam_segments = 0

        group_key = group_columns[0] if len(group_columns) == 1 else group_columns

        for _, group in plot_df.groupby(group_key, sort=False):
            group = group.sort_values("valid_time")

            if len(group) < 2:
                continue

            lons = longitude_on_projection_branch(
                group["lon"].to_numpy(dtype=float),
                central_longitude,
            )
            lats = group["lat"].to_numpy(dtype=float)

            start_points = np.column_stack([lons[:-1], lats[:-1]])
            end_points = np.column_stack([lons[1:], lats[1:]])

            # A jump greater than 180 degrees crosses the outer boundary of the
            # chosen Robinson map. Omitting only that segment prevents a false
            # line across the full map while retaining points on both sides.
            keep = np.abs(np.diff(lons)) <= 180.0
            skipped_seam_segments += int((~keep).sum())

            if np.any(keep):
                segment_groups.append(
                    np.stack(
                        [start_points[keep], end_points[keep]],
                        axis=1,
                    )
                )

        if not segment_groups:
            raise ValueError("No drawable track segments were found.")

        segments = np.concatenate(segment_groups, axis=0)
        return segments, skipped_seam_segments


    def sustained_wind_knots(frame):
        """Require a known unit; do not guess from the range of wind speeds."""
        factors = {"knots": 1.0, "kt": 1.0, "kts": 1.0,
                   "m/s": 1.9438444924406, "mph": 0.8689762419,
                   "km/h": 0.5399568035}
        candidates = {
            "maximum_sustained_wind_speed_knots": "knots",
            "max_wind_knots": "knots", "wind_speed_knots": "knots",
            "vmax_kt": "knots", "vmax_kts": "knots", "wind_kt": "knots",
            "maximum_sustained_wind_speed_m_s": "m/s",
            "maximum_sustained_wind_speed_mps": "m/s",
            "max_wind_mps": "m/s", "wind_speed_mps": "m/s",
        }
        column, units = WIND_COLUMN, WIND_UNITS
        if column is None:
            column = next((c for c in candidates if c in frame.columns), None)
            units = candidates.get(column)
        if column is None or column not in frame.columns:
            raise ValueError(
                "Cannot identify the sustained-wind column. Set WIND_COLUMN and "
                "WIND_UNITS near the top of this script. Available columns: "
                + ", ".join(map(str, frame.columns)))
        if units not in factors:
            raise ValueError("WIND_UNITS must be knots, kt, kts, m/s, mph, or km/h.")
        values = pd.to_numeric(frame[column], errors="coerce") * factors[units]
        values = values.where(values >= 0)
        if not values.notna().any():
            raise ValueError(f"No valid sustained wind speeds in {column}.")
        print(f"Intensity column: {column} ({units})")
        return values


    def build_intensity_segments(frame, central_longitude):
        """Split at 34/64/96 kt crossings using linear interpolation between fixes.

        Below-storm-strength and missing-intensity portions are omitted from the
        colored member layer. Neutral official mean tracks remain unchanged.
        """
        groups = {"member": [], "hurricane": [], "major_hurricane": []}
        skipped = 0
        for _, group in frame.groupby(["track_id", "sample"], sort=False):
            group = group.sort_values("valid_time")
            points = np.column_stack([
                longitude_on_projection_branch(group["lon"], central_longitude),
                group["lat"].to_numpy(dtype=float)])
            winds = group["_wind_kt"].to_numpy(dtype=float)
            for i in range(len(points) - 1):
                a, b = points[i:i+2]
                if abs(b[0] - a[0]) > 180:
                    skipped += 1
                    continue
                w0, w1 = winds[i:i+2]
                if not np.isfinite([w0, w1]).all():
                    continue
                cuts = [0.0, 1.0]
                if w1 != w0:
                    for threshold in (
                        TROPICAL_STORM_THRESHOLD_KT,
                        HURRICANE_THRESHOLD_KT,
                        MAJOR_HURRICANE_THRESHOLD_KT,
                    ):
                        t = (threshold - w0) / (w1 - w0)
                        if 0 < t < 1:
                            cuts.append(t)
                cuts.sort()
                for lo, hi in zip(cuts[:-1], cuts[1:]):
                    wind = w0 + (w1 - w0) * (lo + hi) / 2
                    if wind < TROPICAL_STORM_THRESHOLD_KT:
                        continue
                    if wind >= MAJOR_HURRICANE_THRESHOLD_KT:
                        key = "major_hurricane"
                    elif wind >= HURRICANE_THRESHOLD_KT:
                        key = "hurricane"
                    else:
                        key = "member"
                    groups[key].append([a + (b-a)*lo, a + (b-a)*hi])
        return {key: np.asarray(value, dtype=float).reshape(-1, 2, 2)
                for key, value in groups.items()}, skipped

    def add_subtitle_legend(fig, subtitle, theme, layout, represented_categories=None):
        """Center only represented intensity keys beneath the dated main title."""
        represented_categories = set(represented_categories or [])
        items = []

        if SHOW_LEGEND:
            legend_entries = [
                ("major_hurricane", theme["major_hurricane"], "Major hurricane"),
                ("hurricane", theme["hurricane"], "Hurricane"),
                ("member", theme["member"], "Tropical storm"),
            ]
            shown = [entry for entry in legend_entries if entry[0] in represented_categories]

            if shown:
                for _, color, label in shown:
                    swatch = DrawingArea(19, 12, 0, 0)
                    swatch.add_artist(Line2D(
                        [0, 19], [6, 6], color=color,
                        linewidth=3.5, solid_capstyle="round"))
                    text = TextArea(label, textprops=dict(
                        color=theme["text"], fontproperties=font_light,
                        fontsize=LEGEND_SIZES[layout]))
                    items.append(HPacker(
                        children=[swatch, text], align="center", pad=0, sep=6))

        if items:
            row = HPacker(children=items, align="center", pad=0, sep=18)
            box = AnchoredOffsetbox(
                loc="upper center", child=row, pad=0, borderpad=0, frameon=False,
                bbox_to_anchor=(0.5, GLOBAL_MAP["legend_y"][layout]),
                bbox_transform=fig.transFigure)
            fig.add_artist(box)
            return box
        return None

    def get_initialization_label(plot_df):
        """Build a concise initialization label when init_time is available."""
        if "init_time" not in plot_df.columns:
            return None

        init_times = pd.to_datetime(plot_df["init_time"], errors="coerce").dropna()
        if init_times.empty:
            return None

        init_time = init_times.iloc[0]
        return init_time.strftime("Initialized %b. %-d, %Y at %H UTC")


    def get_forecast_window_label(plot_df, timezone=SUBTITLE_TIMEZONE):
        """Format the first and last valid dates using numeric lead time."""
        if "lead_time_hours" not in plot_df.columns:
            raise ValueError(
                "SHOW_SUBTITLE is True, but df has no 'lead_time_hours' column."
            )

        valid_local = pd.to_datetime(
            plot_df["valid_time"],
            utc=True,
            errors="coerce",
        ).dt.tz_convert(timezone)
        lead_hours = pd.to_numeric(
            plot_df["lead_time_hours"],
            errors="coerce",
        )

        usable = valid_local.notna() & lead_hours.notna()
        if not usable.any():
            raise ValueError(
                "No valid time/lead-time pairs are available for the subtitle."
            )

        usable_leads = lead_hours.loc[usable]
        min_lead_idx = usable_leads.idxmin()
        max_lead_idx = usable_leads.idxmax()

        local_earliest = valid_local.loc[min_lead_idx]
        local_latest = valid_local.loc[max_lead_idx]

        month_abbr = {
            "Jan": "Jan.",
            "Feb": "Feb.",
            "Mar": "March",
            "Apr": "April",
            "May": "May",
            "Jun": "June",
            "Jul": "July",
            "Aug": "Aug.",
            "Sep": "Sept.",
            "Oct": "Oct.",
            "Nov": "Nov.",
            "Dec": "Dec.",
        }

        formatted_earliest = (
            f"{month_abbr[local_earliest.strftime('%b')]} "
            f"{local_earliest.day}"
        )
        formatted_latest = (
            f"{month_abbr[local_latest.strftime('%b')]} "
            f"{local_latest.day}"
        )

        print(
            "Earliest local time:",
            local_earliest,
            "->",
            formatted_earliest,
        )
        print(
            "Latest local time:",
            local_latest,
            "->",
            formatted_latest,
        )

        return f"{formatted_earliest} to {formatted_latest}"


    def project_segments_vectorized(segments, projection):
        """Project every segment endpoint in one fast Cartopy operation."""
        if len(segments) == 0:
            return np.empty((0, 2, 2), dtype=float)
        flat_points = segments.reshape(-1, 2)
        projected = projection.transform_points(
            ccrs.PlateCarree(),
            flat_points[:, 0],
            flat_points[:, 1],
        )[:, :2]
        projected_segments = projected.reshape(-1, 2, 2)

        finite = np.isfinite(projected_segments).all(axis=(1, 2))
        nonzero = np.any(projected_segments[:, 1] != projected_segments[:, 0], axis=1)
        return projected_segments[finite & nonzero]


    def safe_filename(text):
        cleaned = re.sub(r"[^A-Za-z0-9]+", "_", str(text)).strip("_")
        return cleaned.lower() or "all_tracks"


    def automatic_output_path(plot_df, layout, domain="global"):
        init_times = pd.to_datetime(
            plot_df.get("init_time", pd.Series(dtype="datetime64[ns]")),
            errors="coerce",
        ).dropna()

        if init_times.empty:
            run_label = "unknown_run"
        else:
            run_label = init_times.iloc[0].strftime("%Y%m%d_%H")

        filename = (
            f"weatherlab_all_member_tracks_{domain_slug(domain)}_"
            f"{safe_filename(run_label)}_{layout}.png"
        )
        return os.path.join(mode_dir("global_tracks"), filename)


    def remove_map_outline(ax):
        """Remove the Robinson outline without depending on Cartopy version."""
        for spine in ax.spines.values():
            spine.set_visible(False)

        if hasattr(ax, "outline_patch"):
            ax.outline_patch.set_visible(False)


    def load_great_lakes_feature(shapefile_path, theme, show_coastlines=False):
        """Load the custom Great Lakes polygons used by the chance maps."""
        try:
            gl_gdf = gpd.read_file(shapefile_path)

            if gl_gdf.crs is None:
                raise ValueError("the shapefile has no coordinate reference system")

            if gl_gdf.crs.to_epsg() != 4326:
                gl_gdf = gl_gdf.to_crs("EPSG:4326")

            return ShapelyFeature(
                gl_gdf.geometry,
                ccrs.PlateCarree(),
                edgecolor=theme["coast"] if show_coastlines else "none",
                facecolor=theme["ocean"],
                linewidth=0.25 if show_coastlines else 0,
                alpha=1,
                zorder=103,
            )

        except Exception as exc:
            print(f"Great Lakes outline unavailable: {exc}")
            return None


    # ============================================================
    # Main plotting function
    # ============================================================
    def plot_all_member_tracks(
        df,
        ensemble_mean_df=None,
        dark_mode=DARK_MODE,
        layout=LAYOUT,
        central_longitude=CENTRAL_LONGITUDE,
        show_country_borders=SHOW_COUNTRY_BORDERS,
        show_state_borders=SHOW_STATE_BORDERS,
        show_great_lakes=SHOW_GREAT_LAKES,
        show_credits=SHOW_CREDITS,
        show_title=SHOW_TITLE,
        title_text=TITLE_TEXT,
        show_subtitle=SHOW_SUBTITLE,
        subtitle_timezone=SUBTITLE_TIMEZONE,
        plot_dots=PLOT_DOTS,
        plot_mean_tracks=PLOT_MEAN_TRACKS,
        save_fig=SAVE_FIG,
        output_path=OUTPUT_PATH,
        show_coastlines=SHOW_COASTLINES,
        domain="global",
    ):
        """Map of every member track: global Robinson or a regional domain."""
        central_longitude = domain_central_longitude(domain)
        if layout not in FIGSIZES:
            raise ValueError("LAYOUT must be '3x2' or '1x1'.")

        plot_df = prepare_track_dataframe(df)
        theme = get_theme(dark_mode)

        great_lakes_feature = None
        if show_great_lakes:
            great_lakes_feature = load_great_lakes_feature(
                GREAT_LAKES_SHAPEFILE,
                theme,
                show_coastlines=show_coastlines,
            )

        plot_df["_wind_kt"] = sustained_wind_knots(plot_df)
        drawing_df = smooth_track_dataframe(plot_df, ["track_id", "sample"])
        intensity_segments, skipped_seam_segments = build_intensity_segments(
            drawing_df, central_longitude)
        bounds = domain_lonlat_bounds(domain)
        if bounds is not None:
            # Keep segments touching the regional frame (lon already on the
            # branch centered on the domain), so the key matches what shows.
            lo0, lo1, la0, la1 = bounds
            for key, seg in intensity_segments.items():
                if len(seg):
                    inside = ((seg[:, :, 0] >= lo0) & (seg[:, :, 0] <= lo1)
                              & (seg[:, :, 1] >= la0) & (seg[:, :, 1] <= la1))
                    intensity_segments[key] = seg[inside.any(axis=1)]
        represented_categories = {
            key for key, value in intensity_segments.items() if len(value) > 0
        }
        nonempty_segment_groups = [
            value for value in intensity_segments.values() if len(value) > 0
        ]
        if not nonempty_segment_groups:
            print(f"[{domain}] No tropical-storm-strength tracks in this domain; skipped.")
            return None, None
        segments = np.concatenate(nonempty_segment_groups, axis=0)

        mean_plot_df = None
        mean_segments = None
        mean_skipped_seam_segments = 0

        if plot_mean_tracks:
            mean_plot_df = prepare_mean_track_dataframe(ensemble_mean_df)
            mean_drawing_df = smooth_track_dataframe(mean_plot_df, ["track_id"])
            mean_segments, mean_skipped_seam_segments = build_track_segments(
                mean_drawing_df,
                central_longitude,
                group_columns=["track_id"],
            )

        unique_storms = plot_df["track_id"].nunique()
        unique_samples = plot_df["sample"].nunique()
        unique_tracks = (
            plot_df[["track_id", "sample"]]
            .drop_duplicates()
            .shape[0]
        )

        print(f"Track points: {len(plot_df):,}")
        print(f"Unique storm IDs: {unique_storms:,}")
        print(f"Unique ensemble samples: {unique_samples:,}")
        print(f"Individual storm/member tracks: {unique_tracks:,}")
        print(f"Line segments: {len(segments):,}")
        print(f"Segments split at map seam: {skipped_seam_segments:,}")

        if mean_plot_df is not None:
            mean_track_ids = mean_plot_df["track_id"].dropna().unique()
            print(f"Official ensemble-mean track IDs: {len(mean_track_ids):,}")
            for track_id in mean_track_ids:
                print(f"  {track_id}")
            print(f"Official mean segments: {len(mean_segments):,}")
            print(
                "Official mean segments split at map seam: "
                f"{mean_skipped_seam_segments:,}"
            )

        fig = plt.figure(
            figsize=FIGSIZES[layout],
            facecolor=theme["fig"],
        )

        projection = domain_projection(domain)
        projected_segments = project_segments_vectorized(
            segments,
            projection,
        )
        print(f"Projected line segments: {len(projected_segments):,}")

        projected_mean_segments = None
        if mean_segments is not None:
            projected_mean_segments = project_segments_vectorized(
                mean_segments,
                projection,
            )
            print(
                "Projected official mean segments: "
                f"{len(projected_mean_segments):,}"
            )

        ax = fig.add_axes(
            MAP_POSITIONS[layout],
            projection=projection,
        )

        apply_domain_extent(ax, domain, layout)
        ax.set_facecolor(theme["ocean"])
        remove_map_outline(ax)

        land = cfeature.NaturalEarthFeature(
            "physical",
            "land",
            domain_feature_scale(domain),
            facecolor=theme["land"],
            edgecolor="none",
        )
        coastlines = cfeature.NaturalEarthFeature(
            "physical",
            "coastline",
            domain_feature_scale(domain),
            facecolor="none",
            edgecolor=theme["coast"],
        )
        countries = cfeature.NaturalEarthFeature(
            "cultural",
            "admin_0_boundary_lines_land",
            domain_feature_scale(domain),
            facecolor="none",
            edgecolor=theme["border"],
        )
        states_provinces = cfeature.NaturalEarthFeature(
            "cultural",
            "admin_1_states_provinces_lines",
            domain_feature_scale(domain),
            facecolor="none",
            edgecolor=theme["border"],
        )

        ax.add_feature(land, zorder=1)

        for key, category_segments in intensity_segments.items():
            if len(category_segments) == 0:
                continue
            zorders = {"member": 30, "hurricane": 31, "major_hurricane": 32}
            track_collection = LineCollection(
                project_segments_vectorized(category_segments, projection),
                colors=theme[key], linewidths=MEMBER_LINEWIDTH,
                alpha=MEMBER_ALPHA, transform=ax.transData,
                zorder=zorders[key],
                capstyle="butt", joinstyle="round")
            ax.add_collection(track_collection)

        if projected_mean_segments is not None:
            mean_glow = LineCollection(
                projected_mean_segments,
                colors=MEAN_GLOW_COLOR,
                linewidths=MEAN_GLOW_LINEWIDTH,
                alpha=MEAN_GLOW_ALPHA,
                transform=ax.transData,
                zorder=45,
                capstyle="butt",
                joinstyle="round",
            )
            ax.add_collection(mean_glow)

            mean_core = LineCollection(
                projected_mean_segments,
                colors=theme["mean"],
                linewidths=MEAN_CORE_LINEWIDTH,
                alpha=MEAN_CORE_ALPHA,
                transform=ax.transData,
                zorder=46,
                capstyle="butt",
                joinstyle="round",
            )
            ax.add_collection(mean_core)

        if plot_dots:
            dot_lons = longitude_on_projection_branch(
                plot_df["lon"].to_numpy(dtype=float),
                central_longitude,
            )
            ax.scatter(
                dot_lons,
                plot_df["lat"],
                s=np.where(plot_df["_wind_kt"] >= TROPICAL_STORM_THRESHOLD_KT, DOT_SIZE, 0),
                c=np.where(
                    plot_df["_wind_kt"] >= MAJOR_HURRICANE_THRESHOLD_KT,
                    theme["major_hurricane"],
                    np.where(
                        plot_df["_wind_kt"] >= HURRICANE_THRESHOLD_KT,
                        theme["hurricane"], theme["member"]
                    ),
                ),
                linewidth=0,
                alpha=DOT_ALPHA,
                transform=ccrs.PlateCarree(),
                zorder=32,
            )

        if show_coastlines:
            add_coastline_with_glow(
                ax, theme, dark_mode, domain_feature_scale(domain), zorder=100)

        if show_country_borders:
            ax.add_feature(
                countries,
                linewidth=0.45,
                alpha=0.55 if dark_mode else 0.72,
                zorder=101,
            )

        if show_state_borders:
            ax.add_feature(
                states_provinces,
                linewidth=0.35,
                alpha=0.40 if dark_mode else 0.62,
                zorder=102,
            )

        if great_lakes_feature is not None:
            ax.add_feature(great_lakes_feature)

        # Put the forecast date range directly in the main title. SHOW_SUBTITLE is
        # retained as the switch controlling whether the date range is displayed.
        forecast_window_text = (
            get_forecast_window_label(plot_df, timezone=subtitle_timezone)
            if show_subtitle else None
        )
        display_title = title_text
        if forecast_window_text:
            display_title = f"{title_text} • {forecast_window_text}"

        if show_title:
            fig.text(
                0.5,
                TITLE_Y[layout],
                display_title,
                ha="center",
                va="top",
                color=theme["text"],
                fontproperties=font_bold,
                fontsize=TITLE_SIZES[layout],
            )

        # With the dates moved into the title, this row contains only the dynamic
        # legend entries and remains centered beneath the title.
        legend_box = add_subtitle_legend(
            fig, None, theme, layout,
            represented_categories=represented_categories,
        )
        fit_map_below_legend(fig, ax, legend_box, domain, layout)

        if show_credits:
            credit_size = CREDIT_SIZES[layout]
            credit_y = GLOBAL_MAP["credit_y"][layout]

            fig.text(
                0.018,
                credit_y,
                DATA_CREDIT_TEXT,
                ha="left",
                va="bottom",
                color=theme["text"],
                fontproperties=font_light,
                fontsize=credit_size,
            )
            fig.text(
                0.982,
                credit_y,
                BYLINE_CREDIT_TEXT,
                ha="right",
                va="bottom",
                color=theme["text"],
                fontproperties=font_light,
                fontsize=credit_size,
            )

        if save_fig:
            if output_path is None:
                output_path = automatic_output_path(plot_df, layout, domain)

            output_path = save_figure(
                fig,
                output_path,
                dpi=global_map_dpi(layout),
                facecolor=fig.get_facecolor(),
                edgecolor="none",
                bbox_inches=None,
                pad_inches=0,
            )

        plt.show()
        return fig, ax


    # ============================================================
    # Run when df exists in the notebook/session
    # ============================================================
    if "df" in globals():
        if PLOT_MEAN_TRACKS and "ensemble_mean_df" not in globals():
            raise RuntimeError(
                "PLOT_MEAN_TRACKS is True, but ensemble_mean_df was not found. "
                "Run the updated Weather Lab downloader first."
            )

        for map_domain in MAP_DOMAINS:
            print(f"\n=== Track map: {map_domain} ===")
            fig, ax = plot_all_member_tracks(
                df,
                domain=map_domain,
                ensemble_mean_df=globals().get("ensemble_mean_df"),
                dark_mode=DARK_MODE,
                layout=LAYOUT,
                central_longitude=CENTRAL_LONGITUDE,
                show_coastlines=SHOW_COASTLINES,
                show_country_borders=SHOW_COUNTRY_BORDERS,
                show_state_borders=SHOW_STATE_BORDERS,
                show_great_lakes=SHOW_GREAT_LAKES,
                show_credits=SHOW_CREDITS,
                show_title=SHOW_TITLE,
                title_text=TITLE_TEXT,
                show_subtitle=SHOW_SUBTITLE,
                subtitle_timezone=SUBTITLE_TIMEZONE,
                plot_dots=PLOT_DOTS,
                plot_mean_tracks=PLOT_MEAN_TRACKS,
                save_fig=SAVE_FIG,
                output_path=OUTPUT_PATH,
            )
    else:
        print(
            "No dataframe named df found. Run the Weather Lab downloader first, "
            "or import this file and call "
            "plot_all_member_tracks(df, ensemble_mean_df)."
        )


for_each_color_mode(run_global_tracks)


### 2. Cumulative probabilities (always on)
Gridded `cumulative_probability_fields` product. Same frame, title, key row, credits and 4K size as the track map (shared style cell).

Also writes Datawrapper GeoJSON files (`EXPORT_DATAWRAPPER_GEOJSON`) for the tropical-storm- and hurricane-force wind chances.

In [ ]:
"""Cumulative probability maps from Weather Lab's gridded product.

Downloads <model>_<run>_cumulative_probability_fields.nc.gz.base64, decodes it
to netCDF (dims: max_lead_time x lat x lon, values 0-1), keeps only the chosen
lead-time slice, saves a small compressed copy to data/ and draws the maps.
"""

import base64
import gzip

import xarray as xr
import matplotlib.colors as mcolors
from scipy.ndimage import gaussian_filter, zoom
import cartopy.crs as ccrs

# ---------------------------------------------------------------- settings
CUMPROB_PRODUCTS = {
    # variable in file              : (switch, legend title, filename tag)
    "34_knot_strike_probability": (CUMPROB_SHOW_TROPICAL_STORM,
                                   "Chance of tropical-storm-force winds",
                                   "tropical_storm"),
    "64_knot_strike_probability": (CUMPROB_SHOW_HURRICANE,
                                   "Chance of hurricane-force winds",
                                   "hurricane"),
    "50_knot_strike_probability": (CUMPROB_SHOW_50KT,
                                   "Chance of 50-knot winds",
                                   "50kt"),
    "track_probability": (CUMPROB_SHOW_TRACK,
                          "Chance of a tropical cyclone nearby",
                          "track"),
}

# Chance categories (percent). Shading starts at CHANCE_FLOOR.
CHANCE_FLOOR = 5.0
CHANCE_BOUNDS = [CHANCE_FLOOR, 30, 60, 100.01]
CHANCE_LABELS = ["Some (5%-30%)", "Medium (30%-60%)", "High (60%+)"]
CHANCE_COLORS = ["#f2b4ab", "#d35f4f", "#8e1f1b"]
SMOOTH_SIGMA = 1.0                         # grid cells (0.25 deg); 0 = off
REGIONAL_UPSAMPLE = 5   # regional maps: interpolate 0.25 deg -> 0.05 deg for smooth edges
# Map frame, title, legend row, credits, colors and 4K size come from
# GLOBAL_MAP in the shared-style cell, so they match the track map.


# ---------------------------------------------------------------- data
def fetch_cumulative_fields():
    """Return (model, run, local .nc path) for the best available product."""
    for model in CUMPROB_MODEL_PREFERENCE:
        runs = [RUN] + [r for r in candidate_runs() if r != RUN]
        found = find_latest_run(lambda r: cumprob_url(model, r), runs)
        if found:
            break
    else:
        raise RuntimeError("No cumulative probability file found for "
                           f"{CUMPROB_MODEL_PREFERENCE}")
    if model != TRACK_MODEL or found != RUN:
        print(f"Note: cumulative fields from {model} {found} "
              f"(tracks are {TRACK_MODEL} {RUN})")

    b64_path = download(cumprob_url(model, found), output_dir("data"))
    nc_path = b64_path.replace(".nc.gz.base64", ".nc")
    with open(b64_path, "rb") as f:
        gz_bytes = base64.b64decode(f.read())
    os.remove(b64_path)
    gz_path = nc_path + ".gz"
    with open(gz_path, "wb") as f:
        f.write(gz_bytes)
    del gz_bytes
    with gzip.open(gz_path, "rb") as src, open(nc_path, "wb") as dst:
        shutil.copyfileobj(src, dst, length=16 << 20)
    os.remove(gz_path)
    return model, found, nc_path


def load_cumulative_slice(nc_path, max_lead_hours=None):
    """Select one max_lead_time slice for every probability variable."""
    with xr.open_dataset(nc_path) as ds:
        lead = ds["max_lead_time"].values
        if np.issubdtype(lead.dtype, np.timedelta64):
            lead_h = (lead / np.timedelta64(1, "h")).astype(int)
        else:                                  # undecoded -> already hours
            lead_h = lead.astype(int)
        idx = (len(lead_h) - 1 if max_lead_hours is None
               else int(np.argmin(np.abs(lead_h - max_lead_hours))))
        prob_vars = [v for v in ds.data_vars if "probability" in v]
        sliced = ds[prob_vars].isel(max_lead_time=idx).load()
        init = pd.Timestamp(ds["init_time"].values)
    sliced = (sliced * 100).astype("float32")
    sliced.attrs.update(units="percent", init_time=str(init),
                        max_lead_time_hours=int(lead_h[idx]))
    return sliced, init, int(lead_h[idx])


# ---------------------------------------------------------------- plotting
def draw_cumprob_map(field_pct, lat, lon, title, window_text, out_path,
                     domain="global"):
    data = np.nan_to_num(field_pct, nan=0.0)
    if SMOOTH_SIGMA:
        data = gaussian_filter(data, sigma=SMOOTH_SIGMA, mode=("nearest", "wrap"))
    bounds = domain_lonlat_bounds(domain, margin=2.0)
    if bounds is not None:
        # Regional: switch to -180..180 longitudes and crop (Mercator cannot
        # take the +/-90 latitude edges of the global grid).
        order = np.argsort(((lon + 180.0) % 360.0) - 180.0)
        lon = ((lon[order] + 180.0) % 360.0) - 180.0
        data = data[:, order]
        lo0, lo1, la0, la1 = bounds
        keep_x = (lon >= lo0) & (lon <= lo1)
        keep_y = (lat >= la0) & (lat <= la1)
        lon, lat = lon[keep_x], lat[keep_y]
        data = data[np.ix_(keep_y, keep_x)]
        if REGIONAL_UPSAMPLE and REGIONAL_UPSAMPLE > 1:
            data = np.clip(zoom(data, REGIONAL_UPSAMPLE, order=1), 0, 100)
            lon = np.linspace(lon[0], lon[-1], data.shape[1])
            lat = np.linspace(lat[0], lat[-1], data.shape[0])

    data = np.where(data < CHANCE_FLOOR, np.nan, data)

    # Cell edges: lon 0..360 keeps the 180-degree seam inside the mesh.
    dlon, dlat = float(lon[1] - lon[0]), float(lat[1] - lat[0])
    lon_edges = np.append(lon - dlon / 2, lon[-1] + dlon / 2)
    lat_edges = np.clip(np.append(lat - dlat / 2, lat[-1] + dlat / 2), -90, 90)

    cmap = mcolors.ListedColormap(CHANCE_COLORS)
    cmap.set_bad(alpha=0)
    norm = mcolors.BoundaryNorm(CHANCE_BOUNDS, cmap.N)

    fig, ax, theme = new_global_map(LAYOUT, DARK_MODE, domain)
    ax.pcolormesh(lon_edges, lat_edges, data, cmap=cmap, norm=norm,
                  shading="flat", alpha=0.96, antialiased=False,
                  transform=ccrs.PlateCarree(), zorder=50)
    add_global_overlays(ax, theme, DARK_MODE,
                        show_coastlines=GLOBAL_MAP["show_coastlines_probability"],
                        scale=domain_feature_scale(domain))
    add_global_title(fig, f"{title} • {window_text}", theme, LAYOUT)
    legend = add_global_legend_row(fig, list(zip(CHANCE_COLORS, CHANCE_LABELS)),
                                   theme, LAYOUT, kind="box")
    fit_map_below_legend(fig, ax, legend, domain, LAYOUT)
    add_global_credits(fig, theme, LAYOUT, data_text=data_credit_text(cp_run))
    save_global_map(fig, out_path, LAYOUT)
    plt.show()
    plt.close(fig)


# ---------------------------------------------------------------- run
cp_model, cp_run, cp_nc_path = fetch_cumulative_fields()
cumprob_ds, cp_init, cp_lead = load_cumulative_slice(cp_nc_path,
                                                     CUMPROB_MAX_LEAD_HOURS)

compact_path = os.path.join(
    output_dir("data"), f"{cp_model}_{cp_run}_cumulative_probability_{cp_lead:03d}h.nc")
cumprob_ds.to_netcdf(compact_path, encoding={
    v: {"zlib": True, "complevel": 4} for v in cumprob_ds.data_vars})
print(f"Saved compact fields: {compact_path}")
if not KEEP_RAW_CUMPROB_FILE:
    os.remove(cp_nc_path)

for v in cumprob_ds.data_vars:
    print(f"  {v:28s} max {float(cumprob_ds[v].max()):5.1f}%")

init_et = cp_init.tz_localize("UTC").tz_convert("America/New_York")
end_et = init_et + pd.Timedelta(hours=cp_lead)
cp_window = f"{ap_date(init_et)} to {ap_date(end_et)}"

def draw_all_cumprob_maps():
    for map_domain in MAP_DOMAINS:
        for var, (enabled, title, tag) in CUMPROB_PRODUCTS.items():
            if not enabled:
                continue
            if var not in cumprob_ds:
                print(f"Skipping {var}: not in file")
                continue
            draw_cumprob_map(
                cumprob_ds[var].values, cumprob_ds["lat"].values,
                cumprob_ds["lon"].values, title, cp_window,
                os.path.join(mode_dir("probability_maps"),
                             f"cumulative_{tag}_chances_{domain_slug(map_domain)}_"
                             f"{cp_model}_{cp_run}_{cp_lead:03d}h_{LAYOUT}"),
                domain=map_domain)


for_each_color_mode(draw_all_cumprob_maps)


# ---------------------------------------------------------------- Datawrapper
# GeoJSON polygons of the chance categories for Datawrapper locator maps
# (Add area markers -> import GeoJSON). WGS84 lon/lat, -180..180, RFC 7946
# ring order. Each feature carries its category, percent range and fill color.
import json
from matplotlib.path import Path as MplPath

DATAWRAPPER_PRODUCTS = {
    "34_knot_strike_probability": ("tropical_storm", "Tropical-storm-force winds"),
    "64_knot_strike_probability": ("hurricane", "Hurricane-force winds"),
}


def _signed_area(ring):
    x, y = ring[:, 0], ring[:, 1]
    return 0.5 * np.sum(x[:-1] * y[1:] - x[1:] * y[:-1])


def _band_polygons(lon, lat, data, lo, hi):
    """Contour one band (lo <= value < hi) into GeoJSON polygon coordinates."""
    fig = plt.figure()
    ax = fig.add_subplot()
    cs = ax.contourf(lon, lat, data, levels=[lo, hi])
    if hasattr(cs, "collections") and not hasattr(cs, "get_paths"):
        paths = [p for c in cs.collections for p in c.get_paths()]
    else:
        paths = cs.get_paths()
    plt.close(fig)

    rings = []
    for path in paths:
        for ring in path.to_polygons(closed_only=True):
            ring = np.round(np.asarray(ring, float), DATAWRAPPER_DECIMALS)
            keep = np.r_[True, np.any(np.diff(ring, axis=0) != 0, axis=1)]
            ring = ring[keep]
            if len(ring) < 4 or abs(_signed_area(ring)) < DATAWRAPPER_MIN_AREA_DEG2:
                continue
            if not np.array_equal(ring[0], ring[-1]):
                ring = np.vstack([ring, ring[:1]])
            rings.append(ring)
    if not rings:
        return []

    # Nesting depth decides shell (even) vs hole (odd); holes join their parent.
    areas = np.array([abs(_signed_area(r)) for r in rings])
    order = np.argsort(-areas)
    rings = [rings[i] for i in order]
    mpl_paths = [MplPath(r) for r in rings]
    boxes = [(r[:, 0].min(), r[:, 0].max(), r[:, 1].min(), r[:, 1].max()) for r in rings]
    parent, depth = [-1] * len(rings), [0] * len(rings)
    for i, ring in enumerate(rings):
        px, py = ring[0]
        for j in range(i - 1, -1, -1):          # smallest enclosing ring first
            x0, x1, y0, y1 = boxes[j]
            if x0 <= px <= x1 and y0 <= py <= y1 and mpl_paths[j].contains_point((px, py)):
                parent[i], depth[i] = j, depth[j] + 1
                break

    polygons = {}
    for i, ring in enumerate(rings):
        if depth[i] % 2 == 0:                    # shell: counterclockwise
            r = ring if _signed_area(ring) > 0 else ring[::-1]
            polygons[i] = [r.tolist()]
    for i, ring in enumerate(rings):
        if depth[i] % 2 == 1 and parent[i] in polygons:   # hole: clockwise
            r = ring if _signed_area(ring) < 0 else ring[::-1]
            polygons[parent[i]].append(r.tolist())
    return list(polygons.values())


def export_datawrapper_geojson(var, tag, product_label):
    """Write one GeoJSON FeatureCollection (one feature per chance category)."""
    lon = cumprob_ds["lon"].values.astype(float)
    lat = cumprob_ds["lat"].values.astype(float)
    data = np.nan_to_num(cumprob_ds[var].values.astype(float), nan=0.0)
    if SMOOTH_SIGMA:
        data = gaussian_filter(data, sigma=SMOOTH_SIGMA, mode=("nearest", "wrap"))
    data = np.clip(data, 0, 100)

    # -180..180 with a wrapped column at +180 so areas close at the date line.
    lon180 = ((lon + 180.0) % 360.0) - 180.0
    order = np.argsort(lon180)
    lon180, data = lon180[order], data[:, order]
    lon180 = np.append(lon180, 180.0)
    data = np.hstack([data, data[:, :1]])

    edges = CHANCE_BOUNDS
    if DATAWRAPPER_BANDS == "cumulative":
        bands = [(edges[i], edges[-1]) for i in range(len(edges) - 1)]
    else:
        bands = [(edges[i], edges[i + 1]) for i in range(len(edges) - 1)]

    features = []
    for (lo, hi), label, color in zip(bands, CHANCE_LABELS, CHANCE_COLORS):
        polys = _band_polygons(lon180, lat, data, lo, hi)
        if not polys:
            continue
        category = label.split(" (")[0]
        features.append({
            "type": "Feature",
            "properties": {
                "title": f"{product_label}: {label}",
                "product": product_label,
                "category": category,
                "label": label,
                "min_pct": lo,
                "max_pct": min(hi, 100),
                "band_type": DATAWRAPPER_BANDS,
                "model": cp_model,
                "run": cp_run,
                "window": cp_window,
                "max_lead_hours": cp_lead,
                # simplestyle-spec styling hints
                "fill": color,
                "fill-opacity": 0.85,
                "stroke": color,
                "stroke-width": 0,
                "stroke-opacity": 0,
            },
            "geometry": ({"type": "Polygon", "coordinates": polys[0]}
                         if len(polys) == 1 else
                         {"type": "MultiPolygon", "coordinates": polys}),
        })

    out_path = os.path.join(
        output_dir("datawrapper"), f"datawrapper_{tag}_wind_chances_{cp_model}_{cp_run}_"
                  f"{cp_lead:03d}h_{DATAWRAPPER_BANDS}.geojson")
    with open(out_path, "w") as f:
        json.dump({"type": "FeatureCollection", "features": features}, f,
                  separators=(",", ":"))
    n_rings = sum(len(p) for ft in features for p in
                  ([ft["geometry"]["coordinates"]] if ft["geometry"]["type"] == "Polygon"
                   else ft["geometry"]["coordinates"]))
    print(f"Datawrapper GeoJSON ({len(features)} categories, {n_rings} rings, "
          f"{os.path.getsize(out_path) / 1e6:.2f} MB): {out_path}")
    return out_path


if EXPORT_DATAWRAPPER_GEOJSON:
    for var, (tag, label) in DATAWRAPPER_PRODUCTS.items():
        if var in cumprob_ds:
            export_datawrapper_geojson(var, tag, label)
        else:
            print(f"Datawrapper export skipped: {var} not in file")


### 3. U.S. landfall odds: CONUS + by state (`RUN_LANDFALL_ODDS`)
WeatherNext 3 (`WNV3`) and the 1,000-member `FNV3_LARGE_ENSEMBLE` tracks (`LANDFALL_MODELS`). A member counts once per area if any of its tracks crosses from water onto land there at the given strength; odds = members / total members. A side-by-side comparison follows the per-model results.

In [ ]:
def run_landfall_odds():
    """U.S. landfall odds (CONUS + by state) for each model in LANDFALL_MODELS."""
    import cartopy.io.shapereader as shpreader
    import shapely
    from shapely.geometry import LineString, Point
    from shapely.ops import unary_union
    from shapely.prepared import prep

    TS_KT, HU_KT = 34.0, 64.0
    TOP_N = LANDFALL_TOP_STATES
    # Coarse boxes around U.S. land; only segments touching these are tested.
    BOXES = [(-127, -64, 23, 50.5), (-162, -153, 17, 24), (-180, -129, 50, 72)]
    STATE_POSTAL = {
        "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR",
        "California": "CA", "Colorado": "CO", "Connecticut": "CT",
        "Delaware": "DE", "District of Columbia": "DC", "Florida": "FL",
        "Georgia": "GA", "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL",
        "Indiana": "IN", "Iowa": "IA", "Kansas": "KS", "Kentucky": "KY",
        "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
        "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN",
        "Mississippi": "MS", "Missouri": "MO", "Montana": "MT",
        "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH",
        "New Jersey": "NJ", "New Mexico": "NM", "New York": "NY",
        "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
        "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA",
        "Rhode Island": "RI", "South Carolina": "SC", "South Dakota": "SD",
        "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
        "Virginia": "VA", "Washington": "WA", "West Virginia": "WV",
        "Wisconsin": "WI", "Wyoming": "WY",
    }

    # ------------------------------------------------ U.S. geometry
    shp = shpreader.natural_earth(resolution="10m", category="cultural",
                                  name="admin_1_states_provinces")
    pieces = {}
    for rec in shpreader.Reader(shp).records():
        a = rec.attributes
        if str(a.get("iso_a2", a.get("adm0_a3", ""))).upper() not in ("US", "USA") \
                and str(a.get("adm0_a3", "")).upper() != "USA":
            continue
        code = STATE_POSTAL.get(str(a.get("name", "")).strip())
        if code:
            pieces.setdefault(code, []).append(rec.geometry)
    states = {c: unary_union(g) for c, g in pieces.items()}
    codes = list(states)
    us_land = unary_union(list(states.values()))
    us_prep = prep(us_land)
    state_tree = shapely.STRtree([states[c] for c in codes])
    conus_codes = set(codes) - {"AK", "HI"}

    # ------------------------------------------------ data (per model)
    def load_tracks(MODEL):
        if MODEL == TRACK_MODEL:              # already downloaded in cell 1
            preferred = RUN
        elif globals().get("cp_model") == MODEL:
            preferred = globals().get("cp_run")
        else:
            preferred = None
        runs = ([preferred] if preferred else []) + [r for r in candidate_runs() if r != preferred]
        run = find_latest_run(lambda r: track_urls(MODEL, r)["members"], runs)
        if run is None:
            print(f"No {MODEL} cyclogenesis CSV found; skipping.")
            return None
        path = os.path.join(output_dir("data"), f"{MODEL}_{run}_cyclogenesis.csv")
        if not os.path.exists(path):
            path = download(track_urls(MODEL, run)["members"], output_dir("data"))
        cols = ["track_id", "sample", "valid_time", "lat", "lon",
                "maximum_sustained_wind_speed_knots"]
        tracks = pd.read_csv(path, comment="#", usecols=cols,
                             dtype={"track_id": str}, parse_dates=["valid_time"])
        total_members = tracks["sample"].nunique()
        if LANDFALL_TRACK_IDS:
            wanted = {canonical_id(t) for t in LANDFALL_TRACK_IDS}
            tracks = tracks[tracks["track_id"].map(canonical_id).isin(wanted)]
            scope = ", ".join(sorted(wanted))
        else:
            scope = "all systems"
        print(f"{MODEL} {run}: {len(tracks):,} track points, "
              f"{total_members:,} ensemble members ({scope})")
        return run, tracks, total_members, scope

    def landfall_odds(MODEL):
        loaded = load_tracks(MODEL)
        if loaded is None:
            return None
        run, tracks, total_members, scope = loaded

        # ------------------------------------------------ vectorized segments
        t = tracks.sort_values(["track_id", "sample", "valid_time"]).reset_index(drop=True)
        lon = ((t["lon"].to_numpy(float) + 180) % 360) - 180
        lat = t["lat"].to_numpy(float)
        wind = pd.to_numeric(t["maximum_sustained_wind_speed_knots"],
                             errors="coerce").to_numpy(float)
        same = ((t["track_id"].to_numpy()[1:] == t["track_id"].to_numpy()[:-1])
                & (t["sample"].to_numpy()[1:] == t["sample"].to_numpy()[:-1]))
        in_box = np.zeros(len(t), bool)
        for x0, x1, y0, y1 in BOXES:
            in_box |= (lon >= x0 - 1) & (lon <= x1 + 1) & (lat >= y0 - 1) & (lat <= y1 + 1)
        seg = np.where(same & (in_box[:-1] | in_box[1:])
                       & (np.abs(np.diff(lon)) <= 180))[0]
        near = np.unique(np.concatenate([seg, seg + 1]))
        on_land = np.zeros(len(t), bool)
        on_land[near] = shapely.contains_xy(us_land, lon[near], lat[near])
        seg = seg[~on_land[seg]]                      # landfall starts offshore
        print(f"Testing {len(seg):,} offshore segments near U.S. land...")

        events = []
        for i in seg:
            line = LineString([(lon[i], lat[i]), (lon[i + 1], lat[i + 1])])
            if not (on_land[i + 1] or us_prep.intersects(line)):
                continue
            overlap = line.intersection(us_land)
            if overlap.is_empty:
                continue
            fracs = [line.project(Point(c), normalized=True)
                     for g in getattr(overlap, "geoms", [overlap])
                     if g.geom_type in ("LineString", "Point")
                     for c in (g.coords[0], g.coords[-1])]
            if not fracs:
                continue
            frac = min(fracs)
            w0, w1 = wind[i], wind[i + 1]
            w = (w0 + frac * (w1 - w0)) if np.isfinite([w0, w1]).all() else np.nanmax([w0, w1])
            if not np.isfinite(w):
                continue
            pt = line.interpolate(frac, normalized=True)
            state = codes[int(state_tree.query_nearest(pt)[0])]
            t0, t1 = t.at[i, "valid_time"], t.at[i + 1, "valid_time"]
            events.append({
                "track_id": t.at[i, "track_id"], "sample": t.at[i, "sample"],
                "state": state, "landfall_time": t0 + frac * (t1 - t0),
                "landfall_lat": pt.y, "landfall_lon": pt.x, "landfall_wind_kt": w,
                "intensity": ("hurricane" if w >= HU_KT else
                              "tropical storm" if w >= TS_KT else "below tropical storm"),
            })
        ev = pd.DataFrame(events, columns=[
            "track_id", "sample", "state", "landfall_time", "landfall_lat",
            "landfall_lon", "landfall_wind_kt", "intensity"])

        # ------------------------------------------------ odds (per ensemble member)
        def pct(mask_df):
            return 100.0 * mask_df["sample"].nunique() / total_members

        conus = ev[ev["state"].isin(conus_codes)]
        summary = pd.DataFrame([{
            "region": "contiguous United States", "model": MODEL, "run": run,
            "scope": scope, "total_members": total_members,
            "tropical_storm_or_stronger_pct": pct(conus[conus.landfall_wind_kt >= TS_KT]),
            "hurricane_pct": pct(conus[conus.landfall_wind_kt >= HU_KT]),
        }])

        rows = []
        for code, grp in ev.groupby("state"):
            rows.append({
                "state": code,
                "tropical_storm_or_stronger_pct": pct(grp[grp.landfall_wind_kt >= TS_KT]),
                "hurricane_pct": pct(grp[grp.landfall_wind_kt >= HU_KT]),
                "members_ts_plus": grp.loc[grp.landfall_wind_kt >= TS_KT, "sample"].nunique(),
                "members_hurricane": grp.loc[grp.landfall_wind_kt >= HU_KT, "sample"].nunique(),
            })
        by_state = (pd.DataFrame(rows, columns=["state", "tropical_storm_or_stronger_pct",
                                                "hurricane_pct", "members_ts_plus",
                                                "members_hurricane"])
                    .query("members_ts_plus > 0")
                    .sort_values(["tropical_storm_or_stronger_pct", "hurricane_pct"],
                                 ascending=False))

        s = summary.iloc[0]
        print(f"\n{MODEL} CONUS landfall odds ({total_members:,} members):")
        print(f"  Tropical storm or stronger: {s.tropical_storm_or_stronger_pct:.1f}%")
        print(f"  Hurricane:                  {s.hurricane_pct:.1f}%")
        if by_state.empty:
            print("\nNo state shows a tropical-storm-strength landfall.")
        else:
            top = by_state.head(TOP_N)
            print(f"\n{MODEL} states with the highest landfall odds (top {len(top)}):")
            print(top.rename(columns={
                "tropical_storm_or_stronger_pct": "TS+ %",
                "hurricane_pct": "hurricane %",
                "members_ts_plus": "TS+ members",
                "members_hurricane": "hurricane members"})
                .to_string(index=False, float_format=lambda v: f"{v:.1f}"))

        stem = f"{MODEL}_{run}"
        summary.to_csv(os.path.join(output_dir("landfall_odds"), f"conus_landfall_odds_{stem}.csv"),
                       index=False, float_format="%.1f")
        by_state.to_csv(os.path.join(output_dir("landfall_odds"), f"state_landfall_odds_{stem}.csv"),
                        index=False, float_format="%.1f")
        ev.to_csv(os.path.join(output_dir("landfall_odds"), f"us_landfall_events_{stem}.csv"),
                  index=False, float_format="%.2f")
        return summary, by_state

    # ------------------------------------------------ run each model + compare
    results = {}
    for model in LANDFALL_MODELS:
        print(f"\n{'=' * 70}\n{model}\n{'=' * 70}")
        res = landfall_odds(model)
        if res is not None:
            results[model] = res
    if not results:
        return

    summary_all = pd.concat([r[0] for r in results.values()], ignore_index=True)
    print(f"\n{'=' * 70}\nComparison\n{'=' * 70}")
    print("\nCONUS landfall odds:")
    print(summary_all[["model", "run", "total_members",
                       "tropical_storm_or_stronger_pct", "hurricane_pct"]]
          .rename(columns={"total_members": "members",
                           "tropical_storm_or_stronger_pct": "TS+ %",
                           "hurricane_pct": "hurricane %"})
          .to_string(index=False, float_format=lambda v: f"{v:.1f}"))

    labels = {"WNV3": "WN3", "FNV3_LARGE_ENSEMBLE": "1000-member"}
    parts = []
    for model, (_, by_state) in results.items():
        tag = labels.get(model, model)
        parts.append(by_state.set_index("state")[
            ["tropical_storm_or_stronger_pct", "hurricane_pct"]].rename(columns={
                "tropical_storm_or_stronger_pct": f"{tag} TS+ %",
                "hurricane_pct": f"{tag} hurricane %"}))
    by_state_all = pd.concat(parts, axis=1).fillna(0.0)
    if not by_state_all.empty:
        by_state_all = by_state_all.loc[
            by_state_all.filter(like="TS+ %").max(axis=1)
            .sort_values(ascending=False).index]
        print(f"\nStates by landfall odds (top {min(TOP_N, len(by_state_all))}):")
        print(by_state_all.head(TOP_N).reset_index()
              .to_string(index=False, float_format=lambda v: f"{v:.1f}"))

    summary_all.to_csv(os.path.join(output_dir("landfall_odds"), "conus_landfall_odds_comparison.csv"),
                       index=False, float_format="%.1f")
    by_state_all.reset_index().to_csv(
        os.path.join(output_dir("landfall_odds"), "state_landfall_odds_comparison.csv"),
        index=False, float_format="%.1f")
    print(f"\nSaved CSVs to {output_dir('landfall_odds')}")


if RUN_LANDFALL_ODDS:
    run_landfall_odds()
else:
    print("Landfall odds: skipped (RUN_LANDFALL_ODDS = False)")


### 4. Individual storm tracks (`GENERATE_INDIVIDUAL_STORM_TRACKS`)

In [ ]:
def run_individual_storm_tracks():
    """Individual storm tracks (isolated scope)."""
    # Cyclone track plotting script with dateline-safe longitudes, undistorted map extents, and edge-safe labels.
    # Assumes df and ensemble_mean_df already exist in the calling environment
    # unless you import and call plot_cyclone_tracks(...) directly.

    import os
    import re
    import time
    from functools import lru_cache
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import matplotlib.cm as cm
    import matplotlib.colors as mcolors
    from matplotlib import font_manager
    from matplotlib.collections import LineCollection
    from matplotlib.lines import Line2D
    from matplotlib.transforms import Bbox
    import matplotlib.patheffects as pe

    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    from cartopy.io import shapereader


    # ============================================================
    # User switches
    # ============================================================
    KEEP_TRACK_IDS = STORM_TRACK_IDS

    CYCLONE_NAME = STORM_DISPLAY_NAME
    SHOW_STORM_NAME = True

    # Storm-name multiline spacing.
    STORM_LABEL_LINE_SPACING = 1.18

    # Thinner storm-label underglow so multiline labels remain readable without
    # the white stroke crowding adjacent lines.
    STORM_LABEL_OUTER_STROKE_WIDTH = 7.0
    STORM_LABEL_INNER_STROKE_WIDTH = 3.5

    DARK_MODE = globals()["DARK_MODE"]

    COLOR_BY = "minimum_sea_level_pressure_hpa"
    # COLOR_BY = "maximum_sustained_wind_speed_knots"

    INVERT_COLORMAP = True

    # Display switches.
    SHOW_LEGEND = True
    LEGEND_BACKGROUND = True  # False removes the fill behind the legend.
    LEGEND_TITLE = ("Storm tracks\nand intensity"
                    if storm_is_named(STORM_TRACK_IDS, STORM_DISPLAY_NAME)
                    else "Potential storm\ntracks and intensity")
    STORM_NAME_SIZES = {"3x2": 29, "1x1": 26}
    # Legend title and storm name share the same size for each layout.
    LEGEND_CATEGORY_SIZE = 17
    COLOR_CODE_BY_INTENSITY = False
    # Overrides the continuous pressure/wind color scale when enabled.
    COLOR_BY_STORM_STRENGTH = True
    WIND_COLUMN = "maximum_sustained_wind_speed_knots"
    TROPICAL_STORM_THRESHOLD_KT = 34.0
    HURRICANE_THRESHOLD_KT = 64.0
    MAJOR_HURRICANE_THRESHOLD_KT = 96.0  # Category 3+
    HURRICANE_COLOR_LIGHT = "#dc303b"
    HURRICANE_COLOR_DARK = "#ff555f"
    MAJOR_HURRICANE_COLOR_LIGHT = "#8f1d9e"
    MAJOR_HURRICANE_COLOR_DARK = "#d26cff"
    # Below 34 kt and missing-wind portions are omitted from the colored layer.

    # Major speed switch. At global-publishing resolution, 50m boundaries are
    # much faster to clip and render than 10m boundaries while retaining good
    # visual detail. Set False when you specifically need 10m coastlines.
    FAST_RENDER = True
    FAST_MAP_RESOLUTION = "50m"
    DETAILED_MAP_RESOLUTION = "10m"

    # Print elapsed time for the major plotting stages.
    SHOW_TIMINGS = True

    # Draw U.S. state boundaries.
    SHOW_STATE_BORDERS = True

    # Draw non-U.S. first-order subdivision boundaries used in this map style.
    # This currently includes Canadian provinces/territories and Mexican states.
    SHOW_PROVINCE_BORDERS = True

    # Local Natural Earth 50m admin-1 shapefile. Natural Earth's 50m admin-1
    # product contains only the United States and Canada, so Mexico is loaded
    # separately from the 10m admin-1 product below.
    ADMIN1_STATES_PROVINCES_SHP = (
        "assets/shapefiles/"
        "ne_50m_admin_1_states_provinces/"
        "ne_50m_admin_1_states_provinces.shp"
    )

    # Mexico is not included in Natural Earth's 50m admin-1 dataset. Use the
    # 10m dataset for Mexican state boundaries; Cartopy downloads it once and
    # then reuses its local cache on subsequent runs.
    MEXICO_ADMIN1_RESOLUTION = "10m"

    # Draw international country boundaries wherever they fall within the
    # selected map extent. Set to False to hide them.
    SHOW_COUNTRY_BORDERS = True

    # Draw shorelines for the five Great Lakes only. Other inland lakes remain
    # unoutlined so they do not compete with state and province boundaries.
    SHOW_GREAT_LAKES = True

    # Fixed member-track color used when COLOR_CODE_BY_INTENSITY = False.
    MEMBER_TRACK_COLOR_LIGHT = "#03afca"
    MEMBER_TRACK_COLOR_DARK = "#63c5ff"

    # One map is saved per layout, e.g. ["1x1"] for a single square map.
    LAYOUTS = ["1x1", "3x2"]
    EXTENT_PADDING_DEGREES = 7

    # Tighter optional per-layout padding. Set USE_LAYOUT_EXTENT_PADDING = False
    # to use EXTENT_PADDING_DEGREES for every layout.
    USE_LAYOUT_EXTENT_PADDING = True
    LAYOUT_EXTENT_PADDING_DEGREES = {
        "1x1": 2.5,
        "3x2": 3.5,
    }

    PLOT_MEMBERS = True
    PLOT_MEAN_TRACK = True
    PLOT_DOTS = False

    # Credit switch.
    SHOW_CREDITS = globals()["SHOW_CREDITS"]
    DATA_CREDIT_TEXT = globals()["DATA_CREDIT_TEXT"]
    BYLINE_CREDIT_TEXT = globals()["BYLINE_CREDIT_TEXT"]

    # Legend intensity cue. For pressure, lower values are more intense, so the
    # arrow points downward toward the bottom of the color scale.
    SHOW_INTENSITY_ARROW = True
    INTENSITY_LABEL = "More\nintense"

    SAVE_FIG = True

    # If SAVE_DIR is defined in your notebook/session, this uses it. Otherwise,
    # it saves to the current working directory.
    SAVE_DIR = globals()["SAVE_DIR"]

    # Leave this as None to build the filename automatically from CYCLONE_NAME and
    # layout. Set it to a path string only when you want to override that behavior
    # (the layout is appended, e.g. "..._1x1.jpg").
    OUTPUT_PATH = None

    DPI = 200
    JPEG_QUALITY = globals()["JPEG_QUALITY"]  # set in the settings cell

    FIGSIZES = {
        "3x2": (15, 10),
        "1x1": (10, 10),
    }

    LAYOUT_ASPECT_RATIOS = {
        "3x2": 3 / 2,
        "1x1": 1.0,
    }


    # ============================================================
    # Franklin fonts
    # ============================================================
    FONT_BLACK = FONT_DIR + "/FranklinITCStd-Black.otf"
    FONT_BOLD = FONT_DIR + "/FranklinITCStd-Bold.otf"
    FONT_LIGHT = FONT_DIR + "/FranklinITCStd-Light.otf"


    def load_font(path, size=None):
        if os.path.exists(path):
            return font_manager.FontProperties(fname=path, size=size)
        return font_manager.FontProperties(size=size)


    font_black = load_font(FONT_BLACK)
    font_bold = load_font(FONT_BOLD)
    font_bold.set_weight("bold")
    font_light = load_font(FONT_LIGHT)
    font_light.set_weight(300)
    font_light_17 = load_font(FONT_LIGHT, 17)
    font_light_15 = load_font(FONT_LIGHT, 15)
    font_light_14 = load_font(FONT_LIGHT, 14)
    font_light_13 = load_font(FONT_LIGHT, 13)
    font_light_12 = load_font(FONT_LIGHT, 12)


    # ============================================================
    # Helpers
    # ============================================================
    def canonical_track_ids(values):
        """Return comparable storm IDs while preserving IDs in the source data.

        WeatherNext can append a parenthetical occurrence suffix to cyclogenesis
        IDs (for example, ``EP152026(2)``), while the paired official-mean file
        retains the base ID (``EP152026``). Removing only a trailing numeric
        suffix lets one user-facing ID select the corresponding rows in both
        dataframes without changing the IDs used for grouping and plotting.
        """
        return (
            pd.Series(values, copy=False)
            .astype("string")
            .str.strip()
            .str.upper()
            .str.replace(r"\s*\(\d+\)\s*$", "", regex=True)
        )


    def filter_storms(df, keep_ids, dataframe_label="member tracks"):
        if "track_id" not in df.columns:
            raise ValueError(
                f"The {dataframe_label} dataframe is missing required column "
                "'track_id'."
            )

        requested_ids = canonical_track_ids(keep_ids).dropna().unique().tolist()
        available_ids = canonical_track_ids(df["track_id"])
        filtered_df = df[available_ids.isin(requested_ids)].copy()

        matched_source_ids = sorted(
            filtered_df["track_id"].dropna().astype(str).unique().tolist()
        )
        print(f"Keeping storms: {requested_ids}")
        print(f"Matched {dataframe_label} IDs: {matched_source_ids}")
        print(f"Filtered {dataframe_label} down to {len(filtered_df):,} rows")

        if filtered_df.empty:
            source_ids = sorted(df["track_id"].dropna().astype(str).unique())
            preview = source_ids[:30]
            suffix = " ..." if len(source_ids) > len(preview) else ""
            raise ValueError(
                f"No {dataframe_label} matched {requested_ids}. "
                f"Available track IDs include: {preview}{suffix}"
            )

        return filtered_df


    def storm_strength_segments(group):
        """Split intervals at 34/64/96 kt crossings and assign TS/HU/MH classes."""
        points = group[["_plot_lon", "lat"]].to_numpy(dtype=float)
        winds = group["_strength_wind_kt"].to_numpy(dtype=float)
        segments, strength_class = [], []
        for i in range(len(points) - 1):
            a, b = points[i:i+2]
            w0, w1 = winds[i:i+2]
            if not np.isfinite([*a, *b, w0, w1]).all() or min(w0, w1) < 0:
                continue
            cuts = [0., 1.]
            if w1 != w0:
                for threshold in (
                    TROPICAL_STORM_THRESHOLD_KT,
                    HURRICANE_THRESHOLD_KT,
                    MAJOR_HURRICANE_THRESHOLD_KT,
                ):
                    fraction = (threshold - w0) / (w1 - w0)
                    if 0 < fraction < 1:
                        cuts.append(fraction)
            cuts.sort()
            for start, end in zip(cuts[:-1], cuts[1:]):
                wind = w0 + (w1 - w0) * (start + end) / 2
                if wind < TROPICAL_STORM_THRESHOLD_KT:
                    continue
                segments.append([a + (b-a)*start, a + (b-a)*end])
                if wind >= MAJOR_HURRICANE_THRESHOLD_KT:
                    strength_class.append(2)
                elif wind >= HURRICANE_THRESHOLD_KT:
                    strength_class.append(1)
                else:
                    strength_class.append(0)
        return (
            np.asarray(segments, dtype=float).reshape(-1, 2, 2),
            np.asarray(strength_class, dtype=np.int8),
        )


    def rgb_rainbow_24(nc=24, invert=True):
        path1 = np.linspace(0.8 * np.pi, 1.8 * np.pi, nc)
        path2 = np.linspace(-0.33 * np.pi, 0.33 * np.pi, nc)

        y = np.concatenate([
            np.linspace(0.3, 0.85, nc * 2 // 5),
            np.linspace(0.9, 0.0, nc - nc * 2 // 5)
        ])

        u = 0.40 * np.sin(path1)
        v = 0.55 * np.sin(path2) + 0.1

        cdata = np.empty((nc, 3))

        for i in range(nc):
            r = y[i] + 1.13983 * v[i]
            g = y[i] - 0.39465 * u[i] - 0.5806 * v[i]
            b = y[i] + 2.03211 * u[i]

            cdata[i] = [
                np.clip(r, 0, 1),
                np.clip(g, 0, 1),
                np.clip(b, 0, 1)
            ]

        if invert:
            cdata = cdata[::-1]

        return mcolors.ListedColormap(cdata, "Hoymeyer_cb")


    def get_theme(dark_mode=True):
        if dark_mode:
            return {
                **GLOBAL_MAP["themes"][True],   # same base colors as other maps
                "mean": "#ffffff",
                "member": MEMBER_TRACK_COLOR_DARK,
                "hurricane": HURRICANE_COLOR_DARK,
                "major_hurricane": MAJOR_HURRICANE_COLOR_DARK,
                "panel": GLOBAL_MAP["themes"][True]["fig"],
                "panel_alpha": 0.88,
                "panel_edge": "none",
            }

        return {
            **GLOBAL_MAP["themes"][False],
            "mean": "black",
            "member": MEMBER_TRACK_COLOR_LIGHT,
            "hurricane": HURRICANE_COLOR_LIGHT,
            "major_hurricane": MAJOR_HURRICANE_COLOR_LIGHT,
            "panel": "white",
            "panel_alpha": 0.88,
            "panel_edge": "none",
        }


    def unwrap_track_longitudes(plot_df):
        """Unwrap every member through time, then align members to one branch.

        Treating every longitude as an unordered cloud can put points from the
        same member on opposite sides of the 180-degree seam. This version first
        unwraps each chronological track so adjacent points always take the short
        path across the dateline. It then shifts whole members by multiples of
        360 degrees so their starting points share the same longitude branch.
        """
        result = plot_df.copy()
        result["_plot_lon"] = np.nan

        groups = []
        start_longitudes = []

        for _, group in result.groupby(["track_id", "sample"], sort=False):
            group = group.sort_values("valid_time")
            raw_lons = group["lon"].to_numpy(dtype=float)
            wrapped = (raw_lons + 180.0) % 360.0 - 180.0
            unwrapped = np.rad2deg(np.unwrap(np.deg2rad(wrapped)))
            groups.append((group.index, unwrapped))
            start_longitudes.append(unwrapped[0])

        start_angles = np.deg2rad(np.mod(start_longitudes, 360.0))
        shared_start = np.rad2deg(np.arctan2(
            np.mean(np.sin(start_angles)),
            np.mean(np.cos(start_angles)),
        ))

        for index, unwrapped in groups:
            shift = 360.0 * np.round((shared_start - unwrapped[0]) / 360.0)
            result.loc[index, "_plot_lon"] = unwrapped + shift

        # Shift the entire ensemble together for readable diagnostic extents. This
        # does not alter track geometry or introduce a new seam.
        ensemble_center = 0.5 * (
            result["_plot_lon"].min() + result["_plot_lon"].max()
        )
        common_shift = 360.0 * np.floor((ensemble_center + 180.0) / 360.0)
        result["_plot_lon"] -= common_shift

        return result


    def prepare_official_mean_tracks(mean_df, track_ids, member_plot_df):
        """Filter and dateline-align the downloaded official mean tracks.

        The paired ensemble-mean file is kept separate from the member dataframe.
        Its longitudes are unwrapped through time and shifted onto the same
        longitude branch as the member tracks so dateline crossings remain
        continuous.
        """
        if mean_df is None:
            print("No ensemble-mean dataframe; skipping mean track.")
            return None

        result = mean_df.copy()

        required = {"track_id", "valid_time", "lat", "lon"}
        missing = sorted(required - set(result.columns))
        if missing:
            raise ValueError(
                "The ensemble-mean dataframe is missing required columns: "
                + ", ".join(missing)
            )

        if track_ids is not None:
            # Genesis id numbers have no official mean; skip it quietly.
            wanted = set(canonical_track_ids(track_ids).dropna())
            if not canonical_track_ids(result["track_id"]).isin(wanted).any():
                print(f"No official ensemble mean for {sorted(wanted)}; "
                      "plotting members only.")
                return None
            result = filter_storms(
                result,
                track_ids,
                dataframe_label="official ensemble-mean tracks",
            )

        result["valid_time"] = pd.to_datetime(result["valid_time"])
        result = result.dropna(subset=["valid_time", "lat", "lon"])
        result = result.sort_values(["track_id", "valid_time"])

        if result.empty:
            raise ValueError(
                "No official ensemble-mean points remain after storm filtering."
            )

        member_for_alignment = member_plot_df.copy()
        member_for_alignment["_canonical_track_id"] = canonical_track_ids(
            member_for_alignment["track_id"]
        ).to_numpy()
        member_starts = (
            member_for_alignment
            .sort_values("valid_time")
            .groupby(["_canonical_track_id", "sample"], as_index=False)
            .first()
            .groupby("_canonical_track_id")["_plot_lon"]
            .median()
        )
        fallback_member_start = member_plot_df["_plot_lon"].median()

        result["_plot_lon"] = np.nan

        for track_id, group in result.groupby("track_id", sort=False):
            group = group.sort_values("valid_time")
            raw_lons = group["lon"].to_numpy(dtype=float)
            wrapped = (raw_lons + 180.0) % 360.0 - 180.0
            unwrapped = np.rad2deg(np.unwrap(np.deg2rad(wrapped)))

            canonical_id = canonical_track_ids([track_id]).iloc[0]
            member_start = member_starts.get(canonical_id, fallback_member_start)
            shift = 360.0 * np.round((member_start - unwrapped[0]) / 360.0)
            result.loc[group.index, "_plot_lon"] = unwrapped + shift

        print(
            "Using downloaded official ensemble mean: "
            f"{len(result):,} points"
        )
        return result


    def set_track_centered_extent(ax, extent):
        """Set an extent on a Plate Carree map centered on the track envelope."""
        lon_min, lon_max, lat_min, lat_max = map(float, extent)
        lon_half_span = 0.5 * (lon_max - lon_min)
        ax.set_xlim(-lon_half_span, lon_half_span)
        ax.set_ylim(lat_min, lat_max)


    def geographic_to_projection(ax, points):
        """Convert longitude/latitude pairs to the active map projection."""
        points = np.asarray(points, dtype=float)
        return ax.projection.transform_points(
            ccrs.PlateCarree(),
            points[:, 0],
            points[:, 1],
        )[:, :2]


    def project_points_vectorized(points, projection):
        """Project an entire longitude/latitude array in one operation."""
        points = np.asarray(points, dtype=float)
        projected = projection.transform_points(
            ccrs.PlateCarree(),
            points[:, 0],
            points[:, 1],
        )[:, :2]
        finite = np.isfinite(projected).all(axis=1)
        return projected[finite], finite


    def project_segments_vectorized(segments, projection):
        """Project all line-segment endpoints at once for much faster drawing."""
        segments = np.asarray(segments, dtype=float)
        flat_points = segments.reshape(-1, 2)
        projected_flat = projection.transform_points(
            ccrs.PlateCarree(),
            flat_points[:, 0],
            flat_points[:, 1],
        )[:, :2]
        projected_segments = projected_flat.reshape(-1, 2, 2)
        finite = np.isfinite(projected_segments).all(axis=(1, 2))
        return projected_segments[finite], finite


    def get_padded_track_extent(plot_df, padding=7):
        lon_min = np.nanmin(plot_df["_plot_lon"]) - padding
        lon_max = np.nanmax(plot_df["_plot_lon"]) + padding
        lat_min = np.nanmin(plot_df["lat"]) - padding
        lat_max = np.nanmax(plot_df["lat"]) + padding
        return [lon_min, lon_max, lat_min, lat_max]


    def fit_extent_to_layout_aspect(extent, layout):
        """Expand an extent just enough to match the output frame's aspect.

        Nothing is cropped and the map is not stretched. The function preserves
        the center of the selected extent when possible, expanding only longitude
        or latitude so a Plate Carree map fills the requested 1x1 or 3x2 canvas.
        """
        lon_min, lon_max, lat_min, lat_max = map(float, extent)
        target_aspect = LAYOUT_ASPECT_RATIOS.get(layout, 3 / 2)

        lon_span = lon_max - lon_min
        lat_span = lat_max - lat_min

        if lon_span <= 0 or lat_span <= 0:
            raise ValueError(f"Invalid map extent: {extent}")

        current_aspect = lon_span / lat_span

        if current_aspect > target_aspect:
            # The map is too wide for the frame: add the minimum latitude needed.
            required_lat_span = lon_span / target_aspect
            lat_center = 0.5 * (lat_min + lat_max)
            lat_min = lat_center - 0.5 * required_lat_span
            lat_max = lat_center + 0.5 * required_lat_span

            # Shift, rather than crop, if the expanded frame crosses a pole.
            if lat_min < -90:
                lat_max += -90 - lat_min
                lat_min = -90
            if lat_max > 90:
                lat_min -= lat_max - 90
                lat_max = 90

        elif current_aspect < target_aspect:
            # The map is too tall for the frame: add the minimum longitude needed.
            required_lon_span = lat_span * target_aspect
            lon_center = 0.5 * (lon_min + lon_max)
            lon_min = lon_center - 0.5 * required_lon_span
            lon_max = lon_center + 0.5 * required_lon_span

        return [lon_min, lon_max, lat_min, lat_max]


    def remove_all_edges(ax):
        ax.set_frame_on(False)

        if hasattr(ax, "outline_patch"):
            ax.outline_patch.set_visible(False)

        if hasattr(ax, "background_patch"):
            ax.background_patch.set_edgecolor("none")
            ax.background_patch.set_linewidth(0)

        ax.patch.set_edgecolor("none")
        ax.patch.set_linewidth(0)

        for spine in ax.spines.values():
            spine.set_visible(False)

        ax.set_xticks([])
        ax.set_yticks([])


    def pressure_ticks():
        return np.array([900, 920, 940, 960, 980, 1000])


    def wind_ticks():
        return np.array([40, 100, 160, 220])


    @lru_cache(maxsize=2)
    def get_great_lake_geometries(resolution="50m"):
        """Return Natural Earth geometries for the five Great Lakes only."""
        lake_path = shapereader.natural_earth(
            resolution=resolution,
            category="physical",
            name="lakes",
        )

        great_lake_names = (
            "superior",
            "michigan",
            "huron",
            "erie",
            "ontario",
        )

        geometries = []
        for record in shapereader.Reader(lake_path).records():
            name_fields = " ".join(
                str(value).lower()
                for key, value in record.attributes.items()
                if key.lower().startswith("name") and value
            )
            if any(name in name_fields for name in great_lake_names):
                geometries.append(record.geometry)

        return geometries


    def normalize_multiline_label(text):
        """
        Support normal newline escapes plus the common /n shorthand.
        This lets CYCLONE_NAME be written as either:
            "Tropical\nDepression\nTwo"
        or:
            "Tropical/nDepression/nTwo"
        """
        return str(text).replace("/n", "\n")


    def storm_output_path(cyclone_name, layout, save_dir=None):
        """Build a filesystem-safe output name from the displayed storm name."""
        save_dir = save_dir or mode_dir("storm_tracks")
        plain_name = normalize_multiline_label(cyclone_name).replace("\n", " ")
        slug = re.sub(r"[^A-Za-z0-9]+", "_", plain_name).strip("_").lower()
        if not slug:
            slug = "cyclone"
        return os.path.join(save_dir, f"{slug}_tracks_{layout}.jpg")


    def expand_extent_about_center(extent, factor):
        """Expand an extent without changing its center or aspect ratio."""
        lon_min, lon_max, lat_min, lat_max = map(float, extent)
        lon_center = 0.5 * (lon_min + lon_max)
        lat_center = 0.5 * (lat_min + lat_max)
        lon_half = 0.5 * (lon_max - lon_min) * factor
        lat_half = 0.5 * (lat_max - lat_min) * factor
        return [
            lon_center - lon_half,
            lon_center + lon_half,
            lat_center - lat_half,
            lat_center + lat_half,
        ]


    def position_storm_label_away_from_tracks(ax, plot_df, text_artist, extent):
        """Place the label beside the ensemble start without crossing tracks.

        Candidate positions radiate outward from the median starting point. The
        nearest fully visible, collision-free position wins. The map is enlarged
        only when the immediate area around the starting point has no clear space.
        """
        first_points = (
            plot_df
            .sort_values("valid_time")
            .groupby(["track_id", "sample"], as_index=False)
            .first()
        )
        start_lon = float(first_points["_plot_lon"].median())
        start_lat = float(first_points["lat"].median())

        projected_start = geographic_to_projection(
            ax,
            [[start_lon, start_lat]],
        )[0]
        # Project all label-collision coordinates in one batch. Calling Cartopy
        # once per ensemble member becomes expensive for 1,000-member runs.
        label_df = plot_df.sort_values(
            ["track_id", "sample", "valid_time"]
        ).reset_index(drop=True)
        projected_all_points = geographic_to_projection(
            ax,
            label_df[["_plot_lon", "lat"]].to_numpy(dtype=float),
        )
        member_positions = label_df.groupby(
            ["track_id", "sample"],
            sort=False,
        ).indices
        projected_tracks = [
            projected_all_points[positions]
            for positions in member_positions.values()
            if len(positions) >= 2
        ]

        # Text extents need a renderer, but the Natural Earth layers do not need to
        # be drawn first. Avoiding canvas.draw() here leaves the final save/show as
        # the only expensive geographic render.
        try:
            renderer = ax.figure.canvas.get_renderer()
        except (AttributeError, RuntimeError):
            # Non-Agg GUI backends may not create a renderer until their first draw.
            ax.figure.canvas.draw()
            renderer = ax.figure.canvas.get_renderer()
        axes_bbox = ax.get_window_extent(renderer=renderer)

        def evaluate_candidates():
            start_display = ax.transData.transform([projected_start])[0]
            start_axes = ax.transAxes.inverted().transform([start_display])[0]
            start_x, start_y = map(float, start_axes)

            # Search close to the cyclone's starting point first. Each anchor is
            # placed just outside the label edge, so small gaps remain genuinely
            # close even for long or multiline storm names.
            display_tracks = [
                ax.transData.transform(track)
                for track in projected_tracks
            ]
            all_points = (
                np.vstack(display_tracks)
                if display_tracks
                else np.empty((0, 2), dtype=float)
            )

            # Combine all tracks once so label collision checks run as vectorized
            # NumPy comparisons instead of one Python geometry call per member.
            display_segment_groups = [
                np.stack([track[:-1], track[1:]], axis=1)
                for track in display_tracks
                if len(track) >= 2
            ]
            display_segments = (
                np.concatenate(display_segment_groups, axis=0)
                if display_segment_groups
                else np.empty((0, 2, 2), dtype=float)
            )

            if display_segments.size:
                segment_x_min = display_segments[:, :, 0].min(axis=1)
                segment_x_max = display_segments[:, :, 0].max(axis=1)
                segment_y_min = display_segments[:, :, 1].min(axis=1)
                segment_y_max = display_segments[:, :, 1].max(axis=1)

            all_results = []
            order = 0

            # Test one distance ring at a time and stop at the first ring containing
            # a fully visible, collision-free position. This preserves the intended
            # near-start placement while avoiding most candidate/path comparisons.
            for gap in (0.018, 0.035, 0.055, 0.080, 0.115, 0.155):
                ring_candidates = [
                    (start_x - gap, start_y, "right", "center"),
                    (start_x + gap, start_y, "left", "center"),
                    (start_x, start_y + gap, "center", "bottom"),
                    (start_x, start_y - gap, "center", "top"),
                    (start_x - gap, start_y + gap, "right", "bottom"),
                    (start_x + gap, start_y + gap, "left", "bottom"),
                    (start_x - gap, start_y - gap, "right", "top"),
                    (start_x + gap, start_y - gap, "left", "top"),
                ]
                ring_results = []

                for x, y, ha, va in ring_candidates:
                    text_artist.set_position((x, y))
                    text_artist.set_ha(ha)
                    text_artist.set_va(va)
                    text_artist.set_multialignment(ha)

                    raw_bbox = text_artist.get_window_extent(renderer=renderer)
                    bbox = Bbox.from_extents(
                        raw_bbox.x0 - 12,
                        raw_bbox.y0 - 12,
                        raw_bbox.x1 + 12,
                        raw_bbox.y1 + 12,
                    )

                    fully_inside = (
                        raw_bbox.x0 >= axes_bbox.x0 + 4
                        and raw_bbox.x1 <= axes_bbox.x1 - 4
                        and raw_bbox.y0 >= axes_bbox.y0 + 4
                        and raw_bbox.y1 <= axes_bbox.y1 - 4
                    )

                    if display_segments.size:
                        # Conservative segment-envelope test: fast and slightly
                        # more generous about keeping labels away from tracks.
                        collision = int(np.any(
                            (segment_x_max >= bbox.x0)
                            & (segment_x_min <= bbox.x1)
                            & (segment_y_max >= bbox.y0)
                            & (segment_y_min <= bbox.y1)
                        ))
                    else:
                        collision = 0

                    if all_points.size:
                        dx = np.maximum.reduce([
                            bbox.x0 - all_points[:, 0],
                            np.zeros(len(all_points)),
                            all_points[:, 0] - bbox.x1,
                        ])
                        dy = np.maximum.reduce([
                            bbox.y0 - all_points[:, 1],
                            np.zeros(len(all_points)),
                            all_points[:, 1] - bbox.y1,
                        ])
                        clearance = float(np.min(np.hypot(dx, dy)))
                    else:
                        clearance = float("inf")

                    # Measure the visible distance between the start and the actual
                    # label box. This keeps the label close to cyclogenesis.
                    start_dx = max(
                        raw_bbox.x0 - start_display[0],
                        0,
                        start_display[0] - raw_bbox.x1,
                    )
                    start_dy = max(
                        raw_bbox.y0 - start_display[1],
                        0,
                        start_display[1] - raw_bbox.y1,
                    )
                    start_distance = float(np.hypot(start_dx, start_dy))

                    # Stay inside the map, avoid every line, remain close to the
                    # start, then use overall clearance as the final tie-breaker.
                    result = (
                        0 if fully_inside else 1,
                        collision,
                        start_distance,
                        -clearance,
                        order,
                        x,
                        y,
                        ha,
                        va,
                    )
                    ring_results.append(result)
                    all_results.append(result)
                    order += 1

                clear_results = [
                    result
                    for result in ring_results
                    if result[0] == 0 and result[1] == 0
                ]
                if clear_results:
                    return min(clear_results)

            return min(all_results)

        base_extent = list(extent)
        best = None
        chosen_factor = 1.0

        # Find the smallest 8% expansion with a visible, collision-free position
        # near the start. Most storms will use factor 1.0 (no expansion).
        for factor in np.arange(1.0, 2.61, 0.08):
            trial_extent = expand_extent_about_center(base_extent, float(factor))
            set_track_centered_extent(ax, trial_extent)
            best = evaluate_candidates()
            chosen_factor = float(factor)
            if best[0] == 0 and best[1] == 0:
                break

        _, _, _, _, _, x, y, ha, va = best
        text_artist.set_position((x, y))
        text_artist.set_ha(ha)
        text_artist.set_va(va)
        text_artist.set_multialignment(ha)

        final_extent = expand_extent_about_center(base_extent, chosen_factor)
        set_track_centered_extent(ax, final_extent)
        return final_extent


    # ============================================================
    # Main plotting function
    # ============================================================
    def plot_cyclone_tracks(
        df,
        ensemble_mean_df=None,
        track_ids=None,
        cyclone_name="Bavi",
        show_storm_name=True,
        dark_mode=True,
        fast_render=FAST_RENDER,
        show_timings=SHOW_TIMINGS,
        color_by="minimum_sea_level_pressure_hpa",
        invert_colormap=True,
        show_legend=True,
        color_code_by_intensity=True,
        show_state_borders=True,
        show_province_borders=True,
        show_country_borders=True,
        show_great_lakes=True,
        extent_padding_degrees=7,
        use_layout_extent_padding=True,
        layout_extent_padding_degrees=None,
        plot_members=True,
        plot_mean_track=True,
        plot_dots=True,
        show_credits=True,
        data_credit_text="Data: Google DeepMind",
        byline_credit_text="BEN NOLL/THE WASHINGTON POST",
        show_intensity_arrow=True,
        intensity_label="More\nintense",
        layout="3x2",
        save_fig=True,
        output_path=None,
        color_by_storm_strength=COLOR_BY_STORM_STRENGTH,
    ):
        if color_by_storm_strength:
            color_code_by_intensity = False
        plot_started = time.perf_counter()
        plot_df = df.copy()

        if track_ids is not None:
            plot_df = filter_storms(plot_df, track_ids, dataframe_label="member tracks")

        if plot_df.empty:
            raise ValueError("No member-track rows are available to plot.")

        plot_df["valid_time"] = pd.to_datetime(plot_df["valid_time"])
        plot_df["init_time"] = pd.to_datetime(plot_df["init_time"])
        plot_df = plot_df.sort_values(["track_id", "sample", "valid_time"])
        if color_by_storm_strength:
            if WIND_COLUMN not in plot_df:
                raise ValueError(f"Storm-strength colors require {WIND_COLUMN!r} in knots.")
            plot_df["_strength_wind_kt"] = pd.to_numeric(plot_df[WIND_COLUMN], errors="coerce")
            if not plot_df["_strength_wind_kt"].ge(0).any():
                raise ValueError("No valid sustained wind speeds are available.")

        required_columns = ["lat", "lon"]
        if color_code_by_intensity:
            required_columns.append(color_by)
        plot_df = plot_df.dropna(subset=required_columns)

        if plot_df.empty:
            raise ValueError("No valid storm-track points remain after filtering.")

        theme = get_theme(dark_mode)
        # Zoomed-in storm maps use the same detailed scale as regional maps.
        map_resolution = REGIONAL_FEATURE_SCALE
        print(f"Map feature resolution: {map_resolution}")
        cmap = rgb_rainbow_24(24, invert=invert_colormap) if color_code_by_intensity else None

        plot_df = unwrap_track_longitudes(plot_df)

        official_mean_df = None
        if plot_mean_track:
            official_mean_df = prepare_official_mean_tracks(
                ensemble_mean_df,
                track_ids,
                plot_df,
            )

        # Only compute intensity/colorbar settings when member tracks are actually
        # color-coded by intensity. Otherwise every member uses theme["member"].
        if color_code_by_intensity:
            if color_by == "minimum_sea_level_pressure_hpa":
                vmin, vmax = 900, 1000
                legend_title = "Air pressure"
                legend_units = "hPa"
                ticks = pressure_ticks()

            elif color_by == "maximum_sustained_wind_speed_knots":
                plot_df["_wind_mph"] = plot_df[color_by] * 1.15078
                color_by = "_wind_mph"
                vmin, vmax = 40, 220
                legend_title = "Wind gusts"
                legend_units = "mph"
                ticks = wind_ticks()

            else:
                vmin = np.floor(np.nanmin(plot_df[color_by]) / 10) * 10
                vmax = np.ceil(np.nanmax(plot_df[color_by]) / 10) * 10
                legend_title = color_by.replace("_", " ").title()
                legend_units = ""
                ticks = np.linspace(vmin, vmax, 4)

            norm = plt.Normalize(vmin=vmin, vmax=vmax)
        else:
            norm = None

        figsize = FIGSIZES.get(layout, FIGSIZES["3x2"])

        if layout_extent_padding_degrees is None:
            layout_extent_padding_degrees = LAYOUT_EXTENT_PADDING_DEGREES

        if use_layout_extent_padding:
            active_extent_padding = layout_extent_padding_degrees.get(
                layout,
                extent_padding_degrees
            )
        else:
            active_extent_padding = extent_padding_degrees

        # The tracks define the domain automatically. First create their tight
        # padded envelope, then expand only the narrower dimension enough to match
        # the selected output ratio without distorting or cropping the map.
        extent_source_df = plot_df
        if official_mean_df is not None:
            extent_source_df = pd.concat(
                [
                    plot_df[["_plot_lon", "lat"]],
                    official_mean_df[["_plot_lon", "lat"]],
                ],
                ignore_index=True,
            )

        extent = get_padded_track_extent(
            extent_source_df,
            padding=active_extent_padding,
        )

        # Match the geographic frame to the requested export ratio. This removes
        # letterboxing while preserving Cartopy's undistorted map geometry.
        extent = fit_extent_to_layout_aspect(extent, layout)

        print(f"Map extent: {extent}")

        # Put the projection seam on the opposite side of the globe from the
        # storm. A dateline-crossing track therefore remains continuous near the
        # middle of the map instead of being split at +/-180 degrees.
        track_center_lon = 0.5 * (extent[0] + extent[1])
        projection_center_lon = (
            (track_center_lon + 180.0) % 360.0
            - 180.0
        )
        map_projection = ccrs.PlateCarree(
            central_longitude=projection_center_lon
        )

        fig = plt.figure(figsize=figsize, facecolor=theme["fig"])
        ax = fig.add_axes([0, 0, 1, 1], projection=map_projection)

        ax.set_facecolor(theme["ocean"])
        set_track_centered_extent(ax, extent)

        # Preserve correct map geometry. The extent above already matches the
        # requested output ratio, so the axes fill the canvas without stretching.
        ax.set_aspect("equal", adjustable="box")

        land = cfeature.NaturalEarthFeature(
            "physical", "land", map_resolution,
            facecolor=theme["land"],
            edgecolor="none",
        )

        coastline = cfeature.NaturalEarthFeature(
            "physical", "coastline", map_resolution,
            facecolor="none",
            edgecolor=theme["coast"],
        )

        borders = cfeature.NaturalEarthFeature(
            "cultural", "admin_0_boundary_lines_land", map_resolution,
            facecolor="none",
            edgecolor=theme["border"]
        )

        # Same base map as the global/Gulf maps: land + coastline (with
        # underglow) + country/state LINE layers, all at one resolution so the
        # boundaries never trace a second, mismatched coastline.
        ax.add_feature(land, zorder=1)
        add_global_overlays(ax, theme, dark_mode, show_coastlines=True,
                            scale=map_resolution)

        remove_all_edges(ax)

        # Track which storm-strength classes actually make it into the plotted
        # member segments so the legend only shows categories present in the data.
        # 0 = tropical storm, 1 = hurricane, 2 = major hurricane.
        present_strength_classes = set()

        if plot_members:
            all_segments = []
            all_segment_values = []

            for _, group in plot_df.groupby(["track_id", "sample"]):
                group = group.sort_values("valid_time")

                if len(group) < 2:
                    continue

                if color_by_storm_strength:
                    segments, strength_values = storm_strength_segments(group)
                    if not len(segments):
                        continue
                    all_segment_values.append(strength_values)
                else:
                    points = group[["_plot_lon", "lat"]].to_numpy()
                    segments = np.stack([points[:-1], points[1:]], axis=1)
                all_segments.append(segments)

                if color_code_by_intensity:
                    all_segment_values.append(group[color_by].to_numpy()[1:])

            if all_segments:
                combined_segments = np.concatenate(all_segments, axis=0)
                projected_segments, finite_segment_mask = (
                    project_segments_vectorized(
                        combined_segments,
                        map_projection,
                    )
                )

                print(
                    "Vector-projected member segments: "
                    f"{len(projected_segments):,}"
                )

                if color_by_storm_strength:
                    strength_values = np.concatenate(all_segment_values)[finite_segment_mask]
                    present_strength_classes.update(
                        int(value) for value in np.unique(strength_values)
                    )
                    lc = LineCollection(
                        projected_segments,
                        colors=np.select(
                            [strength_values == 2, strength_values == 1],
                            [theme["major_hurricane"], theme["hurricane"]],
                            default=theme["member"],
                        ),
                        linewidth=1.35, alpha=0.72, zorder=30,
                        capstyle="butt", transform=ax.transData,
                    )
                elif color_code_by_intensity:
                    lc = LineCollection(
                        projected_segments,
                        cmap=cmap,
                        norm=norm,
                        linewidth=1.35,
                        alpha=0.82,
                        zorder=30,
                        transform=ax.transData,
                    )
                    segment_values = np.concatenate(all_segment_values)
                    lc.set_array(segment_values[finite_segment_mask])
                else:
                    lc = LineCollection(
                        projected_segments,
                        colors=theme["member"],
                        linewidth=1.35,
                        alpha=0.72,
                        zorder=30,
                        transform=ax.transData,
                    )

                # Coordinates are already in the map projection, so Cartopy does
                # not need to transform thousands of paths during rendering.
                ax.add_collection(lc)

            if plot_dots:
                dot_points, finite_dot_mask = project_points_vectorized(
                    plot_df[["_plot_lon", "lat"]].to_numpy(),
                    map_projection,
                )
                scatter_kwargs = dict(
                    s=13,
                    linewidth=0,
                    alpha=0.88 if color_code_by_intensity else 0.78,
                    transform=ax.transData,
                    zorder=35,
                )
                if color_by_storm_strength:
                    winds = plot_df["_strength_wind_kt"].to_numpy()[finite_dot_mask]
                    keep = np.isfinite(winds) & (winds >= TROPICAL_STORM_THRESHOLD_KT)
                    dot_points = dot_points[keep]
                    scatter_kwargs["c"] = np.select(
                        [
                            winds[keep] >= MAJOR_HURRICANE_THRESHOLD_KT,
                            winds[keep] >= HURRICANE_THRESHOLD_KT,
                        ],
                        [theme["major_hurricane"], theme["hurricane"]],
                        default=theme["member"],
                    )
                elif color_code_by_intensity:
                    scatter_kwargs.update(
                        c=plot_df[color_by].to_numpy()[finite_dot_mask],
                        cmap=cmap,
                        norm=norm,
                    )
                else:
                    scatter_kwargs["color"] = theme["member"]

                ax.scatter(
                    dot_points[:, 0],
                    dot_points[:, 1],
                    **scatter_kwargs,
                )

        if plot_mean_track and official_mean_df is not None:
            for _, mean_track in official_mean_df.groupby("track_id", sort=False):
                mean_track = mean_track.sort_values("valid_time")
                projected_mean, _ = project_points_vectorized(
                    mean_track[["_plot_lon", "lat"]].to_numpy(),
                    map_projection,
                )

                if len(projected_mean) < 2:
                    continue

                ax.plot(
                    projected_mean[:, 0],
                    projected_mean[:, 1],
                    color=theme["mean"],
                    linewidth=7,
                    alpha=0.30,
                    transform=ax.transData,
                    zorder=45
                )

                ax.plot(
                    projected_mean[:, 0],
                    projected_mean[:, 1],
                    color=theme["mean"],
                    linewidth=2.3,
                    alpha=0.95,
                    transform=ax.transData,
                    zorder=46
                )

        if show_storm_name and cyclone_name:
            label_started = time.perf_counter()
            display_cyclone_name = normalize_multiline_label(cyclone_name)

            selected_storm_font = font_bold.get_file()
            print(
                "Storm-label font: "
                f"{selected_storm_font or 'Matplotlib bold sans-serif fallback'} "
                "(weight bold)"
            )

            storm_label_artist = ax.text(
                0.5,
                0.5,
                display_cyclone_name,
                transform=ax.transAxes,
                ha="center",
                va="center",
                multialignment="center",
                linespacing=STORM_LABEL_LINE_SPACING,
                color=theme["text"],
                fontproperties=font_bold,
                fontweight="bold",
                fontsize=STORM_NAME_SIZES.get(layout, 29),
                zorder=300,
                clip_on=False,
                path_effects=[
                    pe.withStroke(
                        linewidth=STORM_LABEL_OUTER_STROKE_WIDTH,
                        foreground=theme["fig"],
                        alpha=0.85,
                    ),
                    pe.withStroke(
                        linewidth=STORM_LABEL_INNER_STROKE_WIDTH,
                        foreground=theme["fig"],
                        alpha=0.55,
                    ),
                    pe.Normal(),
                ],
            )

            extent = position_storm_label_away_from_tracks(
                ax,
                plot_df,
                storm_label_artist,
                extent,
            )
            print(f"Label-safe map extent: {extent}")
            if show_timings:
                print(
                    "Label placement: "
                    f"{time.perf_counter() - label_started:.2f} seconds"
                )

        if show_credits:
            credit_bbox = dict(
                facecolor=theme["panel"],
                alpha=theme["panel_alpha"],
                edgecolor=theme["panel_edge"],
                boxstyle="square,pad=0.22",
            )

            fig.text(
                0.000,
                0.000,
                data_credit_text,
                ha="left",
                va="bottom",
                color=theme["text"],
                fontproperties=font_light_13,
                fontsize=13,
                zorder=1000,
                bbox=credit_bbox,
            )

            fig.text(
                1.000,
                0.000,
                byline_credit_text,
                ha="right",
                va="bottom",
                color=theme["text"],
                fontproperties=font_light_13,
                fontsize=13,
                zorder=1000,
                bbox=credit_bbox,
            )

        if show_legend and color_by_storm_strength and present_strength_classes:
            # Highest intensity first. Only include classes represented by plotted
            # member-track segments, so unused legend categories disappear.
            legend_specs = [
                (2, theme["major_hurricane"], "Major hurricane"),
                (1, theme["hurricane"], "Hurricane"),
                (0, theme["member"], "Tropical storm"),
            ]
            handles = [
                Line2D([0], [0], color=color, linewidth=3, label=label)
                for strength_class, color, label in legend_specs
                if strength_class in present_strength_classes
            ]

            legend_font = font_light.copy()
            legend_font.set_size(LEGEND_CATEGORY_SIZE)
            legend_title_font = font_bold.copy()
            legend_title_font.set_size(STORM_NAME_SIZES.get(layout, 29))
            legend = ax.legend(
                handles=handles, loc="upper left", prop=legend_font,
                title=LEGEND_TITLE, title_fontproperties=legend_title_font,
                alignment="left",
                frameon=LEGEND_BACKGROUND,
                facecolor=theme["panel"], edgecolor="none",
                framealpha=theme["panel_alpha"], labelcolor=theme["text"],
                borderaxespad=0.8, borderpad=0.7, labelspacing=0.5,
                handlelength=1.25, handletextpad=0.55,
            )
            legend.get_title().set_color(theme["text"])
            legend.get_title().set_horizontalalignment("left")
            legend.get_title().set_multialignment("left")
            legend.set_zorder(1000)

        # The intensity legend only describes the plot when intensity color-coding
        # is enabled. SHOW_LEGEND can independently hide it when color-coding is on.
        if show_legend and color_code_by_intensity:
            legend_width = 0.135 if layout == "3x2" else 0.175
            legend_height = 0.245 if layout == "3x2" else 0.270

            legend_ax = fig.add_axes([
                1.0 - legend_width,
                0.0,
                legend_width,
                legend_height
            ])

            legend_ax.set_facecolor(theme["panel"])
            legend_ax.patch.set_alpha(theme["panel_alpha"])
            legend_ax.set_xticks([])
            legend_ax.set_yticks([])
            legend_ax.set_zorder(1000)

            for spine in legend_ax.spines.values():
                spine.set_visible(False)

            legend_title_size = 17 if layout == "3x2" else 17
            legend_tick_size = 14 if layout == "3x2" else 14
            intensity_text_size = 14 if layout == "3x2" else 14

            legend_ax.text(
                0.5,
                0.96,
                legend_title,
                transform=legend_ax.transAxes,
                ha="center",
                va="top",
                color=theme["text"],
                fontproperties=font_light_17,
                fontsize=legend_title_size,
                zorder=1005,
                clip_on=True
            )

            cax = legend_ax.inset_axes([0.66, 0.10, 0.17, 0.70])

            sm = cm.ScalarMappable(cmap=cmap, norm=norm)
            sm.set_array([])

            cbar = plt.colorbar(sm, cax=cax, orientation="vertical")
            cbar.outline.set_visible(False)
            cbar.set_ticks(ticks)

            tick_labels = [f"{int(t)}" for t in ticks]
            tick_labels[-1] = f"{tick_labels[-1]} {legend_units}"
            cbar.set_ticklabels(tick_labels)

            cbar.ax.yaxis.set_ticks_position("left")
            cbar.ax.yaxis.set_label_position("left")

            cbar.ax.tick_params(
                labelcolor=theme["text"],
                color=theme["text"],
                size=0,
                pad=7
            )

            for label in cbar.ax.get_yticklabels():
                label.set_fontproperties(font_light_14)
                label.set_fontsize(legend_tick_size)
                label.set_color(theme["text"])
                label.set_ha("right")
                label.set_clip_on(True)

            cax.set_facecolor("none")

            for spine in cax.spines.values():
                spine.set_visible(False)

            if show_intensity_arrow:
                # Breakaway cue placed inside the legend panel, separate from the
                # numeric tick labels. Keep the established 3x2 placement, but move
                # the cue farther left in the tighter 1x1 legend.
                intensity_x = 0.25 if layout == "3x2" else 0.18

                legend_ax.text(
                    intensity_x,
                    0.56,
                    intensity_label,
                    transform=legend_ax.transAxes,
                    ha="center",
                    va="center",
                    linespacing=0.90,
                    color=theme["text"],
                    fontproperties=font_light_14,
                    fontsize=intensity_text_size,
                    zorder=1006,
                    clip_on=False,
                )

                legend_ax.annotate(
                    "",
                    xy=(intensity_x, 0.22),
                    xytext=(intensity_x, 0.43),
                    xycoords="axes fraction",
                    textcoords="axes fraction",
                    arrowprops=dict(
                        arrowstyle="-|>",
                        color=theme["text"],
                        linewidth=1.4,
                        mutation_scale=12,
                        shrinkA=0,
                        shrinkB=0,
                    ),
                    zorder=1006,
                )

        remove_all_edges(ax)

        if save_fig:
            if output_path is None:
                output_path = storm_output_path(cyclone_name, layout)

            save_started = time.perf_counter()
            output_path = save_figure(   # format/quality from the settings cell
                fig,
                output_path,
                dpi=DPI,
                facecolor=fig.get_facecolor(),
                edgecolor="none",
                bbox_inches=None,
                pad_inches=0
            )
            if show_timings:
                print(
                    "Map render and save: "
                    f"{time.perf_counter() - save_started:.2f} seconds"
                )

        if show_timings:
            print(
                "Total before display: "
                f"{time.perf_counter() - plot_started:.2f} seconds"
            )

        plt.show()

        return fig, ax


    # ============================================================
    # Run when the downloader dataframes exist in the environment
    # ============================================================
    if "df" in globals():
        if PLOT_MEAN_TRACK and "ensemble_mean_df" not in globals():
            raise RuntimeError(
                "PLOT_MEAN_TRACK is True, but ensemble_mean_df was not found. "
                "Run download_weatherlab_cyclone_tracks_with_mean.py first."
            )

        for layout in LAYOUTS:
            output_path = (None if OUTPUT_PATH is None else
                           f"{os.path.splitext(OUTPUT_PATH)[0]}_{layout}")
            fig, ax = plot_cyclone_tracks(
                df,
                ensemble_mean_df=globals().get("ensemble_mean_df"),
                track_ids=KEEP_TRACK_IDS,
                cyclone_name=CYCLONE_NAME,
                show_storm_name=SHOW_STORM_NAME,
                dark_mode=DARK_MODE,
                fast_render=FAST_RENDER,
                show_timings=SHOW_TIMINGS,
                color_by=COLOR_BY,
                invert_colormap=INVERT_COLORMAP,
                show_legend=SHOW_LEGEND,
                color_code_by_intensity=COLOR_CODE_BY_INTENSITY,
                color_by_storm_strength=COLOR_BY_STORM_STRENGTH,
                show_state_borders=SHOW_STATE_BORDERS,
                show_province_borders=SHOW_PROVINCE_BORDERS,
                show_country_borders=SHOW_COUNTRY_BORDERS,
                show_great_lakes=SHOW_GREAT_LAKES,
                extent_padding_degrees=EXTENT_PADDING_DEGREES,
                use_layout_extent_padding=USE_LAYOUT_EXTENT_PADDING,
                layout_extent_padding_degrees=LAYOUT_EXTENT_PADDING_DEGREES,
                plot_members=PLOT_MEMBERS,
                plot_mean_track=PLOT_MEAN_TRACK,
                plot_dots=PLOT_DOTS,
                show_credits=SHOW_CREDITS,
                data_credit_text=DATA_CREDIT_TEXT,
                byline_credit_text=BYLINE_CREDIT_TEXT,
                show_intensity_arrow=SHOW_INTENSITY_ARROW,
                intensity_label=INTENSITY_LABEL,
                layout=layout,
                save_fig=SAVE_FIG,
                output_path=output_path
            )
    else:
        print(
            "No dataframe named df found. Run the downloader first, or import "
            "this file and call plot_cyclone_tracks(df, ensemble_mean_df, ...)."
        )


if GENERATE_INDIVIDUAL_STORM_TRACKS:
    for_each_color_mode(run_individual_storm_tracks)
else:
    print("Individual storm tracks: skipped (GENERATE_INDIVIDUAL_STORM_TRACKS = False)")


### 5. Intensity chart (`GENERATE_INTENSITY_CHART`)

In [ ]:
def run_intensity_chart():
    """Intensity chart (isolated scope)."""
    import matplotlib.dates as mdates
    import matplotlib.patheffects as pe
    from matplotlib.transforms import blended_transform_factory

    # Uses the WNV3 member dataframe ``df`` downloaded in the first cell
    # (the old separate FNV3 download, which overwrote ``df``, is gone).
    STORM_ID = list(STORM_TRACK_IDS)   # one or more IDs, combined per member
    STORM_NAME = STORM_CHART_NAME
    DARK_MODE = globals()["DARK_MODE"]
    ASPECT_RATIO = "3x2"  # "3x2" or "16x9"
    DPI = 200
    SHOW_CREDITS = globals()["SHOW_CREDITS"]
    DATA_CREDIT_TEXT = globals()["DATA_CREDIT_TEXT"]
    BYLINE_CREDIT_TEXT = globals()["BYLINE_CREDIT_TEXT"]
    CREDIT_FONT_SIZE = 13

    # ===== Fonts =====
    font_path = FONT_DIR + "/FranklinITCStd-Black.otf"
    font_properties = font_manager.FontProperties(fname=font_path)

    font_path2 = FONT_DIR + "/FranklinITCStd-Bold.otf"
    font_properties2 = font_manager.FontProperties(fname=font_path2, size=36)

    font_path6 = FONT_DIR + "/FranklinITCStd-Light.otf"
    font_properties6 = font_manager.FontProperties(fname=font_path6, size=18)

    font_path7 = FONT_DIR + "/FranklinITCStd-Bold.otf"
    font_properties7 = font_manager.FontProperties(fname=font_path7, size=18)

    font_credit = font_manager.FontProperties(fname=font_path6, size=13)


    month_abbr = {
        "Jan": "Jan.",
        "Feb": "Feb.",
        "Mar": "March",
        "Apr": "April",
        "May": "May",
        "Jun": "June",
        "Jul": "July",
        "Aug": "Aug.",
        "Sep": "Sept.",
        "Oct": "Oct.",
        "Nov": "Nov.",
        "Dec": "Dec.",
    }


    DARK_CATEGORY_COLORS = {
        "Tropical storm": "#fbdedc",
        "Category 1": "#f2b4ab",
        "Category 2": "#e58a7c",
        "Category 3": "#d35f4f",
        "Category 4": "#be2c25",
        "Category 5": "#8e1f1b",
    }

    LIGHT_CATEGORY_COLORS = {
        "Tropical storm": "#d07d72",
        "Category 1": "#c96050",
        "Category 2": "#c54634",
        "Category 3": "#b83829",
        "Category 4": "#9e2319",
        "Category 5": "#7f1712",
    }


    def get_category_colors(dark_mode=True):
        return DARK_CATEGORY_COLORS if dark_mode else LIGHT_CATEGORY_COLORS


    def get_theme(dark_mode=True):
        if dark_mode:
            return {
                "fig": "#0f1115",
                "ax": "#0f1115",
                "text": "#f4f4f4",
                "median": "#ffffff",
                "outer_band": "#8a8f98",
                "inner_band": "#d5d5d5",
            }

        return {
            "fig": "white",
            "ax": "white",
            "text": "#2a2a2a",
            "median": "#2a2a2a",
            "outer_band": "#d5d5d5",
            "inner_band": "#666666",
        }


    FIGSIZES = {
        "3x2": (15, 10),
        "16x9": (19.2, 10.8),
    }


    def clean_filename(text):
        return "".join(
            c for c in text
            if c.isalnum() or c in (" ", "_", "-")
        ).strip()


    def format_month_label(dt):
        month = dt.strftime("%b")
        day = dt.strftime("%-d")
        return f"{month_abbr.get(month, month)} {day}"


    def set_first_middle_last_date_ticks(ax, dates, text_color):
        first_date = pd.to_datetime(dates.min())
        last_date = pd.to_datetime(dates.max())
        middle_date = first_date + (last_date - first_date) / 2

        tick_dates = [first_date, middle_date, last_date]
        tick_labels = [format_month_label(d) for d in tick_dates]

        ax.set_xticks(tick_dates)
        ax.set_xticklabels(
            tick_labels,
            fontproperties=font_properties6,
            color=text_color
        )


    def plot_intensity_plume(
        df,
        storm_id,
        storm_name,
        dark_mode=True,
        aspect_ratio="3x2",
        dpi=100,
        show_credits=True,
        data_credit_text="Data: Google DeepMind",
        byline_credit_text="BEN NOLL/THE WASHINGTON POST",
        credit_font_size=13,
    ):
        theme = get_theme(dark_mode)
        category_colors = get_category_colors(dark_mode)
        category_line_alpha = 0.90 if dark_mode else 0.70

        wanted = {canonical_id(t) for t in np.atleast_1d(storm_id)}
        storm_df = df[df["track_id"].map(canonical_id).isin(wanted)].copy()

        if storm_df.empty:
            raise ValueError(f"No rows found for storm_id: {storm_id}")

        storm_df["valid_time"] = pd.to_datetime(storm_df["valid_time"])
        storm_df["init_time"] = pd.to_datetime(storm_df["init_time"])

        pivot_df = storm_df.pivot_table(
            index="valid_time",
            columns="sample",
            values="maximum_sustained_wind_speed_knots",
            aggfunc="max",  # if a member has 2+ chosen tracks, keep strongest
        ).sort_index()

        vmax_min = pivot_df.min(axis=1)
        vmax_max = pivot_df.max(axis=1)
        vmax_10 = pivot_df.quantile(0.1, axis=1)
        vmax_90 = pivot_df.quantile(0.9, axis=1)
        vmax_median = pivot_df.median(axis=1)

        window = 2
        smooth = lambda s: s.rolling(
            window=window,
            min_periods=1,
            center=True
        ).mean()

        vmax_min_s = smooth(vmax_min)
        vmax_max_s = smooth(vmax_max)
        vmax_10_s = smooth(vmax_10)
        vmax_90_s = smooth(vmax_90)
        vmax_median_s = smooth(vmax_median)

        cat_thresholds = {
            "Tropical storm": 34,
            "Category 1": 64,
            "Category 2": 83,
            "Category 3": 96,
            "Category 4": 113,
            "Category 5": 137,
        }

        member_max = pivot_df.max().dropna()
        total_members = len(member_max)

        if total_members == 0:
            raise ValueError(
                f"No valid ensemble-member wind data found for storm_id: {storm_id}"
            )

        # Cumulative category reach: a member is counted in every threshold it
        # reaches or exceeds. For example, a Category 3 member is also included
        # in the Tropical Storm, Category 1 and Category 2 totals.
        category_summary = {}

        for category, threshold in cat_thresholds.items():
            members_reaching_category = member_max[member_max >= threshold]
            member_count = len(members_reaching_category)
            member_percentage = member_count / total_members * 100

            category_summary[category] = {
                "threshold": threshold,
                "count": member_count,
                "percentage": member_percentage,
            }

        strongest_member = member_max.idxmax()
        max_wind_kt = member_max.max()
        max_wind_mph = max_wind_kt * 1.15078

        peak_upper_decile_kt = np.percentile(member_max, 90)
        peak_upper_decile_mph = peak_upper_decile_kt * 1.15078

        if "minimum_sea_level_pressure_hpa" in storm_df.columns:
            mslp_pivot = storm_df.pivot_table(
                index="valid_time",
                columns="sample",
                values="minimum_sea_level_pressure_hpa",
                aggfunc="min",
            )
            member_min_mslp = mslp_pivot.min()
            min_mslp = member_min_mslp[strongest_member]
            mslp_upper_decile = np.percentile(member_min_mslp, 10)
        else:
            min_mslp = np.nan
            mslp_upper_decile = np.nan

        print(f"\n--- {storm_name} Intensity Summary ---")
        print(f"Track ID: {storm_id}")
        print(f"Total ensemble members: {total_members}")
        print("\nMembers reaching each intensity threshold:")

        for category, stats in category_summary.items():
            plus_sign = "+" if category != "Category 5" else ""
            print(
                f" • {category}{plus_sign}: "
                f"{stats['count']:,} of {total_members:,} "
                f"({stats['percentage']:.1f}%)"
            )

        print(f"Strongest ensemble member: {strongest_member}")
        print(f" • Peak wind: {max_wind_kt:.1f} kt ({max_wind_mph:.1f} mph)")
        print(f" • Min MSLP: {min_mslp:.1f} hPa")
        print(f"Upper decile of peak winds: {peak_upper_decile_kt:.1f} kt ({peak_upper_decile_mph:.1f} mph)")
        print(f"Upper decile of lowest MSLP: {mslp_upper_decile:.1f} hPa")

        fig, ax = plt.subplots(figsize=FIGSIZES[aspect_ratio])
        fig.patch.set_facecolor(theme["fig"])
        ax.set_facecolor(theme["ax"])

        ax.fill_between(
            pivot_df.index,
            vmax_min_s,
            vmax_max_s,
            color=theme["outer_band"],
            alpha=0.40
        )

        ax.fill_between(
            pivot_df.index,
            vmax_10_s,
            vmax_90_s,
            color=theme["inner_band"],
            alpha=0.50
        )

        ax.plot(
            pivot_df.index,
            vmax_median_s,
            color=theme["median"],
            linestyle="--",
            linewidth=5
        )

        trans = blended_transform_factory(ax.transAxes, ax.transData)

        for cat, val in cat_thresholds.items():
            ax.axhline(
                val,
                color=category_colors[cat],
                linestyle="--",
                linewidth=1,
                alpha=category_line_alpha,
            )

            ax.text(
                -0.005,
                val,
                cat,
                transform=trans,
                color=category_colors[cat],
                fontproperties=font_properties7,
                ha="right",
                va="center",
                clip_on=False,
                path_effects=[
                    pe.withStroke(
                        linewidth=1.1,
                        foreground=theme["fig"],
                        alpha=0.95,
                    ),
                    pe.Normal(),
                ],
            )

        fig.text(
            0.01,
            0.955,
            f"{storm_name} forecast intensity",
            ha="left",
            va="top",
            color=theme["text"],
            fontproperties=font_properties2,
            fontsize=36
        )

        set_first_middle_last_date_ticks(
            ax=ax,
            dates=pivot_df.index,
            text_color=theme["text"]
        )

        ax.set_yticks([v for v in cat_thresholds.values()])
        ax.set_yticklabels([])

        ax.grid(False)
        ax.tick_params(
            axis="both",
            which="both",
            length=0,
            colors=theme["text"]
        )

        for spine in ax.spines.values():
            spine.set_visible(False)

        # Bottom-corner credits. These are figure-level text objects so they stay
        # anchored consistently for both 3x2 and 16x9 layouts.
        if show_credits:
            fig.text(
                0.01,
                0.015,
                data_credit_text,
                ha="left",
                va="bottom",
                color=theme["text"],
                fontproperties=font_credit,
                fontsize=credit_font_size,
                zorder=1000,
            )

            fig.text(
                0.99,
                0.015,
                byline_credit_text,
                ha="right",
                va="bottom",
                color=theme["text"],
                fontproperties=font_credit,
                fontsize=credit_font_size,
                zorder=1000,
            )

        plt.subplots_adjust(
            left=0.12,
            right=0.975,
            top=0.86,
            bottom=0.11 if show_credits else 0.09
        )

        output_name = f"{clean_filename(storm_name)} forecast intensity.png"
        output_path = os.path.join(mode_dir("intensity_charts"), output_name)

        output_path = save_figure(
            fig,
            output_path,
            dpi=dpi,
            facecolor=fig.get_facecolor(),
            edgecolor="none"
        )

        plt.show()

        return fig, ax



    fig, ax = plot_intensity_plume(
        df=df,
        storm_id=STORM_ID,
        storm_name=STORM_NAME,
        dark_mode=DARK_MODE,
        aspect_ratio=ASPECT_RATIO,
        dpi=DPI,
        show_credits=SHOW_CREDITS,
        data_credit_text=DATA_CREDIT_TEXT,
        byline_credit_text=BYLINE_CREDIT_TEXT,
        credit_font_size=CREDIT_FONT_SIZE,
    )


if GENERATE_INTENSITY_CHART:
    for_each_color_mode(run_intensity_chart)
else:
    print("Intensity chart: skipped (GENERATE_INTENSITY_CHART = False)")


### 6. Storm impact cards: countries at risk (`GENERATE_STORM_IMPACT_CARDS`)
Social-ready 4K cards (3x2 and 1x1), one per storm: the chance each country gets tropical-storm-force (39+ mph) and hurricane/typhoon-force (74+ mph) winds, plus when they most likely arrive.
- **Chances** come from each WNV3 member's quadrant wind radii (34 kt and 64 kt), interpolated hourly and intersected with Natural Earth countries/territories. A member counts once per country.
- **Timing** is the middle half of members' first arrival of tropical-storm-force winds, in each country's local time. Countries with several time zones use the zone nearest where the winds arrive. The subtitle window and formation dates use the time zone of the card's top-listed country.
- **Storms covered:** existing storms and invests in at least half the members, plus unnamed systems. Unnamed systems are new-genesis tracks grouped across members by where and when they form, and they get a card when they form in at least `IMPACT_MIN_GENESIS_SHARE` of members. New tracks that just restart an existing storm (same member, within 24 hours and 500 km of where its track ended) are folded back into that storm. Their cards add a "Storm forms" row.
- **Names:** names and current strength come from NHC (Atlantic, eastern/central Pacific) and JTWC (all other basins). Override any of them with `IMPACT_STORM_NAMES`.
- **Regional intensity conventions:** each card follows the agency for where the storm is. That's NHC's Saffir-Simpson scale (Atlantic and eastern/central Pacific), JMA (western Pacific), IMD (north Indian Ocean), Météo-France (southwest Indian Ocean), and the Australian scale (Australian region and South Pacific). This sets the storm's title name, the track-color key (labeled with the scale, except Saffir-Simpson, which U.S. readers know) and the wind wording, such as "gale-force" and "very destructive winds". Model winds are 1-minute averages, so they're converted with `WIND_AVERAGING_FACTOR` before classifying against agencies that use 10-minute (or 3-minute) winds.
- **Milestone chance:** every card also shows the chance (out of all members) that the storm becomes its region's headline category, with when it most likely gets there. That's a major hurricane (NHC, 96+ kt), a super typhoon (western Pacific, 130+ kt 1-minute, JTWC/Hong Kong usage), a severe tropical cyclone (Australian scale, Category 3+), an extremely severe cyclonic storm (IMD) or an intense tropical cyclone (Météo-France).
- **Peak sustained winds block (on the map):** floor, most likely and ceiling values of members' peak 1-minute sustained wind (10th percentile, median, 90th percentile by default; `IMPACT_WIND_PERCENTILES`). Unnamed storms count only the members where the storm forms. Values are in mph, rounded to 5, with the regional category under each. The block sits in the map corner with the fewest tracks and no labels.
- **Peak strength (gold glow):** the stretch of the average track where the members' median wind is within `IMPACT_PEAK_FRACTION` (90%) of its forecast maximum. For a storm that's weakening, this is near its current position.
- **This year's strongest (gold line in the peak-wind block):** the forecast peak is compared with this calendar year's storms in the same basin, excluding the storm itself. The source is the working best-track files ("b-decks", NCAR mirror, updated daily): NHC's own for the Atlantic and eastern/central Pacific, and records rebuilt from JTWC's real-time status messages elsewhere. These are preliminary operational values and can differ from final best tracks; IBTrACS is the fallback if the b-decks are unreachable. If at least `IMPACT_RECORD_MIN_CHANCE` (10%) of members top the year's strongest, the card gives that chance and names every storm tied for strongest. Otherwise it notes when the most likely peak would rank among the year's three strongest.


In [ ]:
def run_storm_impact_cards():
    """Per-storm social cards: chance, timing and intensity of winds by country.

    Each member's wind field is built from its 34-kt and 64-kt quadrant radii
    (interpolated to hourly), then intersected with Natural Earth countries.
    A member counts once per country; chance = members / total members.
    """
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    import cartopy.io.shapereader as shpreader
    import shapely
    import shapely.ops
    from shapely.geometry import box
    from shapely.affinity import translate
    from matplotlib.collections import LineCollection
    from matplotlib.patches import FancyBboxPatch
    from matplotlib.offsetbox import VPacker
    import matplotlib.patheffects as pe
    import matplotlib.transforms as mtransforms
    import pytz
    import textwrap
    from matplotlib.textpath import TextToPath

    KM_PER_DEG = 111.2
    STEP_HOURS = 1                       # hourly swaths between 6-hour fixes
    ARC_PTS = 7                          # vertices per wind-radius quadrant
    WIND = "maximum_sustained_wind_speed_knots"
    QUADS = ("ne", "se", "sw", "nw")
    R34 = [f"radius_34_knot_winds_{q}_km" for q in QUADS]
    R64 = [f"radius_64_knot_winds_{q}_km" for q in QUADS]

    COUNTRY_RENAME = {"United States of America": "United States",
                      "People's Republic of China": "China",
                      "The Bahamas": "Bahamas"}

    def card_theme():
        """(theme, colors) for the current color mode."""
        return GLOBAL_MAP["themes"][DARK_MODE], (
            {"ts": "#63c5ff", "hu": "#ff555f", "top": "#d77cff",
             "mean": "#ffffff", "form": "#e9e9e9", "muted": "#b9c0c6",
             "track_bg": "#262c33", "highlight": "#8b9196", "halo": "#000000",
             "peak": "#ffd23f", "peak_text": "#ffd23f"}
            if DARK_MODE else
            {"ts": "#03afca", "hu": "#dc303b", "top": "#7a1fa2",
             "mean": "#000000", "form": "#4a4a4a", "muted": "#555555",
             "track_bg": "#e4e4e4", "highlight": "#b0b0b0", "halo": "#ffffff",
             "peak": "#f5b800", "peak_text": "#8a5a00"})

    theme, colors = card_theme()

    # Layout geometry (figure fractions) for each social format.
    CARD = {
        "3x2": {"title_y": 0.957, "title_size": 36, "sub_y": 0.893,
                "sub_size": 21, "key_y": 0.838, "key_size": 17,
                "map": [0.030, 0.075, 0.560, 0.735],
                "panel": [0.625, 0.075, 0.345, 0.735], "row_in": 1.06,
                "rows": IMPACT_PANEL_ROWS.get("3x2", 6), "bar_frac": 0.50,
                "name_size": 23, "pct_size": 32, "detail_size": 16.5,
                "header_size": 15.5, "bar_in": 0.20, "label_size": 15},
        "1x1": {"title_y": 0.962, "title_size": 30, "sub_y": 0.905,
                "sub_size": 17.5, "key_y": 0.857, "key_size": 14.5,
                "map": [0.030, 0.405, 0.940, 0.425],   # resized to fit rows
                "panel": [0.055, 0.065, 0.890, 0.320], "row_in": 0.85,
                "rows": IMPACT_PANEL_ROWS.get("1x1", 4), "bar_frac": 0.55,
                "name_size": 19, "pct_size": 25, "detail_size": 14,
                "header_size": 13.5, "bar_in": 0.15, "label_size": 13},
    }

    # ------------------------------------------------ helpers
    def storm_label(sid, g, official):
        if sid in IMPACT_STORM_NAMES:
            return IMPACT_STORM_NAMES[sid]
        if sid in official:                             # NHC or JTWC
            return official[sid]["label"]
        num = int(sid[2:4])
        first = g.sort_values("valid_time").groupby("sample").first()
        lat, lon = first["lat"].median(), first["lon"].median()
        suffix = basin_letter(sid[:2], lon)
        if num >= 90:
            return f"Invest {num}{suffix}"
        region = intensity_region(lat, lon, sid[:2])
        return f"{regional_class(first[WIND].median(), region)} {num:02d}{suffix}"

    def storm_region(sid, g):
        first = g.sort_values("valid_time").groupby("sample").first()
        return intensity_region(first["lat"].median(), first["lon"].median(), sid[:2])

    def date_range(a, b):
        if a.date() == b.date():
            return ap_date(a)
        if a.month == b.month:
            return f"{ap_date(a)}-{b.day}"
        return f"{ap_date(a)}-{ap_date(b)}"

    def pct_text(p, total):
        if p <= 0:
            return f"<{max(1, round(100 / total))}%"
        if p >= 99.5 and p < 100:
            return ">99%"
        return f"{p:.0f}%"

    def densify(g, init):
        """Hourly interpolation of position, wind and radii per member track."""
        out = []
        for _, h in g.groupby(["sample", "track_id"], sort=False):
            h = h.sort_values("valid_time")
            t = (h["valid_time"] - init).dt.total_seconds().to_numpy() / 3600
            tt = t if len(t) < 2 else np.arange(t[0], t[-1] + 1e-9, STEP_HOURS)
            lon_u = np.rad2deg(np.unwrap(np.deg2rad(h["lon"].to_numpy(float))))
            d = {"sample": h["sample"].iloc[0], "hours": tt,
                 "lat": np.interp(tt, t, h["lat"]),
                 "lon": (np.interp(tt, t, lon_u) + 180) % 360 - 180}
            for c in [WIND] + R34 + R64:
                d[c] = np.interp(tt, t, h[c].fillna(0).to_numpy(float))
            out.append(pd.DataFrame(d))
        return pd.concat(out, ignore_index=True)

    def wind_field_coords(lat, lon, radii):
        """Quadrant wind-radius polygons -> (N, 4*ARC_PTS, 2) lon/lat rings."""
        bearings = np.deg2rad(np.concatenate(
            [np.linspace(a, a + 90, ARC_PTS) for a in (0, 90, 180, 270)]))
        r = np.repeat(radii, ARC_PTS, axis=1)
        coslat = np.cos(np.deg2rad(lat))[:, None]
        xs = lon[:, None] + r * np.sin(bearings) / (KM_PER_DEG * coslat)
        ys = np.clip(lat[:, None] + r * np.cos(bearings) / KM_PER_DEG, -89.9, 89.9)
        return np.stack([xs, ys], axis=-1)

    def country_hits(pts, radius_cols):
        """(row index, country index) pairs where the wind field touches land."""
        radii = pts[radius_cols].to_numpy(float)
        rows = np.where(radii.max(axis=1) > 0)[0]
        if not len(rows):
            return np.empty(0, int), np.empty(0, int)
        coords = wind_field_coords(pts["lat"].to_numpy()[rows],
                                   pts["lon"].to_numpy()[rows], radii[rows])
        pi, ci = country_tree.query(shapely.polygons(coords), predicate="intersects")
        hits_p, hits_c = [rows[pi]], [ci]
        for shift, mask in ((-360, coords[..., 0].max(1) > 180),
                            (360, coords[..., 0].min(1) < -180)):
            if mask.any():
                shifted = coords[mask].copy()
                shifted[..., 0] += shift
                p2, c2 = country_tree.query(shapely.polygons(shifted),
                                            predicate="intersects")
                hits_p.append(rows[np.where(mask)[0][p2]])
                hits_c.append(c2)
        return np.concatenate(hits_p), np.concatenate(hits_c)

    def country_table(sid, g, init, total):
        pts = densify(g, init)
        tables = {}
        for key, cols in (("ts", R34), ("hu", R64)):
            pi, ci = country_hits(pts, cols)
            tables[key] = pd.DataFrame({
                "sample": pts["sample"].to_numpy()[pi],
                "hours": pts["hours"].to_numpy()[pi],
                "lat": pts["lat"].to_numpy()[pi],
                "lon": pts["lon"].to_numpy()[pi],
                "country": np.asarray(country_names)[ci]})
        ts, hu = tables["ts"], tables["hu"]
        if ts.empty:
            return pd.DataFrame(), pts
        first = ts.sort_values("hours").groupby(["country", "sample"]).first()
        rows = []
        for country, grp in first.groupby(level="country"):
            n_hu = hu.loc[hu["country"] == country, "sample"].nunique()
            # The country's own time zone (nearest zone to where winds arrive).
            tz = local_zone(country, grp["lat"].median(),
                            circular_mean_lon(grp["lon"]))
            utc = init + pd.to_timedelta(grp["hours"], unit="h")
            q25, q75 = to_local(utc, tz).quantile([0.25, 0.75])
            rows.append({
                "storm": sid, "country": country, "time_zone": tz,
                "ts_force_pct": 100 * len(grp) / total,
                "hu_force_pct": 100 * n_hu / total,
                "members_ts_force": len(grp), "members_hu_force": n_hu,
                "arrival_hours_median": grp["hours"].median(),
                "likely_start": q25.floor("D"), "likely_end": q75.floor("D"),
            })
        table = (pd.DataFrame(rows)
                 .sort_values(["ts_force_pct", "hu_force_pct"], ascending=False)
                 .reset_index(drop=True))
        return table, pts

    # ------------------------------------------------ map pieces
    def map_extent(pts, g, central_lon):
        strong = pts[pts[WIND] >= 34]
        use = strong if len(strong) > 20 else pts
        rel = (use["lon"] - central_lon + 180) % 360 - 180
        lons = list(rel.quantile([0.02, 0.98]))
        lats = list(use["lat"].quantile([0.02, 0.98]))
        start = g.sort_values("valid_time").groupby("sample").first()
        lons += list((start["lon"] - central_lon + 180) % 360 - 180)
        lats += list(start["lat"])
        pad = 3.0
        lon0, lon1 = min(lons) - pad, max(lons) + pad
        lat0, lat1 = max(-75, min(lats) - pad), min(75, max(lats) + pad)
        return lon0, lon1, lat0, lat1

    def fit_extent(ax, extent, rect, layout):
        lon0, lon1, lat0, lat1 = extent
        src = ccrs.PlateCarree(central_longitude=ax.projection.proj4_params["lon_0"])
        pts = ax.projection.transform_points(
            src, np.array([lon0, lon1, lon0, lon1]), np.array([lat0, lat0, lat1, lat1]))
        x0, x1 = pts[:, 0].min(), pts[:, 0].max()
        y0, y1 = pts[:, 1].min(), pts[:, 1].max()
        fw, fh = GLOBAL_MAP["figsizes"][layout]
        target = (rect[2] * fw) / (rect[3] * fh)
        xc, yc, w, h = (x0 + x1) / 2, (y0 + y1) / 2, x1 - x0, y1 - y0
        if w / h < target:
            w = h * target
        else:
            h = w / target
        ax.set_xlim(xc - w / 2, xc + w / 2)
        ax.set_ylim(yc - h / 2, yc + h / 2)

    def track_segments(g, central_lon, region):
        t_ts, t_hu, t_top = track_thresholds(region)
        groups = {"ts": [], "hu": [], "top": []}
        for _, h in g.groupby(["sample", "track_id"], sort=False):
            h = h.sort_values("valid_time")
            x = (h["lon"].to_numpy(float) - central_lon + 180) % 360 - 180
            x = np.rad2deg(np.unwrap(np.deg2rad(x)))
            y = h["lat"].to_numpy(float)
            w = h[WIND].to_numpy(float)
            for i in range(len(h) - 1):
                wm = 0.5 * (w[i] + w[i + 1])
                if not np.isfinite(wm) or wm < t_ts:
                    continue
                key = "top" if wm >= t_top else "hu" if wm >= t_hu else "ts"
                groups[key].append([(x[i], y[i]), (x[i + 1], y[i + 1])])
        return groups

    def draw_map(fig, rect, layout, system, pts, listed):
        g = system["g"]
        start = g.sort_values("valid_time").groupby("sample").first()
        central_lon = float(np.rad2deg(np.angle(np.exp(1j * np.deg2rad(
            start["lon"].to_numpy(float))).mean())))
        ax = fig.add_axes(rect, projection=ccrs.Mercator(central_longitude=central_lon))
        ax.set_facecolor(theme["ocean"])
        for spine in ax.spines.values():
            spine.set_visible(False)
        extent = map_extent(pts, g, central_lon)
        fit_extent(ax, extent, rect, layout)
        rel_pc = ccrs.PlateCarree(central_longitude=central_lon)

        ax.add_feature(cfeature.NaturalEarthFeature(
            "physical", "land", "10m", facecolor=theme["land"], edgecolor="none"),
            zorder=1)
        for name in listed:
            ax.add_geometries([country_geoms[name]], ccrs.PlateCarree(),
                              facecolor=colors["highlight"], edgecolor="none",
                              zorder=2)
        add_coastline_with_glow(ax, theme, DARK_MODE, scale="10m", zorder=5)
        # State/province lines for every country, country borders on top.
        ax.add_feature(cfeature.NaturalEarthFeature(
            "cultural", "admin_1_states_provinces_lines", "10m",
            facecolor="none", edgecolor=theme["border"]),
            linewidth=0.45, alpha=0.55, zorder=6)
        ax.add_feature(cfeature.NaturalEarthFeature(
            "cultural", "admin_0_boundary_lines_land", "10m",
            facecolor="none", edgecolor=theme["border"]),
            linewidth=0.8, alpha=0.9, zorder=6)
        if great_lakes:                        # water drawn over every border
            ax.add_geometries(great_lakes, ccrs.PlateCarree(),
                              facecolor=theme["ocean"], edgecolor=theme["coast"],
                              linewidth=0.5, zorder=7)

        segs = track_segments(g, central_lon, system["region"])
        for key in ("ts", "hu", "top"):
            if segs[key]:
                ax.add_collection(LineCollection(
                    segs[key], colors=colors[key], linewidths=1.25, alpha=0.75,
                    capstyle="butt", transform=rel_pc, zorder=10))

        m = system["mean"]
        if m is not None and len(m) > 1:
            m = m.sort_values("valid_time")
            mx = (m["lon"].to_numpy(float) - central_lon + 180) % 360 - 180
            mx = np.rad2deg(np.unwrap(np.deg2rad(mx)))
            # Peak-strength stretch: times when the members' median wind is
            # within IMPACT_PEAK_FRACTION of its forecast maximum.
            if IMPACT_HIGHLIGHT_PEAK:
                n_members = g["sample"].nunique()
                by_time = g.groupby("valid_time").agg(
                    wind=(WIND, "median"), n=("sample", "nunique"))
                by_time = by_time[by_time["n"] >= 0.5 * n_members]
                if len(by_time):
                    strong = by_time.index[by_time["wind"]
                                           >= IMPACT_PEAK_FRACTION * by_time["wind"].max()]
                    t0 = strong.min() - pd.Timedelta(hours=3)
                    t1 = strong.max() + pd.Timedelta(hours=3)
                    vt = m["valid_time"].to_numpy()
                    keep = (vt >= np.datetime64(t0)) & (vt <= np.datetime64(t1))
                    idx = np.where(keep)[0]
                    if len(idx):
                        # one fix on each side so short peaks still read as a stretch
                        lo, hi = max(idx[0] - 1, 0), min(idx[-1] + 1, len(m) - 1)
                        ax.plot(mx[lo:hi + 1], m["lat"].to_numpy()[lo:hi + 1],
                                color=colors["peak"], linewidth=15, alpha=0.6,
                                solid_capstyle="round", transform=rel_pc, zorder=19)
            if True:
                ax.plot(mx, m["lat"], color=colors["mean"], linewidth=3.2,
                        solid_capstyle="round", transform=rel_pc, zorder=20,
                        path_effects=[pe.Stroke(linewidth=6.5,
                                                foreground=colors["halo"],
                                                alpha=0.55), pe.Normal()])

        x0 = (start["lon"].median() - central_lon + 180) % 360 - 180
        ax.scatter([x0], [start["lat"].median()], s=170, facecolor=colors["mean"],
                   edgecolor=colors["halo"], linewidth=2.2, transform=rel_pc,
                   zorder=25)

        # Track points (densified) in display pixels: labels and the
        # peak-wind block stay clear of them. `placed` collects label boxes.
        renderer = fig.canvas.get_renderer()
        placed = []
        pts_rel = np.concatenate(
            [np.linspace(s[0], s[1], 6) for k in segs for s in
             np.asarray(segs[k], dtype=float).reshape(-1, 2, 2)]
            or [np.empty((0, 2))])
        proj = ax.projection.transform_points(rel_pc, pts_rel[:, 0], pts_rel[:, 1])
        track_xy = ax.transData.transform(proj[:, :2]) if len(proj) else \
            np.empty((0, 2))
        start_xy = ax.transData.transform(ax.projection.transform_point(
            x0, start["lat"].median(), rel_pc))
        placed.append(mtransforms.Bbox.from_extents(
            start_xy[0] - 20, start_xy[1] - 20, start_xy[0] + 20, start_xy[1] + 20))

        # Places to label: one anchor per listed country; tiny islands get a
        # dot, and every dot is an obstacle that no label or block may cover.
        anchors, keep_out = [], []
        if IMPACT_LABEL_COUNTRIES and listed:
            xl, yl = ax.get_xlim(), ax.get_ylim()
            corners = rel_pc.transform_points(
                ax.projection, np.array([xl[0], xl[1], xl[0], xl[1]]),
                np.array([yl[0], yl[0], yl[1], yl[1]]))
            view = box(corners[:, 0].min(), corners[:, 1].min(),
                       corners[:, 0].max(), corners[:, 1].max())
            for name in listed:
                parts = []
                for shift in (-central_lon, -central_lon + 360, -central_lon - 360):
                    clip = translate(country_geoms[name], shift).intersection(view)
                    parts += list(getattr(clip, "geoms", [clip]))
                parts = [q for q in parts if q.geom_type in ("Polygon", "MultiPolygon")
                         and not q.is_empty]
                if not parts:
                    continue
                biggest = max(parts, key=lambda q: q.area)
                p = biggest.representative_point()
                anchors.append((name, p, biggest.area < 1.0))    # square degrees
            for name, p, island in anchors:
                xy = ax.transData.transform(ax.projection.transform_point(
                    p.x, p.y, rel_pc))
                if island:
                    ax.scatter([p.x], [p.y], s=22, color=theme["text"],
                               edgecolor=colors["halo"], linewidth=1.2,
                               transform=rel_pc, zorder=29)
                    r = 0.12 * fig.dpi * CARD[layout]["label_size"] / 15
                    placed.append(mtransforms.Bbox.from_extents(
                        xy[0] - r, xy[1] - r, xy[0] + r, xy[1] + r))
                # Room for the label itself, kept free of the wind block.
                half_w = 0.33 * len(name) * CARD[layout]["label_size"] / 72 * fig.dpi
                half_h = 1.6 * CARD[layout]["label_size"] / 72 * fig.dpi
                keep_out.append(mtransforms.Bbox.from_extents(
                    xy[0] - half_w, xy[1] - half_h, xy[0] + half_w, xy[1] + half_h))

        # Peak-wind block: horizontal or vertical, anywhere on the map, wherever
        # the fewest tracks fall without covering places, dots or the start.
        block = None
        if IMPACT_SHOW_PEAK_WINDS:
            frame = ax.get_window_extent(renderer)
            m_px = 0.15 * fig.dpi
            best = None
            for orient in ("h", "v"):
                w_in, h_in = wind_block_size(layout, system, orient)
                w_px, h_px = w_in * fig.dpi, h_in * fig.dpi
                xs = np.linspace(frame.x0 + m_px, frame.x1 - m_px - w_px, 13)
                ys = np.linspace(frame.y0 + m_px, frame.y1 - m_px - h_px, 11)
                if xs[-1] < xs[0] or ys[-1] < ys[0]:
                    continue                       # does not fit this map
                for x_0 in xs:
                    for y_0 in ys:
                        bb = mtransforms.Bbox.from_bounds(x_0, y_0, w_px, h_px)
                        clash = sum(bb.overlaps(o) for o in placed + keep_out)
                        busy = int(((track_xy[:, 0] >= bb.x0)
                                    & (track_xy[:, 0] <= bb.x1)
                                    & (track_xy[:, 1] >= bb.y0)
                                    & (track_xy[:, 1] <= bb.y1)).sum())
                        # Tie-breaks: horizontal first, then hugging an edge.
                        edge = min(x_0 - frame.x0, frame.x1 - bb.x1,
                                   y_0 - frame.y0, frame.y1 - bb.y1)
                        score = (clash, busy, orient == "v", edge)
                        if best is None or score < best[0]:
                            best = (score, bb, orient)
            if best is not None:
                block = (best[1], best[2])
                placed.append(best[1].padded(0.1 * fig.dpi))

        if anchors:
            # Pass 2: labels. Try spots around each place; skip any that crowd
            # a label or dot or leave the map; pick the one with the fewest
            # track lines underneath, weighted by distance from the place.
            frame = ax.get_window_extent(renderer)
            for name, p, island in anchors:
                t = ax.text(p.x, p.y, name, transform=rel_pc, ha="center",
                            va="center", color=theme["text"],
                            fontproperties=font_map_bold,
                            fontsize=CARD[layout]["label_size"], zorder=30,
                            path_effects=[pe.withStroke(
                                linewidth=3.5, foreground=colors["halo"])])
                base = t.get_window_extent(renderer)
                w, h = base.width, base.height
                pad_x, pad_y = 1.2 * h, 0.8 * h
                best = None
                for ring in (1, 2):
                    for ux, uy in ((0, 0), (0, -1), (0, 1), (-1, 0), (1, 0),
                                   (-1, -1), (1, -1), (-1, 1), (1, 1)):
                        if (ux, uy) == (0, 0) and (ring == 2 or island):
                            continue                 # keep island dots visible
                        # Side spots put the label's near edge beside the place.
                        dx = ux * (0.5 * w + (0.5 + 1.5 * (ring - 1)) * h)
                        dy = uy * 1.3 * ring * h
                        bb = base.translated(dx, dy)
                        if (bb.x0 < frame.x0 or bb.x1 > frame.x1
                                or bb.y0 < frame.y0 or bb.y1 > frame.y1):
                            continue
                        if any(bb.x0 - pad_x < o.x1 and bb.x1 + pad_x > o.x0
                               and bb.y0 - pad_y < o.y1 and bb.y1 + pad_y > o.y0
                               for o in placed):
                            continue
                        busy = int(((track_xy[:, 0] >= bb.x0) & (track_xy[:, 0] <= bb.x1)
                                    & (track_xy[:, 1] >= bb.y0)
                                    & (track_xy[:, 1] <= bb.y1)).sum())
                        # Clear water matters, but staying close matters more.
                        score = 20 * min(busy, 100) / 100 + 6 * np.hypot(dx, dy) / h
                        if best is None or score < best[0]:
                            best = (score, dx, dy, bb)
                if best is None:
                    t.remove()
                    continue
                _, dx, dy, bb = best
                if dx or dy:
                    t.set_transform(t.get_transform() + mtransforms.ScaledTranslation(
                        dx / fig.dpi, dy / fig.dpi, fig.dpi_scale_trans))
                placed.append(bb)
        return ax, block

    # ------------------------------------------------ text pieces
    def add_fitted_text(fig, x, y, text, size, font, color, max_frac=0.94, **kw):
        artist = fig.text(x, y, text, fontproperties=font, fontsize=size,
                          color=color, **kw)
        renderer = fig.canvas.get_renderer()
        while (artist.get_window_extent(renderer).width > max_frac * fig.bbox.width
               and size > 10):
            size -= 1
            artist.set_fontsize(size)
        return artist

    def add_key_row(fig, layout, region):
        """Track-color key under the subtitle; wraps to two lines if needed.
        Returns the key's bottom edge (figure fraction)."""
        scale = REGIONAL_SCALES[region]
        (_, l_ts), (_, l_hu), (_, l_top) = scale["tracks"]
        entries = [(colors["ts"], l_ts), (colors["hu"], l_hu),
                   (colors["top"], l_top), (colors["mean"], "Average track")]
        if IMPACT_HIGHLIGHT_PEAK:
            entries.append((colors["peak"], "Peak strength"))
        rect = CARD[layout]["map"]
        renderer = fig.canvas.get_renderer()

        def item(color, label, size):
            swatch = DrawingArea(22, 12, 0, 0)
            swatch.add_artist(Line2D(
                [0, 22], [6, 6], color=color, solid_capstyle="round",
                alpha=0.6 if label == "Peak strength" else 1,
                linewidth={"Average track": 4, "Peak strength": 9}.get(label, 3)))
            return HPacker(children=[swatch, TextArea(label, textprops=dict(
                color=theme["text"], fontproperties=font_map_light,
                fontsize=size))], align="center", pad=0, sep=6)

        def build(size, split):
            prefix = ([TextArea(f"{scale['scale']}:", textprops=dict(
                color=colors["muted"], fontproperties=font_map_light,
                fontsize=size))] if scale.get("show_scale", True) else [])
            items = [item(c_, l_, size) for c_, l_ in entries]
            rows = ([prefix + items] if not split else
                    [prefix + items[:3], items[3:]])
            child = VPacker(children=[HPacker(children=r, align="center", pad=0,
                                              sep=16) for r in rows],
                            align="left", pad=0, sep=8)
            top = CARD[layout]["key_y"] + size / 72 * 0.6 / fig.get_size_inches()[1]
            box = AnchoredOffsetbox(loc="upper left", child=child, pad=0,
                                    borderpad=0, frameon=False,
                                    bbox_to_anchor=(rect[0] + 0.004, top),
                                    bbox_transform=fig.transFigure)
            fig.add_artist(box)
            fig.canvas.draw()                   # true extent of the key
            return box

        base = CARD[layout]["key_size"]
        for split, smallest in ((False, 0.85 * base), (True, 10)):
            size = base
            while True:
                box = build(size, split)
                bb = box.get_window_extent(renderer)
                if bb.x1 <= 0.965 * fig.bbox.width or size <= smallest:
                    break
                box.remove()
                size -= 0.5
            if bb.x1 <= 0.965 * fig.bbox.width or split:
                return fig.transFigure.inverted().transform((0, bb.y0))[1]
            box.remove()

    def peak_winds(system):
        """Floor / typical / ceiling of members' peak winds (knots)."""
        peaks = system["g"].groupby("sample")[WIND].max().dropna()
        lo, hi = IMPACT_WIND_PERCENTILES
        return {"floor": np.percentile(peaks, lo), "typical": np.median(peaks),
                "ceiling": np.percentile(peaks, hi), "members": len(peaks)}

    def mph(kt):
        return int(5 * round(kt * 1.15078 / 5))

    def record_lines(layout, system, orient="h"):
        text = system.get("record")
        if not text:
            return []
        glued = re.sub(r"(\d) mph", r"\1~mph", text)      # keep "175 mph" together
        return [ln.replace("~", " ")
                for ln in textwrap.wrap(glued, (50 if layout == "3x2" else 44)
                                        if orient == "h" else 28)]

    WIND_KEYS = (("floor", "Floor"), ("typical", "Most likely"), ("ceiling", "Ceiling"))

    def text_width_in(text, prop, size):
        """Width (in) of text at a font size, measured without a figure."""
        p = prop.copy()
        p.set_size(size)
        return TextToPath().get_text_width_height_descent(text, p, ismath=False)[0] / 72

    def wrap_to_width(text, prop, size, room):
        """Text as one line, or split at the word break that best balances two
        lines when it is wider than room (in)."""
        words = text.split()
        if text_width_in(text, prop, size) <= room or len(words) < 2:
            return [text]
        splits = [(" ".join(words[:i]), " ".join(words[i:])) for i in range(1, len(words))]
        return list(min(splits, key=lambda pair: max(
            text_width_in(part, prop, size) for part in pair)))

    def wind_categories(layout, system, orient, sizes, w):
        """Category label under each peak-wind value, wrapped to its column."""
        pad, small = 0.16, sizes[2]
        room = (w - 2 * pad) / 3 - 0.08
        labels = [strength_label(system["peak"][k], system["region"]) for k, _ in WIND_KEYS]
        if orient != "h":
            return [[label] for label in labels]
        return [wrap_to_width(label, font_map_light, small, room) for label in labels]

    def wind_block_layout(layout, system, orient="h"):
        """Sizes (pt) and vertical offsets from the top (in) of the peak-wind
        block, so the box always matches its contents. "h" = three columns,
        "v" = three rows (value on the left, label and category on the right)."""
        big = layout == "3x2"
        title, value, small = (16, 23, 13) if big else (13.5, 18.5, 11)
        if orient == "v":
            value = 21 if big else 17
        pad, line = 0.16, small / 72 * 1.3
        y = {"value": title / 72 * 1.4}
        value_col = 1.30 if big else 1.05
        if orient == "h":
            w = 4.70 if big else 3.75
            cat_rows = max(len(c) for c in wind_categories(
                layout, system, orient, (title, value, small), w))
            y["label"] = y["value"] + value / 72 * 1.25
            y["cat"] = y["label"] + line
            bottom = y["cat"] + cat_rows * line
        else:
            y["row"] = max(value / 72 * 1.2, 2 * line) + 0.08
            bottom = y["value"] + 3 * y["row"] - 0.06
            # Widen the box so the longest category label fits on one line.
            widest = max(text_width_in(strength_label(system["peak"][k], system["region"]),
                                       font_map_light, small) for k, _ in WIND_KEYS)
            w = max(2.95 if big else 2.55, 2 * pad + value_col + widest + 0.04)
        n = len(record_lines(layout, system, orient))
        if n:
            y["rule"] = bottom + 0.08
            y["footer"] = y["rule"] + 0.09
            bottom = y["footer"] + n * line
        return {"w": w, "h": bottom + 2 * pad, "pad": pad, "line": line, "y": y,
                "sizes": (title, value, small), "orient": orient,
                "value_col": value_col}

    def wind_block_size(layout, system, orient="h"):
        lay = wind_block_layout(layout, system, orient)
        return lay["w"], lay["h"]

    def draw_wind_block(fig, layout, system, block):
        """Peak-wind inset in the spot (and orientation) chosen by draw_map."""
        block_bb, orient = block
        stats, region = system["peak"], system["region"]
        lay = wind_block_layout(layout, system, orient)
        title_size, value_size, small_size = lay["sizes"]
        w_in, h_in, pad, line_h = lay["w"], lay["h"], lay["pad"], lay["line"]
        top = h_in - pad
        renderer = fig.canvas.get_renderer()
        bb = block_bb.transformed(fig.transFigure.inverted())
        ax = fig.add_axes([bb.x0, bb.y0, bb.width, bb.height], zorder=50)
        ax.set_xlim(0, w_in)
        ax.set_ylim(0, h_in)
        ax.axis("off")
        ax.add_patch(FancyBboxPatch((0, 0), w_in, h_in,
                                    boxstyle="round,pad=0,rounding_size=0.08",
                                    facecolor=theme["fig"], alpha=0.88,
                                    edgecolor=colors["track_bg"], linewidth=1.2))
        ax.text(pad, top, "Peak sustained winds", va="top", ha="left",
                color=theme["text"], fontproperties=font_map_bold,
                fontsize=title_size)
        to_data = ax.transData.inverted()
        keys = WIND_KEYS
        categories = wind_categories(layout, system, orient, lay["sizes"], w_in)
        if orient == "h":
            col_w = (w_in - 2 * pad) / 3
            spots = [(pad + i * col_w, top - lay["y"]["value"]) for i in range(3)]
            text_spots = [(x, y - (lay["y"]["label"] - lay["y"]["value"]))
                          for x, y in spots]
            value_room = [x + col_w - 0.12 for x, _ in spots]
            text_room = [x + col_w - 0.08 for x, _ in spots]
        else:
            vc = lay["value_col"]
            spots = [(pad, top - lay["y"]["value"] - i * lay["y"]["row"])
                     for i in range(3)]
            text_spots = [(pad + vc, y) for _, y in spots]
            value_room = [pad + vc - 0.10] * 3
            text_room = [w_in - pad] * 3
        values = [ax.text(x, y, f"{mph(stats[k])} mph", va="top", ha="left",
                          color=theme["text"], fontproperties=font_map_bold,
                          fontsize=value_size)
                  for (x, y), (k, _) in zip(spots, keys)]
        # One shared size for the three values, small enough for every slot.
        while value_size > 10 and any(
                v.get_window_extent(renderer).transformed(to_data).x1 > room
                for v, room in zip(values, value_room)):
            value_size -= 0.5
            for v in values:
                v.set_fontsize(value_size)
        for (x, y), room, (key, label), cat_lines in zip(text_spots, text_room,
                                                          keys, categories):
            ax.text(x, y, label, va="top", ha="left", color=colors["muted"],
                    fontproperties=font_map_light, fontsize=small_size)
            for j, part in enumerate(cat_lines):
                cat = ax.text(x, y - (j + 1) * line_h, part,
                              va="top", ha="left", color=theme["text"],
                              fontproperties=font_map_light, fontsize=small_size)
                size = small_size              # last resort for a single long word
                while (cat.get_window_extent(renderer).transformed(to_data).x1 > room
                       and size > 8):
                    size -= 0.5
                    cat.set_fontsize(size)
        lines = record_lines(layout, system, orient)
        if lines:
            ax.plot([pad, w_in - pad], [top - lay["y"]["rule"]] * 2,
                    color=colors["track_bg"], linewidth=1)
            for k, text in enumerate(lines):
                ax.text(pad, top - lay["y"]["footer"] - k * line_h, text,
                        va="top", ha="left", color=colors["peak_text"],
                        fontproperties=font_map_bold, fontsize=small_size)
        return ax

    # ------------------------------------------------ this year's strongest
    IBTRACS_URL = ("https://www.ncei.noaa.gov/data/international-best-track-archive-"
                   "for-climate-stewardship-ibtracs/v04r01/access/csv/"
                   "ibtracs.last3years.list.v04r01.csv")
    IBTRACS_BASIN_NAMES = {"NA": "Atlantic", "EP": "eastern Pacific",
                           "WP": "western Pacific", "NI": "north Indian Ocean",
                           "SI": "south Indian Ocean", "SP": "South Pacific"}

    BDECK_URL = "https://hurricanes.ral.ucar.edu/repository/data/bdecks_open/"
    BDECK_BASINS = {"al": "NA", "ep": "EP", "cp": "EP", "wp": "WP", "io": "NI"}

    def load_bdeck_peaks():
        """Peak 1-minute wind per storm and basin this calendar year from
        working best tracks ("b-decks", NCAR mirror, updated daily). NHC
        basins are NHC's own; JTWC basins are rebuilt from JTWC's real-time
        TC vitals, so they are preliminary operational values."""
        from concurrent.futures import ThreadPoolExecutor
        files = []
        for year in (init.year, init.year + 1):     # SH seasons span two years
            try:
                index = SESSION.get(f"{BDECK_URL}{year}/", timeout=30).text
            except requests.RequestException:
                continue
            files += [(year, f) for f in sorted(set(re.findall(
                r'href="(b[a-z]{2}\d{6}\.dat)"', index)))
                if int(f[3:5]) < 80]                # skip invests/tests
        if not files:
            return None

        def read(item):
            year, f = item
            try:
                text = SESSION.get(f"{BDECK_URL}{year}/{f}", timeout=30).text
            except requests.RequestException:
                return []
            rows = []
            for line in text.splitlines():
                x = [v.strip() for v in line.split(",")]
                if len(x) < 9 or not x[8].isdigit() or not x[2][:4].isdigit():
                    continue
                if int(x[2][:4]) != init.year:
                    continue
                lon = int(x[7][:-1]) / 10 * (1 if x[7].endswith("E") else -1)
                code = f[1:3]
                basin = (BDECK_BASINS.get(code) if code != "sh" else
                         "SI" if 10 <= lon < 135 else "SP")
                rows.append({"BASIN": basin, "SID": f[1:9].upper(),
                             "atcf": f[1:9].upper(),
                             "name": x[27] if len(x) > 27 and x[27] else "UNNAMED",
                             "peak": int(x[8])})
            return rows

        with ThreadPoolExecutor(8) as pool:
            rows = [r for chunk in pool.map(read, files) for r in chunk]
        if not rows:
            return None
        b = pd.DataFrame(rows)
        b["name"] = b.groupby("SID")["name"].transform(
            lambda s: next((n for n in s[::-1] if n not in ("UNNAMED", "INVEST")
                            and not is_number_name(n)), s.iloc[-1]))
        peaks = (b.groupby(["BASIN", "SID"])
                 .agg(name=("name", "first"), atcf=("atcf", "first"),
                      peak=("peak", "max")).reset_index())
        print(f"Season context: {peaks['SID'].nunique()} storms from best-track "
              f"files (b-decks) for {init.year}")
        return peaks

    def load_season_peaks():
        """b-decks first (daily, precise); IBTrACS as a fallback."""
        peaks = load_bdeck_peaks()
        return peaks if peaks is not None else load_ibtracs_peaks()

    def load_ibtracs_peaks():
        """Peak 1-minute wind per storm and basin this calendar year (IBTrACS,
        provisional for recent storms; updated about weekly)."""
        path = os.path.join(output_dir("data"), "ibtracs.last3years.list.v04r01.csv")
        try:
            if not os.path.exists(path):
                path = download(IBTRACS_URL, output_dir("data"))
            ib = pd.read_csv(path, skiprows=[1], low_memory=False,
                             usecols=["SID", "BASIN", "NAME", "ISO_TIME",
                                      "USA_ATCF_ID", "USA_WIND"],
                             keep_default_na=False, na_values=["", " "])
        except Exception as exc:
            print(f"Season context unavailable (IBTrACS): {exc}")
            return None
        ib["USA_WIND"] = pd.to_numeric(ib["USA_WIND"], errors="coerce")
        ib = ib[pd.to_datetime(ib["ISO_TIME"]).dt.year == init.year]
        return (ib.groupby(["BASIN", "SID"])
                .agg(name=("NAME", "first"), atcf=("USA_ATCF_ID", "first"),
                     peak=("USA_WIND", "max"))
                .dropna(subset=["peak"]).reset_index())

    def ibtracs_basin(system):
        """IBTrACS basin code where the storm is forecast to peak."""
        g = system["g"]
        at_peak = g.loc[g.groupby("sample")[WIND].idxmax()]
        lat, lon = at_peak["lat"].median(), circular_mean_lon(at_peak["lon"])
        if lat < 0:
            return "SI" if 10 <= lon < 135 else "SP"
        sid = system["sid"]
        if sid[:2] in ("AL",):
            return "NA"
        if sid[:2] in ("EP", "CP"):
            return "EP"
        return {"NHC": "NA" if basin_name(lat, lon) == "Atlantic" else "EP",
                "JMA": "WP", "IMD": "NI"}.get(intensity_region(lat, lon), "WP")

    def season_word(kt, basin):
        if basin in ("NA", "EP"):
            return "Hurricane" if kt >= 64 else "Tropical Storm"
        if basin == "WP":
            return ("Super Typhoon" if kt >= 130 else "Typhoon" if kt >= 64
                    else "Tropical Storm")
        return "Cyclone"

    def season_named(row, basin):
        name = str(row["name"]).title()
        if name in ("Unnamed", "Not_Named", "Nan") or is_number_name(name):
            return f"a {mph(row['peak'])}-mph storm"
        kt = row["peak"]
        if basin in ("NA", "EP"):
            word = "Hurricane" if kt >= 64 else "Tropical Storm"
        elif basin == "WP":
            word = ("Super Typhoon" if kt >= 130 else "Typhoon" if kt >= 64
                    else "Tropical Storm")
        else:
            word = "Cyclone"
        return f"{word} {name} ({mph(kt)} mph)"

    def season_context(system, total):
        """One line comparing the forecast peak with this year's strongest."""
        if season_peaks is None:
            return None
        basin = ibtracs_basin(system)
        others = season_peaks[(season_peaks["BASIN"] == basin)
                              & (season_peaks["atcf"].astype(str) != system["sid"])]
        if others.empty:
            return None
        ranked = others.sort_values("peak", ascending=False).reset_index(drop=True)
        top = ranked.iloc[0]
        if len(ranked) < 3 or top["peak"] < 64:
            return None                  # too few or too weak to be a benchmark
        tied = ranked[ranked["peak"] == top["peak"]]
        peaks = system["g"].groupby("sample")[WIND].max().dropna()
        chance = 100 * (peaks > top["peak"]).sum() / total
        where = IBTRACS_BASIN_NAMES[basin]
        if chance >= IMPACT_RECORD_MIN_CHANCE:
            if len(tied) == 1:
                rival = season_named(top, basin).replace(" mph)", " mph, preliminary)")
            else:
                names = [str(n).title() for n in tied["name"]]
                joined = (", ".join(names[:-1]) + " and " + names[-1])
                rival = (f"{season_word(top['peak'], basin)}s {joined} "
                         f"({mph(top['peak'])} mph, preliminary)")
            return (f"{chance:.0f}% chance of topping {rival} as "
                    f"the {where}'s strongest storm of {init.year}")
        if len(ranked) >= 3 and np.median(peaks) >= ranked["peak"].iloc[2]:
            return f"Could rank among the {where}'s three strongest storms of {init.year}"
        return None

    def milestone_chance(system, total):
        """Chance (all members) the system reaches its region's milestone,
        plus the UTC window when it most likely first gets there."""
        name, kt, minutes = REGIONAL_SCALES[system["region"]]["milestone"]
        g = system["g"]
        threshold = kt / WIND_AVERAGING_FACTOR[minutes]      # as 1-minute wind
        first = (g[g[WIND] >= threshold].sort_values("valid_time")
                 .groupby("sample")["valid_time"].first())
        now = g[g["lead_time_hours"] == 0]
        already = (len(now) > 0 and now["sample"].nunique() >= 0.5 * total
                   and now[WIND].median() >= threshold)
        q25, q75 = (first.quantile([0.25, 0.75]) if len(first)
                    else (None, None))
        return {"name": name, "pct": 100 * len(first) / total,
                "already": already, "start_utc": q25, "end_utc": q75}

    def panel_rows(system, table, n_rows, total, tz):
        """Formation row (genesis systems only), then the top countries."""
        strong = REGIONAL_SCALES[system["region"]]["winds"][1].capitalize()
        rows = []
        f = system["formation"]
        if f is not None:
            when = f"Most likely forms {formation_dates(f, tz)}"
            rows.append({"name": "Storm forms", "pct": f["pct"], "bars": [
                (f["pct"], colors["form"])], "detail": when})
        m = system["milestone"]
        article = "an" if m["name"][0] in "aeiou" else "a"
        if m["already"]:
            when = f"Already {article} {m['name']}"
        elif m["pct"] > 0:
            when = ("Most likely " + date_range(
                to_local(m["start_utc"], tz).floor("D"),
                to_local(m["end_utc"], tz).floor("D")))
        else:
            when = "Not in any forecast"
        rows.append({"name": f"Becomes {article} {m['name']}", "pct": m["pct"],
                     "bars": [(m["pct"], colors["form"])], "detail": when})
        for r in table.head(max(1, n_rows - len(rows))).itertuples():
            timing = ("Winds arriving now" if r.arrival_hours_median <= 3 else
                      f"Most likely {date_range(r.likely_start, r.likely_end)}")
            rows.append({"name": r.country, "pct": r.ts_force_pct,
                         "bars": [(r.ts_force_pct, colors["ts"]),
                                  (r.hu_force_pct, colors["hu"])],
                         "detail": f"{timing}  ·  {strong}: "
                                   f"{pct_text(r.hu_force_pct, total)}"})
        return rows

    def draw_panel(fig, spec, rows, region, total):
        rect = spec["panel"]
        fw, fh = fig.get_size_inches()
        w_in, h_in = rect[2] * fw, rect[3] * fh
        ax = fig.add_axes(rect)
        ax.set_xlim(0, w_in)
        ax.set_ylim(0, h_in)
        ax.axis("off")
        gale, strong = REGIONAL_SCALES[region]["winds"]

        # Header: what the two bar colors mean (wraps if too wide).
        hs = spec["header_size"]
        line_h = hs / 72 * 1.55
        y = h_in - line_h / 2
        x = 0.0
        renderer = fig.canvas.get_renderer()
        for color, label in ((colors["ts"], f"{gale.capitalize()} winds (39+ mph)"),
                             (colors["hu"], f"{strong.capitalize()} winds (74+ mph)")):
            t = ax.text(x + 0.40, y, label, va="center", ha="left",
                        color=theme["text"], fontproperties=font_map_light,
                        fontsize=hs)
            bb = t.get_window_extent(renderer).transformed(ax.transData.inverted())
            if x > 0 and bb.x1 > w_in:
                x, y = 0.0, y - line_h
                t.set_position((x + 0.40, y))
                bb = t.get_window_extent(renderer).transformed(
                    ax.transData.inverted())
            ax.add_patch(FancyBboxPatch((x, y - 0.075), 0.30, 0.15,
                                        boxstyle="round,pad=0,rounding_size=0.04",
                                        facecolor=color, edgecolor="none"))
            x = bb.x1 + 0.35

        top = y - line_h / 2 - 0.30
        row_h, bar_h = spec["row_in"], spec["bar_in"]
        rounding = bar_h / 2
        for i, row in enumerate(rows):
            y0 = top - i * row_h                      # top of this row
            name = ax.text(0, y0 - 0.02, row["name"], va="top", ha="left",
                           color=theme["text"], fontproperties=font_map_bold,
                           fontsize=spec["name_size"])
            pct = ax.text(w_in, y0 + 0.03, pct_text(row["pct"], total), va="top",
                          ha="right", color=theme["text"],
                          fontproperties=font_map_bold, fontsize=spec["pct_size"])
            # Long labels shrink until they clear the percentage.
            to_data = ax.transData.inverted()
            limit = (pct.get_window_extent(renderer).transformed(to_data).x0
                     - 0.20)
            size = spec["name_size"]
            while (name.get_window_extent(renderer).transformed(to_data).x1 > limit
                   and size > 10):
                size -= 0.5
                name.set_fontsize(size)
            by = y0 - spec["bar_frac"] * row_h
            ax.add_patch(FancyBboxPatch((0, by), w_in, bar_h,
                                        boxstyle=f"round,pad=0,rounding_size={rounding}",
                                        facecolor=colors["track_bg"], edgecolor="none"))
            for value, color in row["bars"]:
                width = w_in * value / 100
                if width <= 0:
                    continue
                ax.add_patch(FancyBboxPatch(
                    (0, by), max(width, bar_h), bar_h,
                    boxstyle=f"round,pad=0,rounding_size={rounding}",
                    facecolor=color, edgecolor=theme["fig"], linewidth=1.5))
            ax.text(0, by - 0.10, row["detail"], va="top", ha="left",
                    color=colors["muted"], fontproperties=font_map_light,
                    fontsize=spec["detail_size"])
        return ax

    def render_card(layout, system, pts, table, total, init):
        spec = dict(CARD[layout])
        tz = table["time_zone"].iloc[0]        # card dates: top country's zone
        rows = panel_rows(system, table, spec["rows"], total, tz)
        if layout == "1x1":
            # Panel only as tall as its rows; the map takes the rest.
            fh = GLOBAL_MAP["figsizes"][layout][1]
            panel_h = (0.75 + len(rows) * spec["row_in"]) / fh
            px, py, pw, _ = spec["panel"]
            spec["panel"] = [px, py, pw, panel_h]
            map_y = py + panel_h + 0.025
            mx, _, mw, _ = spec["map"]
            spec["map"] = [mx, map_y, mw, spec["key_y"] - 0.028 - map_y]
        fig = plt.figure(figsize=GLOBAL_MAP["figsizes"][layout],
                         facecolor=theme["fig"])
        add_fitted_text(fig, 0.030, spec["title_y"], system["title"],
                        spec["title_size"], font_map_bold, theme["text"],
                        va="top", ha="left")
        end = init + pd.Timedelta(hours=float(system["g"]["lead_time_hours"].max()))
        f = system["formation"]
        gale = REGIONAL_SCALES[system["region"]]["winds"][0]
        subtitle = (f"Chance of {gale} winds by country, "
                    f"{date_range(to_local(init, tz), to_local(end, tz))}")
        if f is not None:
            subtitle = (f"Chance of {gale} winds by country · "
                        f"Could form {formation_dates(f, tz)}")
        add_fitted_text(fig, 0.030, spec["sub_y"], subtitle,
                        spec["sub_size"], font_map_light, theme["text"],
                        va="top", ha="left")
        key_bottom = add_key_row(fig, layout, system["region"])
        mx, my, mw, mh = spec["map"]
        if my + mh > key_bottom - 0.012:          # map starts below the key
            spec["map"] = [mx, my, mw, key_bottom - 0.012 - my]
        listed = [r["name"] for r in rows if r["name"] in country_geoms]
        map_ax, block = draw_map(fig, spec["map"], layout, system, pts, listed)
        if block is not None:
            draw_wind_block(fig, layout, system, block)
        draw_panel(fig, spec, rows, system["region"], total)
        add_global_credits(fig, theme, layout,
                           data_text=data_credit_text(RUN, TRACK_MODEL))
        path = os.path.join(mode_dir("impact_cards"),
                            f"{system['slug']}_country_winds_{layout}")
        save_figure(fig, path, dpi=global_map_dpi(layout),
                    facecolor=fig.get_facecolor(), edgecolor="none",
                    bbox_inches=None, pad_inches=0)
        plt.show()
        plt.close(fig)

    # ------------------------------------------------ unnamed (genesis) systems
    def basin_name(lat, lon):
        """Readable ocean name for a genesis point (used in card titles)."""
        lon = (float(lon) + 180) % 360 - 180
        region = intensity_region(lat, lon)
        if region == "IMD":
            return "Arabian Sea" if lon < 77 else "Bay of Bengal"
        if region == "NHC":
            if lat < 0:
                return "South Atlantic"
            if -100 < lon < 0 and not (lat < 17 and lon < -87) \
                    and not (lat < 9 and lon < -77):
                return "Atlantic"
            return "central Pacific" if lon < -140 else "eastern Pacific"
        return {"JMA": "western Pacific", "MFR": "southwest Indian Ocean",
                "BOM": "Australian region", "FMS": "South Pacific"}[region]

    def mean_track(g, min_share=0.5):
        """Average position by lead time where most cluster members exist."""
        n = g["sample"].nunique()
        rows = []
        for lead, grp in g.groupby("lead_time_hours"):
            if grp["sample"].nunique() < min_share * n:
                continue
            ang = np.exp(1j * np.deg2rad(grp["lon"].to_numpy(float))).mean()
            rows.append({"valid_time": grp["valid_time"].iloc[0],
                         "lat": grp["lat"].mean(),
                         "lon": float(np.rad2deg(np.angle(ang)))})
        return pd.DataFrame(rows)

    def genesis_systems(base_ids, total, init):
        """Cluster new-genesis tracks (numeric IDs) into systems across members.

        Tracks reaching tropical-storm strength whose first fixes lie within
        GENESIS_RADIUS_KM and GENESIS_WINDOW_HOURS of each other are one system.
        """
        gen = df[~base_ids.str.match(r"^[A-Z]{2}\d{6}$")]
        if gen.empty:
            return []
        tracks = gen.sort_values("valid_time").groupby(["sample", "track_id"])
        first = tracks.first()
        first = first[tracks[WIND].max() >= 34]
        if first.empty:
            return []
        lat = np.deg2rad(first["lat"].to_numpy(float))
        lon = np.deg2rad(first["lon"].to_numpy(float))
        hrs = (first["valid_time"] - init).dt.total_seconds().to_numpy() / 3600
        a = (np.sin((lat[:, None] - lat[None]) / 2) ** 2 + np.cos(lat[:, None])
             * np.cos(lat[None]) * np.sin((lon[:, None] - lon[None]) / 2) ** 2)
        km = 2 * 6371 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
        near = (km <= GENESIS_RADIUS_KM) & \
               (np.abs(hrs[:, None] - hrs[None]) <= GENESIS_WINDOW_HOURS)
        samples = first.index.get_level_values("sample").to_numpy()
        free = np.ones(len(first), bool)
        found = []
        while free.any():
            support = [len(set(samples[near[i] & free])) if free[i] else -1
                       for i in range(len(first))]
            i = int(np.argmax(support))
            share = support[i] / total
            if share < 0.10:
                break
            members = near[i] & free
            free &= ~members
            found.append((share, first.index[members], first[members]))
        systems = []
        gen_idx = gen.set_index(["sample", "track_id"]).index
        for k, (share, idx, firsts) in enumerate(found, 1):
            lat0 = firsts["lat"].median()
            lon0 = float(np.rad2deg(np.angle(np.exp(1j * np.deg2rad(
                firsts["lon"].to_numpy(float))).mean())))
            basin = basin_name(lat0, lon0)
            q25, q75 = firsts["valid_time"].quantile([0.25, 0.75])   # UTC
            g = gen[gen_idx.isin(idx)].copy()
            print(f"  Possible storm in the {basin}: forms in {100 * share:.0f}% of members "
                  f"near {firsts['lat'].median():.1f}, {firsts['lon'].median():.1f}")
            if share < IMPACT_MIN_GENESIS_SHARE:
                continue
            systems.append({
                "sid": f"genesis_{k}", "label": f"Possible storm in the {basin}",
                "title": IMPACT_TITLE_GENESIS.format(basin=basin),
                "slug": re.sub(r"[^a-z0-9]+", "_", f"possible {basin} storm {k}".lower()),
                "g": g, "region": intensity_region(lat0, lon0),
                "mean": mean_track(g),
                "formation": {"pct": 100 * share, "start_utc": q25, "end_utc": q75},
            })
        return systems

    # ------------------------------------------------ countries
    shp = shpreader.natural_earth(resolution="10m", category="cultural",
                                  name="admin_0_countries")
    country_geoms, country_iso = {}, {}
    for rec in shpreader.Reader(shp).records():
        a = rec.attributes
        name = str(a.get("NAME_EN") or a.get("NAME")).strip()
        name = COUNTRY_RENAME.get(name, name)
        iso = a.get("ISO_A2_EH") if a.get("ISO_A2_EH") not in (None, "-99") \
            else a.get("ISO_A2")
        if iso not in (None, "-99"):
            country_iso.setdefault(name, iso)
        geom = rec.geometry
        country_geoms[name] = (geom if name not in country_geoms
                               else country_geoms[name].union(geom))
    # Name Hawaii and Alaska separately from the rest of the United States.
    us = country_geoms.pop("United States", None)
    if us is not None:
        states_shp = shpreader.natural_earth(resolution="10m", category="cultural",
                                             name="admin_1_states_provinces")
        split = {rec.attributes["name"]: rec.geometry
                 for rec in shpreader.Reader(states_shp).records()
                 if rec.attributes.get("adm0_a3") == "USA"
                 and rec.attributes.get("name") in ("Hawaii", "Alaska")}
        for name, geom in split.items():
            country_geoms[name] = geom
            country_iso[name] = "US"
        country_geoms["United States"] = (
            us.difference(shapely.ops.unary_union(list(split.values())).buffer(0.02))
            if split else us)
    country_names = list(country_geoms)

    # Map extra: the Great Lakes (Natural Earth's land layer fills the lakes
    # in, so they are redrawn as water on top of every border).
    try:
        gl = gpd.read_file(GLOBAL_MAP["great_lakes_shapefile"])
        if gl.crs is not None and gl.crs.to_epsg() != 4326:
            gl = gl.to_crs("EPSG:4326")
        great_lakes = list(gl.geometry)
    except Exception as exc:
        print(f"Great Lakes outline unavailable: {exc}")
        great_lakes = []
    country_tree = shapely.STRtree([country_geoms[n] for n in country_names])

    # ------------------------------------------------ local time zones
    # IANA zone.tab: country code, principal-city coordinates, zone name.
    def iso6709(coord):
        m = re.match(r"([+-])(\d{2})(\d{2})(\d{2})?([+-])(\d{3})(\d{2})(\d{2})?$", coord)
        lat = int(m[2]) + int(m[3]) / 60 + int(m[4] or 0) / 3600
        lon = int(m[6]) + int(m[7]) / 60 + int(m[8] or 0) / 3600
        return (lat if m[1] == "+" else -lat), (lon if m[5] == "+" else -lon)

    zones = {}
    for line in pytz.open_resource("zone.tab").read().decode().splitlines():
        if line and not line.startswith("#"):
            cc, coord, name = line.split("\t")[:3]
            zones.setdefault(cc, []).append((*iso6709(coord), name))

    def circular_mean_lon(lons):
        return float(np.rad2deg(np.angle(np.exp(1j * np.deg2rad(
            np.asarray(lons, float))).mean())))

    def local_zone(country, lat, lon, max_km=2500):
        """Country's time zone nearest (lat, lon); longitude-based fallback.

        Remote territories filed under a distant parent (e.g. Clipperton
        Island under France) fall back too: nearest zone city > max_km away.
        """
        def km(z):
            p1, p2 = np.deg2rad(lat), np.deg2rad(z[0])
            dl = np.deg2rad(((z[1] - lon + 180) % 360) - 180)
            return 6371 * np.arccos(np.clip(np.sin(p1) * np.sin(p2) + np.cos(p1)
                                            * np.cos(p2) * np.cos(dl), -1, 1))
        candidates = zones.get(country_iso.get(country), [])
        if candidates:
            best = min(candidates, key=km)
            if km(best) <= max_km:
                return best[2]
        offset = int(round(((lon + 180) % 360 - 180) / 15))
        return f"Etc/GMT{-offset:+d}" if offset else "UTC"

    def to_local(utc, tz):
        """Naive UTC timestamp or Series -> naive local wall-clock time."""
        if isinstance(utc, pd.Series):
            return utc.dt.tz_localize("UTC").dt.tz_convert(tz).dt.tz_localize(None)
        return pd.Timestamp(utc).tz_localize("UTC").tz_convert(tz).tz_localize(None)

    def formation_dates(f, tz):
        return date_range(to_local(f["start_utc"], tz).floor("D"),
                          to_local(f["end_utc"], tz).floor("D"))

    def stitch_restarts(base_ids, max_hours=24, max_km=500):
        """Re-attach tracker restarts to the storm they continue.

        When a member's storm track ends and a new-genesis track starts within
        max_hours and max_km of that end point, it is the same system; give
        it the storm's ID so it is not counted as a separate new storm.
        """
        named = base_ids.str.match(r"^[A-Z]{2}\d{6}$")
        t = df.assign(_base=base_ids, _tid=df["track_id"].astype(str))
        ends = (t[named].sort_values("valid_time").groupby(["sample", "_base"])
                .last().reset_index())
        starts = (t[~named].sort_values("valid_time").groupby(["sample", "_tid"])
                  .first().reset_index())
        pairs = starts.merge(ends, on="sample", suffixes=("", "_end"))
        if pairs.empty:
            return base_ids
        gap = (pairs["valid_time"] - pairs["valid_time_end"]).dt.total_seconds() / 3600
        lat0, lat1 = np.deg2rad(pairs["lat"]), np.deg2rad(pairs["lat_end"])
        dlon = np.deg2rad(pairs["lon"] - pairs["lon_end"])
        km = 2 * 6371 * np.arcsin(np.sqrt(np.clip(
            np.sin((lat0 - lat1) / 2) ** 2
            + np.cos(lat0) * np.cos(lat1) * np.sin(dlon / 2) ** 2, 0, 1)))
        best = (pairs.assign(km=km)[(gap > 0) & (gap <= max_hours) & (km <= max_km)]
                .sort_values("km").drop_duplicates(["sample", "_tid"]))
        if best.empty:
            return base_ids
        mapping = dict(zip(zip(best["sample"], best["_tid"]), best["_base_end"]))
        stitched = pd.Series(
            [mapping.get((s, tid), b) for s, tid, b in
             zip(df["sample"], t["_tid"], base_ids)], index=base_ids.index)
        counts = best["_base_end"].value_counts()
        print("Tracker restarts re-attached: " + ", ".join(
            f"{n} to {sid}" for sid, n in counts.items()))
        return stitched

    # ------------------------------------------------ storms
    total = df["sample"].nunique()
    init = pd.Timestamp(datetime.strptime(RUN, "%Y_%m_%dT%H_%M"))
    base_ids = stitch_restarts(df["track_id"].astype(str).map(canonical_id))
    if IMPACT_STORM_IDS:
        storm_ids = [canonical_id(s) for s in IMPACT_STORM_IDS]
    else:
        support = df[base_ids.str.match(r"^[A-Z]{2}\d{6}$")].groupby(
            base_ids)["sample"].nunique()
        storm_ids = support[support >= IMPACT_MIN_MEMBER_SHARE * total].index.tolist()
    official = official_storms()
    systems = []
    for sid in storm_ids:
        g = df[base_ids == sid].copy()
        if g.empty:
            print(f"{sid}: not in {TRACK_MODEL} {RUN}")
            continue
        label = storm_label(sid, g, official)
        mean = None
        if ensemble_mean_df is not None:
            mean = ensemble_mean_df[
                ensemble_mean_df["track_id"].map(canonical_id) == sid
            ].sort_values("valid_time")
        systems.append({
            "sid": sid, "label": label, "title": IMPACT_TITLE.format(storm=label),
            "slug": re.sub(r"[^a-z0-9]+", "_", label.lower()).strip("_"),
            "g": g, "region": storm_region(sid, g), "mean": mean,
            "formation": None})
    print(f"Existing storms/invests ({len(systems)}): "
          f"{', '.join(s['label'] for s in systems) or 'none'}")
    if IMPACT_INCLUDE_GENESIS:
        print("Possible new storms (clusters in >= 10% of members):")
        systems += genesis_systems(base_ids, total, init)

    season_peaks = load_season_peaks() if IMPACT_SEASON_CONTEXT else None
    for system in systems:
        system["milestone"] = milestone_chance(system, total)
        system["peak"] = peak_winds(system)
        system["record"] = (season_context(system, total)
                            if IMPACT_SEASON_CONTEXT else None)
        table, pts = country_table(system["sid"], system["g"], init, total)
        shown = (table[(table["ts_force_pct"] >= IMPACT_MIN_CHANCE)
                       & ~table["country"].isin(IMPACT_HIDE_COUNTRIES)]
                 if not table.empty else table)
        m = system["milestone"]
        print(f"\n{system['label']} ({system['sid']}), "
              f"{system['g']['sample'].nunique()}/{total} members; "
              f"becomes {m['name']}: {m['pct']:.0f}%; peak winds "
              f"{mph(system['peak']['floor'])}/{mph(system['peak']['typical'])}/"
              f"{mph(system['peak']['ceiling'])} mph (floor/most likely/ceiling)"
              + (f"; {system['record']}" if system["record"] else ""))
        if shown.empty:
            near = (f" (highest: {table['country'].iloc[0]}, "
                    f"{table['ts_force_pct'].iloc[0]:.0f}%)" if not table.empty else "")
            print(f"  No country reaches a {IMPACT_MIN_CHANCE}% chance of "
                  f"34-kt winds{near}; no card made.")
            continue
        print(shown[["country", "ts_force_pct", "hu_force_pct", "likely_start",
                     "likely_end", "time_zone"]].rename(columns={
                         "ts_force_pct": "TS-force %", "hu_force_pct": "HU-force %"})
              .to_string(index=False, float_format=lambda v: f"{v:.0f}"))
        table.to_csv(os.path.join(
            output_dir("impact_cards"),
            f"{system['slug']}_country_wind_chances_{RUN}.csv"),
            index=False, float_format="%.1f")
        for mode in COLOR_MODES:                  # compute once, draw per mode
            with color_mode(mode):
                theme, colors = card_theme()
                for layout in IMPACT_LAYOUTS:
                    render_card(layout, system, pts, shown, total, init)

if GENERATE_STORM_IMPACT_CARDS:
    run_storm_impact_cards()
else:
    print("Storm impact cards: skipped (GENERATE_STORM_IMPACT_CARDS = False)")
